# MRKR Contralateral TKA — multi-view image + clinical fusion model (Colab)

**T7 / Track C.** Trains the pre-specified image model on the WebDataset shards written by
`notebooks/preprocess_colab.ipynb`, fuses it with the frozen M0 clinical model, freezes
everything, then reads the locked test split **exactly once**.

Nothing in this notebook chooses an architecture, a horizon, a penalty or a threshold that is not
already written down in `config/feasibility.yaml` or `MRKR_Contralateral_TKA_Study_Protocol.docx`.
Where the protocol is silent the notebook says so out loud and records the choice in the frozen
manifest.

---

## What is already frozen before this notebook runs

| artefact | where | what it fixes |
| --- | --- | --- |
| patient splits | `derived-data/cohort/patient_splits.parquet` | train 2,597 / val 371 / test 741 patients; events 373 / 54 / 106 |
| clinical features + imputer | `features_clinical.parquet`, `clinical_imputation_params.json` | train-fitted medians/modes, applied unchanged everywhere |
| M0 clinical Cox model | `m0_clinical_model.json` | 13 design columns (12 identified), coefficients, centering means, baseline survival, the train reverse-KM censoring curve, the three horizons |
| image crops + shards | `shards/{split}-{index:05d}.tar` + `labels.csv` | 512x512 8-bit contralateral crops, one writer per split |
| crop QA sign-off | `outputs/crop_qa_checklist.md` | the gate in section 2 below |

## The pre-specified model (protocol section 14 — no deviation)

One ConvNeXt-Tiny, ImageNet init, **shared across views** -> per-view embedding + a learned
**view-type embedding** -> **mask-aware attention set aggregator** -> concat the **M0 13-column
clinical design vector** -> **10 hazard logits** (ten six-month intervals after the day-90
landmark) -> **censoring-aware discrete-time NLL**. AdamW, gradient clipping, cosine decay,
warmup. Augmentation capped by protocol section 13. Five seeds, hazards averaged. Early stop on
validation NLL. One horizon-specific recalibration fitted on validation and frozen before test.

There is **no architecture tournament**. With 533 primary events, a tournament is the fastest way
to manufacture an overfitted result, and the protocol forbids it.

## Reading order and the two hard stops

1. Sections 0-8 touch **train and validation only**.
2. Section 9 (**freeze**) writes `FROZEN_MANIFEST.json`. After that, nothing about the model,
   the ensemble rule, the recalibration or the thresholds may change.
3. Section 10 trains the protocol section 25 robustness model — *after* the lock, by design.
4. Section 11 is the **sealed test cell**. It reads the test shards once, writes a one-time
   marker on Drive, and refuses to run again.
5. Sections 12-13 (explainability, subgroups) reuse the artefacts section 11 produced. They do
   **not** re-read the test shards.

Every cell runs top to bottom with no hidden state. Nothing is silently capped: where a limit
bites, the cell prints the count it dropped and why.

## 0. Mount Drive, operator flags

Every knob that changes between runs is in this one cell. Paths are resolved from
`config/feasibility.yaml` in the next cell, never hard-coded, and no personal path or account
name appears anywhere in this notebook: the Drive root is the Colab mount point and the project
folder is discovered by reading the staged config back.

In [ ]:
# ---- 0. Mount Drive + operator flags -----------------------------------------------------------
import os, sys, io, json, math, time, tarfile, hashlib, shutil, random, tempfile, inspect, warnings
from pathlib import Path
from collections import defaultdict

IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")           # re-running is a no-op once mounted

# Override only if you run this outside Colab with Drive mounted elsewhere. MRKR_DRIVE_ROOT is the
# same environment variable src/config.py honours, so nothing personal has to be written down.
DRIVE_ROOT = Path(os.environ.get("MRKR_DRIVE_ROOT", "/content/drive/MyDrive"))

# ---- operator flags (the ONLY knobs) -----------------------------------------------------------
SEEDS                 = [20250720, 20250721, 20250722, 20250723, 20250724]  # 5 pre-specified seeds
RESUME                = True     # skip a seed whose checkpoint is COMPLETE
FORCE_RETRAIN         = False    # ignore checkpoints and retrain from scratch (records a deviation)
SMOKE_ONLY            = False    # run the 1-batch forward/backward test and stop
MAX_EPOCHS_OVERRIDE   = None     # None = config model_image.max_epochs
BATCH_SIZE_OVERRIDE   = None     # None = config model_image.batch_size (32 patients)
GRAD_ACCUM_STEPS      = 1        # raise this instead of lowering the batch size on a small GPU
NUM_WORKERS           = 2
AMP                   = True     # mixed precision; the loss is still computed in float32
EVENT_AWARE_MINIBATCH = False    # protocol section 15 allows it; OFF keeps natural prevalence
                                 # everywhere. When ON, the loss carries the exact inverse
                                 # sampling weights so its expectation is the natural-prevalence
                                 # loss (see section 6). Validation/test NEVER resample.
UNLOCK_TEST           = False    # section 11 also requires this to be True. Read section 11 first.

print("Colab:", IN_COLAB, "| Drive root:", DRIVE_ROOT, "exists:", DRIVE_ROOT.exists())
print("SEEDS =", SEEDS, "| RESUME =", RESUME, "| SMOKE_ONLY =", SMOKE_ONLY,
      "| EVENT_AWARE_MINIBATCH =", EVENT_AWARE_MINIBATCH, "| UNLOCK_TEST =", UNLOCK_TEST)

# ---- local scratch (fast, ephemeral) -----------------------------------------------------------
_BASE = Path("/content") if os.path.isdir("/content") else Path(tempfile.gettempdir())
LOCAL_ROOT  = _BASE / "mrkr_t7";  LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_META  = LOCAL_ROOT / "meta"
LOCAL_CACHE = LOCAL_ROOT / "cache"; LOCAL_CACHE.mkdir(parents=True, exist_ok=True)
print("local scratch:", LOCAL_ROOT)

## 1. Stage the T7 metadata bundle (one-time, on the Mac)

`notebooks/preprocess_colab.ipynb` already staged six files into
`<project>/<preprocess.colab_metadata_dir>/`. This notebook needs **eight more** — the clinical
feature table, the frozen imputer, the frozen M0 and M1 model contracts, and the four QA artefacts
that make the gate in section 2 machine-checkable (the signed checklist plus the image,
laterality and outcome-record audit summaries). Run this **on the Mac, from the project root**:

```bash
DEST="$(python3 -c 'import os,yaml; c=yaml.safe_load(open("config/feasibility.yaml")); d=os.path.expandvars(os.environ.get("MRKR_DRIVE_ROOT") or c["transfer"]["dest_root"]); print(os.path.join(os.path.dirname(d), c["preprocess"]["colab_metadata_dir"]))')"
mkdir -p "$DEST/derived-data/cohort" "$DEST/outputs/tables"
cp derived-data/cohort/features_clinical.parquet          "$DEST/derived-data/cohort/"
cp derived-data/cohort/clinical_imputation_params.json    "$DEST/derived-data/cohort/"
cp derived-data/cohort/m0_clinical_model.json             "$DEST/derived-data/cohort/"
cp derived-data/cohort/m1_klg_model.json                  "$DEST/derived-data/cohort/"
cp outputs/crop_qa_checklist.md                           "$DEST/outputs/"
cp outputs/tables/image_audit_summary.csv                 "$DEST/outputs/tables/" 2>/dev/null || echo "MISSING image_audit_summary.csv - section 23 audit not scored yet"
cp outputs/tables/laterality_audit_summary.csv            "$DEST/outputs/tables/" 2>/dev/null || echo "MISSING laterality_audit_summary.csv - laterality audit not run yet"
cp outputs/tables/outcome_audit_summary.csv               "$DEST/outputs/tables/" 2>/dev/null || echo "MISSING outcome_audit_summary.csv - section 23 outcome-record audit not run yet"
ls -la "$DEST/derived-data/cohort" "$DEST/outputs" "$DEST/outputs/tables"
```

`features_clinical.parquet` is **patient-level PHI-adjacent data** (`empi_anon`, dates, comorbidity
flags). It goes to the same private Drive folder the DICOMs are already in and **nowhere else** —
not a bucket, not a gist, not a git remote.

The bundle deliberately does **not** carry `src/`. Everything this notebook needs from `src/` is
either (a) transcribed inline with an executable equivalence check against the artefact the real
function produced, or (b) not needed at all. Section 4 shows the check.

In [ ]:
# ---- 1. Resolve every Drive path from the staged config -----------------------------------------
import yaml
import numpy as np
import pandas as pd

# The bundle folder NAME is not hard-coded: each candidate <project>/<dir>/config/feasibility.yaml
# under the Drive root is read and the one whose folder is named by its own
# preprocess.colab_metadata_dir wins. Identical discovery to preprocess_colab.ipynb cell 1.
META_DRIVE = DRIVE_PROJECT = META_DIRNAME = None
_tried = []
for _cand in sorted(DRIVE_ROOT.glob("*/*/config/feasibility.yaml")):
    _name = str(yaml.safe_load(_cand.read_text()).get("preprocess", {}).get("colab_metadata_dir", ""))
    _tried.append(f"{_cand.parent.parent.name!r} (its config names {_name!r})")
    if _name and _cand.parent.parent.name == _name:
        META_DRIVE, DRIVE_PROJECT, META_DIRNAME = _cand.parent.parent, _cand.parent.parent.parent, _name
        break
assert META_DRIVE is not None, (
    f"metadata bundle not found under {DRIVE_ROOT}. Expected <project folder>/<the folder named by "
    f"preprocess.colab_metadata_dir>/config/feasibility.yaml. Candidates seen: {_tried or 'none'}. "
    f"Run the staging blocks from preprocess_colab.ipynb cell 1 AND from the markdown above.")

if LOCAL_META.exists():
    shutil.rmtree(LOCAL_META)
shutil.copytree(META_DRIVE, LOCAL_META)          # local disk reads, not Drive-FUSE reads

CFG_PATH = LOCAL_META / "config" / "feasibility.yaml"
CFG = yaml.safe_load(CFG_PATH.read_text())
MI  = CFG["model_image"]
PP  = CFG["preprocess"]
QA  = CFG["crop_qa"]
MC  = CFG["model_clinical"]

_dest_root   = Path(str(CFG["transfer"]["dest_root"]))
DICOM_ROOT   = DRIVE_PROJECT / _dest_root.name
SHARD_DIR    = DRIVE_PROJECT / str(PP["shards"]["out_dir"])
CKPT_DIR     = DRIVE_PROJECT / str(MI["checkpoint_dir"])
RESULTS_DIR  = DRIVE_PROJECT / "results_t7"
SEAL_DIR     = DRIVE_PROJECT / "test_seal"
for d in (CKPT_DIR, RESULTS_DIR, SEAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

COHORT       = LOCAL_META / "derived-data" / "cohort"
FEATURES_PQ  = COHORT / "features_clinical.parquet"
IMPUTER_JSON = COHORT / "clinical_imputation_params.json"
M0_JSON      = COHORT / "m0_clinical_model.json"
M1_JSON      = COHORT / "m1_klg_model.json"        # protocol Table 7 M1 = M0 + inferred KLG
SPLITS_PQ    = COHORT / "patient_splits.parquet"
LABELS_CSV   = SHARD_DIR / "labels.csv"
RUN_JSON     = SHARD_DIR / "preprocess_run.json"
CHECKLIST_MD = LOCAL_META / str(QA["checklist_md"])

for p in (FEATURES_PQ, IMPUTER_JSON, M0_JSON, M1_JSON, SPLITS_PQ):
    assert p.exists(), f"missing from the bundle: {p.name}. Re-run the staging block above."
assert SHARD_DIR.exists(), f"no shard directory at {SHARD_DIR}; run preprocess_colab.ipynb first"
assert LABELS_CSV.exists(), f"no sidecar at {LABELS_CSV}; preprocess_colab.ipynb cell 8 writes it"

VIEWS = list(PP["views_kept"])                     # ['frontal', 'lateral', 'sunrise']
IMAGE_EXT = str(PP["shards"]["image_ext"])         # 'png' -- the WebDataset tuple key
OUT_SIZE = int(PP["out_size"])                     # 512
assert IMAGE_EXT == "png" and VIEWS == ["frontal", "lateral", "sunrise"], \
    "the shard contract changed; re-read the preprocess HANDOFF cell before continuing"

print("Drive project :", DRIVE_PROJECT)
print("shards        :", SHARD_DIR)
print("checkpoints   :", CKPT_DIR)
print("results       :", RESULTS_DIR)
print("test seal     :", SEAL_DIR)
print("views         :", VIEWS, "| image member:", IMAGE_EXT, "| crop size:", OUT_SIZE)

## 2. HARD GATE — training is refused unless the QA evidence exists and reads PASS

This is not a reminder. It is an assertion block, and it is the reason a signed
`outputs/crop_qa_checklist.md` exists at all.

96% of the pre-index frontals are bilateral, and `standardize_to_left` plus border masking remove
the only laterality cues. A finished crop is therefore **anatomically identical whether the correct
or the wrong half was taken** — a wrong-half run produces a model that predicts contralateral
arthroplasty from the *index* knee, trains cleanly, validates cleanly, and is worthless. No metric
computed later in this notebook can detect it. The only detector is a human looking at the
two-panel evidence sheet, and the only machine-checkable trace of that human is the signed
checklist.

The gate therefore requires, all five:

1. `outputs/crop_qa_checklist.md` **exists** in the staged bundle;
2. it carries a **sign-off block** (`src/crop_qa.py` deletes that block entirely when the run does
   not qualify as a gate — a synthetic-fixture or smoke run has nothing to sign), and its
   `Result (PASS / FAIL)` row reads **PASS**;
3. the protocol section 23 image audit was scored over at least
   `crop_qa.image_audit_min_images` (400) images with **no item above the 2% critical-error
   threshold**;
4. the protocol section 7 laterality audit sampled at least
   `crop_qa.laterality_audit_min_patients` (200) patients;
5. the protocol section 23 **outcome-record** audit sampled at least
   `crop_qa.outcome_audit_min_records` (200) records. Section 23 mandates two distinct manual
   samples — images *and* outcome records — and the second one gates just as hard as the first:
   an unvalidated endpoint makes every metric in this notebook a measurement of the wrong label.
   `src/crop_qa.py` writes `outputs/tables/outcome_audit_summary.csv` and section 1 stages it.

If any of these fails the cell raises. Do not comment it out. If a genuine deviation is needed,
record it in `outputs/protocol_deviations.md` with a date and a reason first — that is what makes
it a deviation rather than an accident.

In [ ]:
# ---- 2. HARD GATE ------------------------------------------------------------------------------
import re, csv

GATE_FAILURES = []
GATE_EVIDENCE = {}


def _gate(ok, msg, evidence=None):
    GATE_EVIDENCE[msg] = evidence
    if not ok:
        GATE_FAILURES.append(msg)
    print(("  PASS  " if ok else "  FAIL  ") + msg + (f"   [{evidence}]" if evidence else ""))


print("GATE 1/5 - crop QA checklist")
_gate(CHECKLIST_MD.exists(), f"{QA['checklist_md']} exists in the staged bundle", str(CHECKLIST_MD))
_txt = CHECKLIST_MD.read_text() if CHECKLIST_MD.exists() else ""

print("GATE 2/5 - the checklist carries a signature block reading PASS")
_has_block = "## 6. Sign-off" in _txt and "Result (PASS / FAIL)" in _txt
_gate(_has_block, "a sign-off block is present (src/crop_qa.py removes it for a non-gate run)")
_no_banner = not re.search(r"NOT A GATE|SYNTHETIC|DEGRADED", _txt, re.I)
_gate(_no_banner, "the checklist carries no NOT-A-GATE / synthetic / degraded banner")
_m = re.search(r"\|\s*Result \(PASS / FAIL\)\s*\|\s*([^|]*?)\s*\|", _txt)
_result = (_m.group(1) if _m else "").strip().upper()
_gate(_result == "PASS", "Result (PASS / FAIL) reads PASS", f"found {_result!r}")
_m2 = re.search(r"\|\s*Reviewer name\s*\|\s*([^|]*?)\s*\|", _txt)
_m3 = re.search(r"\|\s*Date \(YYYY-MM-DD\)\s*\|\s*([^|]*?)\s*\|", _txt)
_gate(bool(_m2 and _m2.group(1).strip()) and bool(_m3 and _m3.group(1).strip()),
      "reviewer name and sign-off date are filled in",
      f"name={'set' if _m2 and _m2.group(1).strip() else 'BLANK'}, "
      f"date={(_m3.group(1).strip() if _m3 else '') or 'BLANK'}")

print("GATE 3/5 - protocol section 23 image-level audit (>= %d images, <= %.0f%% critical error)"
      % (int(QA["image_audit_min_images"]), 100 * float(QA["critical_error_threshold"])))
_img_csv = LOCAL_META / str(QA["image_audit_csv"])
if not _img_csv.exists():
    _gate(False, f"{QA['image_audit_csv']} exists (run: python3 -m src.crop_qa --score <workbook>)")
else:
    _ia = pd.read_csv(_img_csv)
    _n_rows = int(pd.to_numeric(_ia["n_workbook_rows"], errors="coerce").max())
    _gate(_n_rows >= int(QA["image_audit_min_images"]),
          "the audit workbook holds at least the required number of images",
          f"{_n_rows} rows vs {QA['image_audit_min_images']} required")
    _await = _ia["status"].astype(str).str.contains("awaiting", case=False, na=False)
    _gate(not _await.any(), "every scored item has two completed reviewer columns",
          f"awaiting: {sorted(_ia.loc[_await, 'item'].tolist())}")
    _over = _ia["exceeds_threshold"].astype(str).str.lower().isin(["true", "1"])
    _gate(not _over.any(), "no item exceeds the critical-error threshold",
          f"over: {sorted(_ia.loc[_over, 'item'].tolist())}")

print("GATE 4/5 - protocol section 7 laterality audit (>= %d patients)"
      % int(QA["laterality_audit_min_patients"]))
_lat_csv = LOCAL_META / str(QA["laterality_audit_csv"])
if not _lat_csv.exists():
    _gate(False, f"{QA['laterality_audit_csv']} exists (written by python3 -m src.crop_qa)")
else:
    _la = pd.read_csv(_lat_csv).set_index("metric")["value"]
    _n_pat = int(float(_la.get("n_patients_sampled", 0)))
    _n_req = int(float(_la.get("min_patients_required", QA["laterality_audit_min_patients"])))
    _gate(_n_pat >= _n_req, "the laterality audit sampled enough patients",
          f"{_n_pat} sampled vs {_n_req} required")

print("GATE 5/5 - protocol section 23 outcome-record audit (>= %d records)"
      % int(QA["outcome_audit_min_records"]))
# Section 23 requires TWO manual samples: images AND outcome records. The image audit above
# validates the PREDICTOR; this one validates the LABEL. A model trained against an
# unadjudicated endpoint is precise about the wrong thing, and no metric below can detect it.
_out_csv = LOCAL_META / str(QA["outcome_audit_csv"])
if not _out_csv.exists():
    _gate(False, f"{QA['outcome_audit_csv']} exists (written by python3 -m src.crop_qa)")
else:
    _oa = pd.read_csv(_out_csv).set_index("metric")["value"]
    _n_rec = int(float(_oa.get("n_sampled", 0)))
    _n_rec_req = int(float(_oa.get("min_required", QA["outcome_audit_min_records"])))
    _gate(_n_rec >= _n_rec_req, "the outcome-record audit sampled enough records",
          f"{_n_rec} sampled vs {_n_rec_req} required")
    _n_ev = int(float(_oa.get("n_events_sampled", 0)))
    _gate(_n_ev > 0, "the outcome-record sample actually contains events to adjudicate",
          f"{_n_ev} events sampled")

print()
if GATE_FAILURES:
    raise AssertionError(
        "TRAINING REFUSED. The crop QA / laterality evidence does not clear the gate:\n  - "
        + "\n  - ".join(GATE_FAILURES)
        + "\n\nThe missing evidence is produced by:\n"
          "  python3 -m src.crop_qa --dicom-root <DICOM root>        (contact sheet + workbooks)\n"
          "  python3 -m src.crop_qa --score outputs/tables/image_audit_workbook.csv\n"
          "  (the same run writes the laterality and outcome-record audit summaries)\n"
          "then a reviewer signs outputs/crop_qa_checklist.md with Result = PASS and you re-run\n"
          "the staging block in section 1. Do not edit or bypass this cell.")
GATE_PASSED = True
print("GATE PASSED - the crop QA sign-off and all three section 7 / 23 audits are on record. Training may proceed.")

## 3. Environment: install, record versions, seed everything

`timm` and `webdataset` are not in the Colab base image; `torch` and `torchvision` are. Versions are
**recorded**, not guessed: the exact resolved versions of every package that touches a number go
into `results_t7/environment.json` and into the frozen manifest, which is what protocol section 28
asks for (environment lockfile, fixed seeds, hardware description).

Seeding covers `random`, `numpy`, `torch` (CPU and CUDA), the DataLoader workers and the CUDA
convolution algorithm choice. cuDNN is put in deterministic mode. That costs throughput and is
worth it: with 373 training events, a run you cannot reproduce is a run you cannot defend.

In [ ]:
# ---- 3a. Install the two packages Colab does not ship -------------------------------------------
import subprocess, importlib

def _pip(*args):
    print("pip", *args)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

try:
    import timm                                    # noqa: F401
except ImportError:
    _pip("timm>=1.0,<2.0")
try:
    import webdataset                              # noqa: F401
except ImportError:
    _pip("webdataset>=0.2.86,<0.3")

In [ ]:
# ---- 3b. Record the environment and seed everything ---------------------------------------------
import torch, torchvision, timm, webdataset as wds
import platform

ENVIRONMENT = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "timm": timm.__version__,
    "webdataset": getattr(wds, "__version__", "unknown"),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_device": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
    "cuda_capability": (list(torch.cuda.get_device_capability(0)) if torch.cuda.is_available() else None),
    "cudnn": (torch.backends.cudnn.version() if torch.cuda.is_available() else None),
}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    print("WARNING: no GPU. Runtime > Change runtime type > GPU. Training on CPU is not viable "
          "at 512x512 and the checkpoints written here would be untrustworthy.")


def seed_everything(seed: int) -> None:
    """Seed every generator this notebook can reach, and force deterministic cuDNN."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def worker_init_fn(worker_id: int) -> None:
    """Per-worker seeding so augmentation is reproducible for a given (seed, epoch)."""
    s = torch.initial_seed() % (2 ** 31)
    np.random.seed(s + worker_id)
    random.seed(s + worker_id)


seed_everything(int(CFG["reproducibility"]["random_seed"]))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2))
for k, v in ENVIRONMENT.items():
    print(f"  {k:16s} {v}")
print("\nbase seed:", CFG["reproducibility"]["random_seed"], "| device:", DEVICE)

## 4. Shard integrity

Four checks, all of them assertions:

1. **No tar mixes splits.** `preprocess_colab.ipynb` opens one writer per split; if a single tar
   ever contained two splits, every "patient-level split" guarantee downstream would be a fiction.
   The check is done from member names, not from the file name.
2. **Per-split member counts equal the sidecar.** `labels.csv` is the join key for labels, views and
   masks; if it disagrees with the tars, some patient is being trained on a label that belongs to a
   different image.
3. **Sample structure.** Each sample is exactly `{key}.png` then `{key}.json`, contiguous, and the
   json's `key` matches.
4. **The frozen cohort counts.** train 2,597 / val 371 / test 741 patients, events 373 / 54 / 106.
   A mismatch means the cohort moved, and the frozen M0 comparison is no longer paired — that is a
   protocol deviation, not a warning.

The **test shards are never opened here.** Their existence is checked (so a missing test shard is
discovered now rather than in the sealed cell) but no member is read.

### The authoritative missing-view mask

`labels.csv` carries both `has_frontal` / `has_lateral` / `has_sunrise` / `n_views` (what the
cohort manifest *intended*) and `n_views_written` (what actually landed in a tar). They diverge
whenever an image failed protocol section 13 QA or failed to decode. **Only the realized set is
used**: the per-patient view set obtained by grouping the sidecar on `empi_anon`, cross-checked
against `n_views_written`. Protocol section 20 is explicit that image-view absence is represented
with a mask and **is not imputed** — there is no "average frontal" anywhere in this notebook.

19 patients have no frontal view at all. They stay in the cohort with `frontal = False` in the
mask; dropping them would silently change the denominator the M0 comparison is paired against.

In [ ]:
# ---- 4a. Sidecar, view masks, and the frozen cohort counts --------------------------------------
LABELS = pd.read_csv(LABELS_CSV, dtype={"empi_anon": str, "sop_uid": str, "key": str})
assert list(LABELS.columns) == list(PP["sidecar_columns"]), (
    "labels.csv columns drifted from preprocess.sidecar_columns; re-read the HANDOFF cell")
SPLITS_DF = pd.read_parquet(SPLITS_PQ)
FEATURES = pd.read_parquet(FEATURES_PQ)
assert FEATURES["empi_anon"].is_unique, "features_clinical.parquet must be one row per patient"
# One dtype for the join key everywhere. An int-vs-str mismatch between the parquet and the CSV
# would make every set intersection below empty and the failure would look like a data problem.
for _df in (LABELS, SPLITS_DF, FEATURES):
    _df["empi_anon"] = _df["empi_anon"].astype(str)
assert set(FEATURES["empi_anon"]) == set(SPLITS_DF["empi_anon"]), \
    "features_clinical.parquet and patient_splits.parquet describe different patients"

FROZEN_COHORT = {"train": {"patients": 2597, "events": 373},
                 "val":   {"patients": 371,  "events": 54},
                 "test":  {"patients": 741,  "events": 106}}
_dev = []
for sp, exp in FROZEN_COHORT.items():
    sub = FEATURES[FEATURES["split"] == sp]
    got = {"patients": int(len(sub)), "events": int(sub["event_indicator"].sum())}
    ok = got == exp
    print(f"  cohort {sp:5s} expected {exp}  got {got}  {'OK' if ok else 'MISMATCH'}")
    if not ok:
        _dev.append(f"{sp}: expected {exp}, got {got}")
assert not _dev, ("the locked cohort moved: " + "; ".join(_dev) + ". The frozen M0 model, the "
                  "sample-size calculation and the sealed-test plan are all keyed to these "
                  "counts. Record a protocol deviation before changing them.")

# The AUTHORITATIVE per-patient view mask: the realized set, never the manifest intent.
_grp = LABELS.groupby("empi_anon")
VIEW_SET = _grp["view"].agg(set)
VIEW_MASK = pd.DataFrame({v: VIEW_SET.map(lambda s, v=v: v in s) for v in VIEWS})
assert (_grp["n_views_written"].nunique() == 1).all(), "n_views_written is not constant per patient"
_nvw = _grp["n_views_written"].first()
assert (VIEW_MASK.sum(axis=1) == _nvw).all(), (
    "the realized view set disagrees with n_views_written; the sidecar is inconsistent")
_intent = LABELS.drop_duplicates("empi_anon").set_index("empi_anon")[
    ["has_frontal", "has_lateral", "has_sunrise"]].astype(bool)
_intent.columns = VIEWS
_diverge = (_intent.loc[VIEW_MASK.index] != VIEW_MASK).any(axis=1).sum()
print(f"\n  patients in shards: {len(VIEW_MASK)}")
print(f"  view availability (REALIZED): " +
      ", ".join(f"{v}={int(VIEW_MASK[v].sum())}" for v in VIEWS))
print(f"  patients with NO frontal: {int((~VIEW_MASK['frontal']).sum())}  "
      f"(kept, masked, never imputed - protocol section 20)")
print(f"  patients where manifest INTENT overstates what landed: {int(_diverge)}  "
      f"(intent columns are ignored everywhere below)")
_multi = LABELS.groupby(["empi_anon", "view"]).size()
print(f"  (patient, view) pairs with more than one image: {int((_multi > 1).sum())}  "
      f"-> each extra image enters the attention set as its own element (section 8), never dropped")

# Patients present in a split but absent from every shard cannot be trained or scored.
for sp in ("train", "val", "test"):
    want = set(FEATURES.loc[FEATURES["split"] == sp, "empi_anon"])
    have = set(LABELS.loc[LABELS["split"] == sp, "empi_anon"])
    missing = want - have
    print(f"  {sp:5s}: {len(have)} patients in shards of {len(want)} in the split; "
          f"{len(missing)} contributed no usable crop")
    assert not (have - want), f"{sp} shards contain patients outside the {sp} split"

In [ ]:
# ---- 4b. Open every tar: no tar mixes splits, counts match the sidecar --------------------------
from glob import glob


def shard_urls(split: str) -> list[str]:
    """Shards for ONE split. Never glob '*.tar' - that mixes splits."""
    return sorted(glob(str(SHARD_DIR / f"{split}-*.tar")))


def scan_tar(path: str) -> dict:
    """Member-level scan: keys, per-key extensions, and the order they appear in."""
    keys, exts, order = [], defaultdict(set), []
    with tarfile.open(path, "r") as tf:
        for m in tf:
            if not m.isfile():
                continue
            name = m.name
            k, _, ext = name.rpartition(".")
            keys.append(k)
            exts[k].add(ext)
            order.append((k, ext))
    return {"keys": keys, "exts": exts, "order": order}


SHARD_REPORT = {}
for sp in ("train", "val", "test"):
    urls = shard_urls(sp)
    if not urls:
        print(f"  {sp:5s}: NO SHARDS " + ("(expected until the seal is broken)" if sp == "test"
                                          else "-- run preprocess_colab.ipynb for this split"))
        SHARD_REPORT[sp] = {"n_shards": 0, "n_samples": 0, "urls": []}
        continue
    if sp == "test":
        # Existence and file size only. Nothing inside a test tar is read before section 11.
        SHARD_REPORT[sp] = {"n_shards": len(urls), "n_samples": None, "urls": urls}
        print(f"  {sp:5s}: {len(urls)} shard(s) present, "
              f"{sum(os.path.getsize(u) for u in urls) / 1e6:.0f} MB - NOT OPENED (sealed)")
        continue
    n_samples, mixed = 0, []
    for u in urls:
        info = scan_tar(u)
        uniq = sorted(set(info["keys"]))
        n_samples += len(uniq)
        for k in uniq:
            assert info["exts"][k] == {IMAGE_EXT, "json"}, \
                f"{Path(u).name}: sample {k[:12]}... has members {sorted(info['exts'][k])}"
        seq = [e for _, e in info["order"]]
        assert seq == [IMAGE_EXT, "json"] * len(uniq), \
            f"{Path(u).name}: members are not contiguous {IMAGE_EXT} then json"
        sub = LABELS[LABELS["key"].isin(uniq)]
        splits_here = set(sub["split"])
        if len(splits_here) > 1:
            mixed.append((Path(u).name, sorted(splits_here)))
        assert splits_here <= {sp}, f"{Path(u).name} carries {sorted(splits_here)}, not {sp}"
    assert not mixed, f"a tar mixes splits: {mixed}"
    n_side = int((LABELS["split"] == sp).sum())
    print(f"  {sp:5s}: {len(urls)} shard(s), {n_samples} samples in tars, {n_side} rows in the "
          f"sidecar  {'OK' if n_samples == n_side else 'MISMATCH'}")
    assert n_samples == n_side, f"{sp}: {n_samples} tar samples vs {n_side} sidecar rows"
    SHARD_REPORT[sp] = {"n_shards": len(urls), "n_samples": n_samples, "urls": urls}

FROZEN_IMAGES = {"train": 4270, "val": 602, "test": 1218}
for sp in ("train", "val"):
    got = SHARD_REPORT[sp]["n_samples"]
    assert got == FROZEN_IMAGES[sp], (
        f"{sp}: {got} kept images but the frozen contract says {FROZEN_IMAGES[sp]}. The image set "
        f"moved; the paired comparison against M0 is no longer on the same patients.")
print(f"\n  kept-image contract: {FROZEN_IMAGES} -> train/val verified, test deferred to section 11")
print("SHARD INTEGRITY OK")

## 5. The frozen clinical design — M0 replayed, never refitted

The fusion model consumes the **same 13-column design vector M0 uses**. Anything else would make
"imaging adds value beyond routine clinical variables" a comparison between two different clinical
models.

### M0 is protocol Table 7's M0, and it contains no KLG

Protocol Table 7 defines **M0 = "Age, sex, comorbidities, pain, image-to-index interval"** and
**M1 = "M0 plus inferred KLG"**; protocol Table 6 lists the dataset-inferred contralateral KLG as a
**secondary comparator only**. An earlier revision of this pipeline put `klg_contra_imp` and
`klg_contra_missing` into M0, where KLG became the dominant coefficient. That is the one predictor
M0 must not have: the primary estimand (protocol Table 8) is *M4 versus M0 at 5 years — does
imaging improve prediction beyond routine clinical variables* — and a comparator that already
carries a radiograph-derived severity grade makes that comparison imaging-against-imaging and
drives the measured increment toward zero. It is corrected here and recorded as deviation **D14**
in `outputs/protocol_deviations.md`.

**The 11 model columns and the 13 design columns are read from the JSON, not reconstructed.**
`preprocessing.model_columns` has 11 entries — the 11 imputed predictors of
`features_clinical.primary_predictors`, which now include `days_to_index_imp` (the image-to-index
interval, a Table 7 predictor that had been silently absent) and exclude every `klg_contra_*`
column. `pain_score_max_missing` is in the parquet and is deliberately **not** a model column: it
is the exact complement of `knee_pain_any_imp` on all 3,709 rows, a Cox model has no intercept, so
the pair spans an unidentified constant and only their difference is estimable. `klg_contra_imp`
and `klg_contra_missing` are also in the parquet and are also **not** M0 columns. Building the
design by globbing `*_imp` / `*_missing` would silently reintroduce both and change the fit.
`design_columns` is 13 because the age term expands into a 3-column restricted cubic spline basis
evaluated at the **persisted train knots**.

`n_parameters` is 13 but `identified_parameters` is **12** — the three `age_rcs*` columns span a
constant, so their level is not identified. Any events-per-parameter statement in the manuscript
must quote **12**, and no individual age-spline hazard ratio may be reported.

### M1 is a separate contract, on a separate patient set

`m1_klg_model.json` carries **M1 = M0 + `klg_contra_imp`**: 12 model columns, 14 design columns,
13 identified. It exists to answer protocol **Secondary objective 2** — *compare raw-image
prediction with an inferred KLG plus clinical comparator in the subset with eligible bilateral
frontal images*. `klg_contra` is missing for 143 of the 3,709 patients, and M1 is fitted and
scored **only where it is observed**; nothing median-imputes a severity grade. The `m1_klg`
training arm added in section 13 uses this design on that subset, and its comparisons live in
their own secondary family evaluated on their own eligible patient set. **The primary comparison
is unchanged: M4 versus M0, on the full common test set.**

**Nothing is refitted.** The imputer is applied from `clinical_imputation_params.json`; the spline
basis is evaluated at the stored knots; the linear predictor and horizon risks come from
`replay_from_json`. Three executable checks prove it:

* the inline `apply_imputer` reproduces, exactly, the `_imp` / `_missing` columns that the real
  `src.features_clinical.apply_imputer` wrote into the parquet;
* the spline basis reproduces the five verification rows stored in the JSON to 0.0;
* the linear predictor averages to 0 on train, because the centering means *are* the train means.

Test rows are not touched. `build_clinical_design` refuses the test split until section 11 opens
the seal.


In [ ]:
# ---- 5a. Frozen clinical contract ---------------------------------------------------------------
import patsy

M0 = json.loads(M0_JSON.read_text())
M1 = json.loads(M1_JSON.read_text())
IMPUTER = json.loads(IMPUTER_JSON.read_text())

MODEL_COLUMNS  = list(M0["preprocessing"]["model_columns"])     # 11 (protocol Table 7 M0)
DESIGN_COLUMNS = list(M0["design_columns"])                     # 13, order is binding
SPLINE         = M0["preprocessing"]["spline"]
HORIZONS       = [float(h["horizon_days"]) for h in M0["horizons"]]     # 365.0, 730.0, 1825.0
assert len(MODEL_COLUMNS) == 11 and len(DESIGN_COLUMNS) == 13
assert "pain_score_max_missing" not in MODEL_COLUMNS and "pain_score_max_missing" in FEATURES.columns
assert HORIZONS == [365.0, 730.0, 1825.0], HORIZONS

# Protocol Table 6 lists inferred KLG as a SECONDARY comparator and Table 7 puts it in M1. An M0
# carrying a radiograph-derived severity grade would make the primary estimand (Table 8: M4 vs M0)
# measure imaging against imaging, so the absence is asserted, not assumed.
assert not [c for c in MODEL_COLUMNS if c.startswith("klg")], MODEL_COLUMNS
assert "klg_contra_imp" in FEATURES.columns, "klg_contra_imp must still be in the parquet for M1"
# Protocol Table 7 names the image-to-index interval as an M0 predictor; it had been absent.
assert "days_to_index_imp" in MODEL_COLUMNS, MODEL_COLUMNS

M1_MODEL_COLUMNS  = list(M1["preprocessing"]["model_columns"])  # 12 = M0 + klg_contra_imp
M1_DESIGN_COLUMNS = list(M1["design_columns"])                  # 14
M1_SPLINE         = M1["preprocessing"]["spline"]               # knots from ELIGIBLE train rows
M1_ELIGIBILITY    = M1["eligibility"]
assert set(MODEL_COLUMNS) < set(M1_MODEL_COLUMNS), "M1 must be a strict superset of M0"
assert [c for c in M1_MODEL_COLUMNS if c not in MODEL_COLUMNS] == ["klg_contra_imp"]
assert len(M1_DESIGN_COLUMNS) == 14 and M1["identified_parameters"] == 13
assert [float(h["horizon_days"]) for h in M1["horizons"]] == HORIZONS

print("M0 model columns (11):", MODEL_COLUMNS)
print("M0 design columns (13):", DESIGN_COLUMNS)
print("excluded on purpose:", M0["preprocessing"]["excluded_model_columns"])
print(f"n_parameters={M0['n_parameters']} but identified_parameters={M0['identified_parameters']}; "
      f"unidentified level: {M0['identifiability']['level_unidentified_columns']}")
print("\nM1 = M0 + inferred KLG (protocol Table 7), fitted on the KLG-eligible subset only:")
print("  M1 model columns (12):", M1_MODEL_COLUMNS)
print(f"  n_parameters={M1['n_parameters']}, identified_parameters={M1['identified_parameters']}")
print(f"  eligible: {M1_ELIGIBILITY['n_eligible']} of "
      f"{M1_ELIGIBILITY['n_eligible'] + M1_ELIGIBILITY['n_ineligible']} patients "
      f"({M1_ELIGIBILITY['rule']}); by split {M1_ELIGIBILITY['n_eligible_by_split']}")
print("horizons (days):", HORIZONS,
      "| the 5-year horizon is", M0["horizons"][-1]["horizon_days"],
      "not", M0["horizons"][-1]["horizon_days_nominal"],
      "- clamped:", M0["horizons"][-1]["clamped"])
print("  administrative censoring lands ON day 1826, so the cumulative/dynamic control set "
      "(T > t) is EMPTY at 1826 and the AUROC is undefined there.")


In [ ]:
# ---- 5b. Frozen transforms, transcribed with an equivalence check --------------------------------
def apply_imputer(df: pd.DataFrame, params: dict) -> pd.DataFrame:
    """Verbatim transcription of src.features_clinical.apply_imputer - no statistic recomputed.

    The bundle carries no source tree, so the 15 lines are inlined here. The cell below asserts
    that this transcription reproduces, bit for bit, the columns the real function already wrote
    into features_clinical.parquet - so drift between the two is a test failure, not a surprise.
    """
    out = df.copy()
    for col, spec in params["columns"].items():
        assert col in out.columns, f"column {col!r} missing when applying the imputer"
        if spec["indicator_column"]:
            out[spec["indicator_column"]] = out[col].isna().astype("int8")
        imp = out[col].fillna(spec["fill_value"])
        if spec["strategy"] == "median":
            imp = imp.astype("float64")
        out[spec["imputed_column"]] = imp
        assert out[spec["imputed_column"]].notna().all(), f"{col!r} still missing after imputation"
    return out


def spline_basis(age, spline: dict) -> pd.DataFrame:
    """Restricted cubic spline basis at the PERSISTED train knots (src.model_clinical.spline_basis)."""
    dm = patsy.dmatrix("cr(age, knots=_k, lower_bound=_lb, upper_bound=_ub) - 1",
                       {"age": np.asarray(age, dtype=float), "_k": list(spline["interior_knots"]),
                        "_lb": float(spline["lower_bound"]), "_ub": float(spline["upper_bound"])},
                       return_type="dataframe")
    dm.columns = list(spline["basis_columns"])
    return dm.reset_index(drop=True)


def replay_from_json(model_json: dict, X: pd.DataFrame):
    """src.model_clinical.replay_from_json: lp and horizon risks from the JSON alone.

        lp(x)     = sum_j (x_j - centering_means[j]) * coefficients[j]
        risk(t|x) = 1 - S0(t) ** exp(lp(x)),  S0 right-continuous from baseline_survival
    """
    cols = list(model_json["design_columns"])
    assert list(X.columns) == cols, "design column order differs from the frozen contract"
    coefs = np.array([float(model_json["coefficients"][c]) for c in cols])
    means = np.array([float(model_json["centering_means"][c]) for c in cols])
    lp = (X.to_numpy(dtype=float) - means) @ coefs
    bt = np.asarray(model_json["baseline_survival"]["times"], dtype=float)
    bs = np.asarray(model_json["baseline_survival"]["survival"], dtype=float)
    risk = {}
    for h in model_json["horizons"]:
        t = float(h["horizon_days"])
        idx = int(np.searchsorted(bt, t, side="right")) - 1
        s0 = 1.0 if idx < 0 else float(bs[idx])
        risk[t] = 1.0 - s0 ** np.exp(lp)
    return lp, risk


SEAL_OPEN = False          # flipped to True only inside section 11

# The two frozen design contracts. M0 is the primary clinical comparator (protocol Table 7);
# M1 adds inferred KLG and is only defined on the KLG-eligible patients, so it carries its own
# eligibility predicate. Nothing else in the notebook is allowed to invent a third one.
DESIGNS = {
    "m0": dict(model_columns=MODEL_COLUMNS, design_columns=DESIGN_COLUMNS, spline=SPLINE,
               eligible=None,
               label="M0 clinical (protocol Table 7: age, sex, comorbidities, pain, "
                     "image-to-index interval)"),
    "m1": dict(model_columns=M1_MODEL_COLUMNS, design_columns=M1_DESIGN_COLUMNS, spline=M1_SPLINE,
               eligible=lambda f: (f["klg_contra_missing"].to_numpy(int) == 0),
               label="M1 = M0 + inferred KLG, KLG-eligible subset (protocol Table 7 / "
                     "Secondary objective 2)"),
}


def build_clinical_design(splits: list[str], design: str = "m0") -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return (patient frame, design matrix) for the requested splits, in a fixed row order.

    ``design`` selects a FROZEN contract from ``DESIGNS``; "m1" additionally drops the patients
    with no inferred contralateral KLG, because protocol Secondary objective 2 restricts that
    comparator to the subset with eligible bilateral frontal images and nothing here may impute
    a radiographic severity grade.
    """
    assert "test" not in splits or SEAL_OPEN, (
        "REFUSED: the test split cannot be read before section 11 opens the seal "
        "(protocol section 12: the locked test set stays unread until model, ensemble rule, "
        "thresholds and analysis script are frozen)")
    spec = DESIGNS[design]
    frame = (FEATURES[FEATURES["split"].isin(splits)]
             .sort_values("empi_anon").reset_index(drop=True))
    if spec["eligible"] is not None:
        frame = frame[spec["eligible"](frame)].reset_index(drop=True)
    raw = list(IMPUTER["columns"].keys())
    imp = apply_imputer(frame[raw].copy(), IMPUTER)
    sp = spec["spline"]
    basis = spline_basis(imp[sp["variable"]].to_numpy(dtype=float), sp)
    rest = imp[[c for c in spec["model_columns"] if c != sp["variable"]]].astype("float64").reset_index(drop=True)
    X = pd.concat([basis, rest], axis=1)
    assert list(X.columns) == spec["design_columns"], (list(X.columns), spec["design_columns"])
    assert X.notna().all().all(), "design matrix has missing values (imputation was skipped?)"
    if design == "m1":
        assert (frame["klg_contra"].notna()).all(), "a KLG-missing patient reached the M1 design"
    return frame, X

In [ ]:
# ---- 5c. Build the DEVELOPMENT design (train + val) and verify the replay ------------------------
DEV_FRAME, DEV_X = build_clinical_design(["train", "val"])
assert (DEV_FRAME["split"] == "test").sum() == 0
print("development design:", DEV_X.shape, DEV_FRAME["split"].value_counts().to_dict())

# check 1: the transcribed imputer reproduces the columns the real function wrote
_derived = ([s["imputed_column"] for s in IMPUTER["columns"].values()] +
            [s["indicator_column"] for s in IMPUTER["columns"].values() if s["indicator_column"]])
_re = apply_imputer(DEV_FRAME[list(IMPUTER["columns"].keys())].copy(), IMPUTER)
_bad = {c: int((_re[c].to_numpy(float) != DEV_FRAME[c].to_numpy(float)).sum()) for c in _derived
        if not np.array_equal(_re[c].to_numpy(float), DEV_FRAME[c].to_numpy(float), equal_nan=True)}
print(f"  imputer transcription vs the stored parquet columns: "
      f"{'IDENTICAL on all %d derived columns' % len(_derived) if not _bad else _bad}")
assert not _bad, "the inline apply_imputer has drifted from src/features_clinical.py"

# check 2: the spline basis reproduces the JSON's own verification rows
_ver = SPLINE["verification"]
_got = spline_basis(_ver["ages"], SPLINE).to_numpy()
_dmax = float(np.abs(_got - np.array(_ver["basis"])).max())
print(f"  spline basis vs the JSON verification rows (ages {_ver['ages']}): max |diff| = {_dmax:.1e}")
assert _dmax < 1e-12

# check 3: replay, and the train-mean-zero property of the centering
M0_LP_DEV, M0_RISK_DEV = replay_from_json(M0, DEV_X)
_is_tr = (DEV_FRAME["split"] == "train").to_numpy()
print(f"  linear predictor: n={M0_LP_DEV.size} mean={M0_LP_DEV.mean():+.6f} "
      f"sd={M0_LP_DEV.std(ddof=1):.6f} range=[{M0_LP_DEV.min():.3f}, {M0_LP_DEV.max():.3f}]")
print(f"  mean lp on TRAIN only = {M0_LP_DEV[_is_tr].mean():+.9f}  "
      f"(the centering means ARE the train means, so this must be ~0)")
assert abs(float(M0_LP_DEV[_is_tr].mean())) < 1e-9
for h in HORIZONS:
    print(f"  M0 predicted risk @ {h:6.0f} d: mean={M0_RISK_DEV[h].mean():.4f} "
          f"range=[{M0_RISK_DEV[h].min():.4f}, {M0_RISK_DEV[h].max():.4f}]")
print("\nM0 REPLAY VERIFIED on train+val. Test rows untouched.")

# ---- the M1 development design: same frozen rules, KLG-eligible patients only -------------------
DEV_FRAME_M1, DEV_X_M1 = build_clinical_design(["train", "val"], design="m1")
assert (DEV_FRAME_M1["split"] == "test").sum() == 0
_e = M1_ELIGIBILITY["n_eligible_by_split"]
assert len(DEV_FRAME_M1) == _e["train"] + _e["val"], (len(DEV_FRAME_M1), _e)
print(f"\nM1 development design: {DEV_X_M1.shape} "
      f"{DEV_FRAME_M1['split'].value_counts().to_dict()} "
      f"({len(DEV_FRAME) - len(DEV_FRAME_M1)} patients dropped: no inferred contralateral KLG)")
M1_LP_DEV, M1_RISK_DEV = replay_from_json(M1, DEV_X_M1)
_is_tr1 = (DEV_FRAME_M1["split"] == "train").to_numpy()
print(f"  M1 linear predictor: mean on TRAIN(eligible) = {M1_LP_DEV[_is_tr1].mean():+.9f} "
      f"(must be ~0 - the centering means ARE the eligible-train means)")
assert abs(float(M1_LP_DEV[_is_tr1].mean())) < 1e-9
# KLG is never imputed for M1: on the eligible subset the imputed column IS the observed value.
assert np.allclose(DEV_FRAME_M1["klg_contra"].to_numpy(float),
                   DEV_FRAME_M1["klg_contra_imp"].to_numpy(float))
print("M1 REPLAY VERIFIED on the KLG-eligible train+val patients. No KLG value was imputed.")

## 6. Discrete-time survival labels and the censoring-aware NLL

Protocol section 14 fixes the grid: **10 six-month intervals after the day-90 landmark**, i.e.
`edges = linspace(0, 1826, 11)`, interval width 182.6 days. `linspace` is used rather than
`arange(11) * 182.6` so both endpoints are exact and the interior edges are the doubling-exact
values (365.2, 730.4, ...). This matters: an event recorded exactly on an interval boundary must
not fall into the wrong interval because of a floating-point ulp.

Interval `k` covers `[edges[k], edges[k+1])`, and the final interval is closed at 1826 so an event
on the last day is not dropped.

**Censoring rule, and the price it charges.** A censored patient contributes only the intervals it
survived *in full*. Censoring inside interval `k` leaves that interval partially observed, so it is
not scored, and a patient censored inside interval 0 contributes nothing at all.

On this cohort that is **367 of the 2,968 development patients (12.4%)** — every patient whose last
observed record falls within 182.6 days of the day-90 landmark. They still receive predictions and
they are still counted in every IPCW evaluation; they simply carry no likelihood term. The cell
below prints the number rather than absorbing it.

The alternative convention scores the censoring interval as "survived", or scores it with a
fractional exposure weight. Scoring it outright would assert that those 367 patients survived to
day 182.6 when the data only show them to a median of about 33 days, which biases `h_0` downward.
The strict convention is the conservative one and it is what runs here — but it *is* a choice, and
a reviewer should know that 12.4% of the development set is silent in the loss because of it.

**The loss.**

$$\mathcal{L}_i \;=\; -\sum_{k}\; a_{ik}\Big[\,y_{ik}\log h_{ik} \;+\; (1-y_{ik})\log(1-h_{ik})\Big]$$

with $a_{ik}$ the at-risk mask, $y_{ik}$ one only in the event interval, and $h_{ik}$ the predicted
discrete hazard. This is the standard censoring-aware discrete-time (logistic-hazard) likelihood.
Reported as the sample-weighted **mean over patients**, so `val_nll` is comparable across epochs,
batch sizes and models — which is what early stopping monitors.

**Prevalence (protocol section 15).** Validation and test never resample: the natural prevalence is
retained. If `EVENT_AWARE_MINIBATCH` is switched on, each patient carries
`w_i = p_natural(i) / p_sampled(i)`, the exact inverse of its sampling probability, so the expected
weighted loss equals the natural-prevalence loss and the calibration that follows is unaffected.
The weights are unit-checked below.

### Hand-worked cases (all four the protocol audit asks for, plus three traps)

| case | T (days) | E | `k_event` | `n_scored` | why |
| --- | --- | --- | --- | --- | --- |
| event in interval 0 | 100 | 1 | 0 | 1 | first interval, one scored term |
| censored mid-interval | 300 | 0 | -1 | 1 | interval 1 partially observed, not scored |
| event exactly on a boundary | 365.2 | 1 | 2 | 3 | `[lo, hi)` puts the boundary in the next interval |
| censored at day 1826 | 1826 | 0 | -1 | 10 | survived all ten intervals |
| event at day 1826 | 1826 | 1 | 9 | 10 | last interval is closed on the right |
| censored on a boundary | 182.6 | 0 | -1 | 1 | interval 0 fully observed, interval 1 not started |
| censored inside interval 0 | 50 | 0 | -1 | 0 | contributes nothing, and says so |

On the real development rows the grid produces **events per interval
`[195, 67, 49, 31, 20, 24, 12, 10, 11, 8]`** against **at-risk counts
`[2601, 2125, 1835, 1601, 1398, 1215, 1062, 930, 821, 714]`** — marginal hazards falling from
0.075 in the first six months to about 0.011 thereafter. Nearly half the events land in interval 0,
which is worth remembering when reading the 1-year AUROC.

In [ ]:
# ---- 6a. Discrete-time grid and label construction (pure numpy) ---------------------------------
N_INTERVALS   = int(MI["survival_head"]["n_intervals"])          # 10
INTERVAL_MON  = int(MI["survival_head"]["interval_months"])      # 6
GRID_MAX_DAYS = float(M0["horizons"][-1]["horizon_days_nominal"])  # 1826
EDGES         = np.linspace(0.0, GRID_MAX_DAYS, N_INTERVALS + 1)
INTERVAL_DAYS = GRID_MAX_DAYS / N_INTERVALS                      # 182.6
assert N_INTERVALS == 10 and INTERVAL_MON == 6
assert MI["survival_head"]["kind"] == "discrete_time" and MI["survival_head"]["loss"] == "censoring_aware_nll"


def discretize_survival(time_days, event, n_intervals=None, edges=None):
    """Discrete-time survival targets on the frozen interval grid.

    Returns (at_risk, target, k_event, n_scored):
      at_risk[i, k]  1 while patient i is under observation in interval k
      target[i, k]   1 in the single interval where the event occurred, else 0
      k_event[i]     interval index of the event, -1 for censored
      n_scored[i]    number of intervals that contribute to the likelihood
    """
    n_intervals = N_INTERVALS if n_intervals is None else int(n_intervals)
    edges = EDGES if edges is None else np.asarray(edges, dtype=float)
    t = np.asarray(time_days, dtype=float)
    e = np.asarray(event, dtype=int)
    assert t.shape == e.shape and t.ndim == 1, "time and event must be 1-D and aligned"
    assert np.all(t >= 0), "negative time from landmark"
    assert np.all(t <= edges[-1] + 1e-9), (
        f"time beyond the {edges[-1]:.0f}-day grid; administrative censoring should have clamped it")
    assert set(np.unique(e)).issubset({0, 1}), "event must be 0/1"

    k_raw    = np.searchsorted(edges[1:], t, side="right")      # interval containing t
    k_ev     = np.minimum(k_raw, n_intervals - 1)               # event on the last edge -> interval 9
    n_scored = np.where(e == 1, k_ev + 1, np.minimum(k_raw, n_intervals)).astype(int)
    k_event  = np.where(e == 1, k_ev, -1).astype(int)

    kk = np.arange(n_intervals)[None, :]
    at_risk = (kk < n_scored[:, None]).astype(np.float64)
    target  = ((kk == k_event[:, None]) & (e[:, None] == 1)).astype(np.float64)
    assert (target.sum(axis=1) == (e == 1)).all(), "exactly one target interval per event"
    assert (target * (1.0 - at_risk)).sum() == 0.0, "an event interval must be at risk"
    return at_risk, target, k_event, n_scored


def dt_nll_numpy(hazards, at_risk, target, sample_weight=None, eps=1e-7):
    """Censoring-aware discrete-time NLL (numpy reference). Returns (mean, per_patient)."""
    h = np.clip(np.asarray(hazards, dtype=float), eps, 1.0 - eps)
    a = np.asarray(at_risk, dtype=float)
    y = np.asarray(target, dtype=float)
    per_patient = -(a * (y * np.log(h) + (1.0 - y) * np.log(1.0 - h))).sum(axis=1)
    w = np.ones(h.shape[0]) if sample_weight is None else np.asarray(sample_weight, dtype=float)
    return float((w * per_patient).sum() / w.sum()), per_patient


print("interval edges (days):", np.round(EDGES, 4).tolist())
print("interval width:", INTERVAL_DAYS, "| edges[2] == 365.2 exactly:", EDGES[2] == 365.2,
      "| edges[-1] == 1826.0 exactly:", EDGES[-1] == 1826.0)

In [ ]:
# ---- 6b. UNIT TESTS: hand-worked labels and hand-computed loss ----------------------------------
_t = np.array([100.0, 300.0, 365.2, 1826.0, 1826.0, 182.6, 0.0, 50.0])
_e = np.array([1,     0,     1,     0,      1,      0,     1,   0])
_lbl = ["event in interval 0", "censored mid-interval", "event exactly on edge 365.2",
        "censored at day 1826", "event at day 1826", "censored exactly on edge 182.6",
        "event at day 0", "censored inside interval 0"]
_exp_k  = [0, -1, 2, -1, 9, -1, 0, -1]
_exp_ns = [1,  1, 3, 10, 10,  1, 1,  0]
_ar, _tg, _ke, _ns = discretize_survival(_t, _e)
print("--- discrete-time labels: hand-computed vs computed ---")
for i in range(len(_t)):
    ok = (_ke[i] == _exp_k[i]) and (_ns[i] == _exp_ns[i])
    print(f"  {_lbl[i]:31s} T={_t[i]:7.1f} E={_e[i]}  hand(k={_exp_k[i]:2d}, n={_exp_ns[i]:2d})  "
          f"got(k={_ke[i]:2d}, n={_ns[i]:2d})  {'OK' if ok else 'MISMATCH'}")
    assert ok
    if i in (0, 2, 4, 7):
        print(f"      at_risk={_ar[i].astype(int).tolist()}  target={_tg[i].astype(int).tolist()}")

print("\n--- censoring-aware NLL: hand-computed vs computed ---")
_h = np.zeros((2, N_INTERVALS)); _h[:, 0] = 0.1; _h[:, 1] = 0.2
_a = np.zeros((2, N_INTERVALS)); _a[0, :2] = 1; _a[1, :2] = 1
_y = np.zeros((2, N_INTERVALS)); _y[0, 1] = 1
_hand_A = -(math.log(0.9) + math.log(0.2))       # survived interval 0, event in interval 1
_hand_B = -(math.log(0.9) + math.log(0.8))       # censored after two full intervals
_mean, _per = dt_nll_numpy(_h, _a, _y)
print(f"  A: event in interval 1, h=[.1,.2]  hand={_hand_A:.9f}  got={_per[0]:.9f}")
print(f"  B: censored after 2 intervals      hand={_hand_B:.9f}  got={_per[1]:.9f}")
print(f"  mean over the two patients         hand={(_hand_A+_hand_B)/2:.9f}  got={_mean:.9f}")
assert abs(_per[0] - _hand_A) < 1e-12 and abs(_per[1] - _hand_B) < 1e-12

_eps = 1e-6
_hp = np.full((1, N_INTERVALS), _eps); _hp[0, 2] = 1 - _eps
_ap, _yp, _, _ = discretize_survival(np.array([500.0]), np.array([1]))
_lp, _ = dt_nll_numpy(_hp, _ap, _yp)
print(f"  perfectly-predicted event in interval 2 (h_2 = 1-1e-6): NLL = {_lp:.3e}   hand ~ 3e-6")
assert _lp < 1e-5

_hc = np.full((1, N_INTERVALS), 0.3)
_ac, _yc, _, _nsc = discretize_survival(np.array([600.0]), np.array([0]))
_lc, _ = dt_nll_numpy(_hc, _ac, _yc)
print(f"  censored at day 600, h=0.3 -> {int(_nsc[0])} survived intervals only: "
      f"hand={-3*math.log(0.7):.9f}  got={_lc:.9f}")
assert _nsc[0] == 3 and abs(_lc - (-3 * math.log(0.7))) < 1e-12
_hc2 = _hc.copy(); _hc2[0, 3:] = 0.99
print(f"  same patient with hazards 3..9 set to 0.99: NLL = {dt_nll_numpy(_hc2,_ac,_yc)[0]:.9f} "
      f"(unchanged - a censored patient scores ONLY the intervals it survived)")
assert abs(dt_nll_numpy(_hc2, _ac, _yc)[0] - _lc) < 1e-12

_hw = np.full((4, N_INTERVALS), 0.2)
_aw, _yw, _, _ = discretize_survival(np.array([100., 100., 900., 900.]), np.array([1, 1, 0, 0]))
print(f"  prevalence correction: unweighted={dt_nll_numpy(_hw,_aw,_yw)[0]:.6f}  "
      f"weighted(0.5,0.5,1.5,1.5)={dt_nll_numpy(_hw,_aw,_yw,np.array([.5,.5,1.5,1.5]))[0]:.6f}")
print("\nDISCRETE-TIME LABEL AND LOSS TESTS PASSED")

## 7. Hazards to risk, the ensemble rule, and the IPCW evaluation

### From hazards to a risk at an arbitrary horizon

The three protocol horizons (365, 730, 1825 days) do not sit on interval boundaries, so
`S(t) = S(edges[k]) * (1 - h_k) ** ((t - edges[k]) / 182.6)` — a piecewise-constant hazard inside
the interval containing `t`. At a boundary this collapses exactly to the product of completed
intervals, so the two conventions agree wherever they overlap, and the function is monotone in `t`.

### The ensemble rule

`model_image.ensemble` is `average_hazard`. Hazards are averaged **across seeds first**, then
converted once. Averaging risks instead is a different estimator and the difference is not small:
for two seeds predicting a constant hazard of 0.10 and 0.30, the 5-year risk from the averaged
hazard is 0.8926 while the average of the two risks is 0.8115. The rule is applied in exactly one
place, `average_hazard`, so it cannot drift.

### IPCW cumulative/dynamic AUROC — identical to M0's

Cases are `T <= t & E = 1`, controls are `T > t`, and patients censored before `t` are unusable
(weight 0). Cases carry `1 / G(T-)`, controls carry `1 / G(t)`, where `G` is the **train reverse
Kaplan-Meier censoring curve stored in `m0_clinical_model.json`**. Reusing the frozen train `G`
rather than re-estimating it on test does two things: it keeps the weights identical between M0 and
the fusion model (which is what makes the paired difference a clean contrast), and it keeps every
nuisance quantity fitted in development data, as protocol section 12 requires. A test-estimated `G`
is reported as a labelled sensitivity, never as the primary.

`G(1825) = 0.2962` but `G(1826) = 0`, and there are zero patients with `T > 1826`. That is the
whole reason the 5-year horizon is day **1825**.

The self-test below re-derives M0's own validation metrics from the raw validation rows and the
JSON, and must reproduce `cindex = 0.667720639`, `auc@365 = 0.713905537`,
**`auc@730 = 0.676235139`** and `auc@1825 = 0.725434695`. The 2-year value, 0.676, is the number
the fusion model has to beat.

### The rest of protocol Table 7's metric list

`harrell_c` is the estimator behind M0's own `cindex` field, so the two are directly comparable.
Table 7 also names **Uno's C** (censoring-robust, case pairs weighted by `1/G(T-)^2`) and
**time-dependent AUPRC**; both are implemented here rather than left out, and the AUPRC is always
printed next to its **weighted no-skill prevalence**, because an average precision without the
prevalence it is competing against cannot be read at all. IPCW **Brier** and the cloglog
**calibration slope / calibration-in-the-large** complete the pre-specified set.

In [ ]:
# ---- 7a. Hazards -> survival -> horizon risk; the ensemble rule ---------------------------------
def hazards_to_survival(hazards):
    """S[:, k] = prod_{j<k} (1 - h_j); S[:, 0] == 1. Shape (n, n_intervals + 1)."""
    h = np.clip(np.asarray(hazards, dtype=float), 0.0, 1.0)
    return np.concatenate([np.ones((h.shape[0], 1)), np.cumprod(1.0 - h, axis=1)], axis=1)


def risk_at_horizon(hazards, horizon_days):
    """1 - S(t), piecewise-constant hazard inside the interval containing t."""
    h = np.clip(np.asarray(hazards, dtype=float), 0.0, 1.0)
    t = float(horizon_days)
    assert 0.0 <= t <= EDGES[-1], f"horizon {t} outside the discrete grid [0, {EDGES[-1]}]"
    S = hazards_to_survival(h)
    k = int(np.searchsorted(EDGES[1:], t, side="right"))
    if k >= h.shape[1]:
        return 1.0 - S[:, -1]
    frac = (t - EDGES[k]) / (EDGES[k + 1] - EDGES[k])
    return 1.0 - S[:, k] * np.power(1.0 - h[:, k], frac)


def average_hazard(hazard_list):
    """config model_image.ensemble == 'average_hazard': average HAZARDS across seeds, convert once."""
    stack = np.stack([np.asarray(h, dtype=float) for h in hazard_list], axis=0)
    assert stack.ndim == 3, "expected a list of (n_patients, n_intervals) hazard arrays"
    return stack.mean(axis=0)


_hh = np.array([[0.05, 0.04, 0.03, 0.02, 0.02, 0.01, 0.01, 0.01, 0.01, 0.01]])
_S = hazards_to_survival(_hh)
print("--- hazards -> survival -> risk: hand-computed vs computed ---")
print(f"  S(365.2) = 0.95*0.96          hand={0.95*0.96:.9f}  got={_S[0,2]:.9f}")
assert abs(_S[0, 2] - 0.95 * 0.96) < 1e-12
_f1 = (365.0 - EDGES[1]) / INTERVAL_DAYS
print(f"  risk(365.0), frac={_f1:.9f}     hand={1-0.95*0.96**_f1:.9f}  got={risk_at_horizon(_hh,365.0)[0]:.9f}")
assert abs(risk_at_horizon(_hh, 365.0)[0] - (1 - 0.95 * 0.96 ** _f1)) < 1e-12
print(f"  risk(365.2) at an exact edge   hand={1-0.95*0.96:.9f}  got={risk_at_horizon(_hh,365.2)[0]:.9f}")
assert abs(risk_at_horizon(_hh, 365.2)[0] - (1 - 0.95 * 0.96)) < 1e-12
_f9 = (1825.0 - EDGES[9]) / INTERVAL_DAYS
print(f"  risk(1825.0), frac={_f9:.9f}    hand={1-_S[0,9]*0.99**_f9:.9f}  got={risk_at_horizon(_hh,1825.0)[0]:.9f}")
assert abs(risk_at_horizon(_hh, 1825.0)[0] - (1 - _S[0, 9] * 0.99 ** _f9)) < 1e-12

print("\n--- ensemble rule: average HAZARD, not average risk ---")
_h1 = np.full((1, N_INTERVALS), 0.10); _h2 = np.full((1, N_INTERVALS), 0.30)
_hb = average_hazard([_h1, _h2])
_r_haz = risk_at_horizon(_hb, 1826.0)[0]
_r_rsk = 0.5 * (risk_at_horizon(_h1, 1826.0)[0] + risk_at_horizon(_h2, 1826.0)[0])
print(f"  averaged hazard 0.5*(0.1+0.3)       hand=0.200000000  got={_hb[0,0]:.9f}")
print(f"  5y risk from averaged HAZARD = 1-0.8^10 = {1-0.8**10:.9f}  got={_r_haz:.9f}")
print(f"  5y risk from averaged RISK              = {_r_rsk:.9f}  (differs by {abs(_r_haz-_r_rsk):.6f})")
assert abs(_hb[0, 0] - 0.2) < 1e-12 and abs(_r_haz - (1 - 0.8 ** 10)) < 1e-12

In [ ]:
# ---- 7b. IPCW cumulative/dynamic evaluation (mirrors src/model_clinical.py) ----------------------
import statsmodels.api as sm

G_TRAIN_GRID = np.asarray(M0["censoring_km_train"]["times"], dtype=float)
G_TRAIN_VALS = np.asarray(M0["censoring_km_train"]["survival"], dtype=float)


def step_value(grid, vals, t, left=False):
    """Right-continuous step function; left=True gives the left limit f(t-)."""
    grid = np.asarray(grid, dtype=float); vals = np.asarray(vals, dtype=float)
    t = np.atleast_1d(np.asarray(t, dtype=float))
    idx = np.searchsorted(grid, t, side=("left" if left else "right")) - 1
    return np.where(idx < 0, 1.0, vals[np.clip(idx, 0, len(vals) - 1)])


def reverse_km(times, events):
    """Reverse Kaplan-Meier G(u) = P(C > u), for the test-estimated sensitivity only."""
    t = np.asarray(times, float); e = 1 - np.asarray(events, int)     # censoring is the "event"
    order = np.argsort(t, kind="mergesort"); t, e = t[order], e[order]
    uniq = np.unique(t)
    n, g, grid, vals = t.size, 1.0, [0.0], [1.0]
    for u in uniq:
        at_risk = int((t >= u).sum())
        d = int(e[t == u].sum())
        if at_risk > 0 and d > 0:
            g *= (1.0 - d / at_risk)
        grid.append(float(u)); vals.append(g)
    return np.asarray(grid), np.asarray(vals)


def ipcw_labels_weights(times, events, horizon, g_grid=None, g_vals=None):
    """y in {1 case, 0 control, -1 unusable} and IPCW weights: cases 1/G(T-), controls 1/G(t)."""
    g_grid = G_TRAIN_GRID if g_grid is None else g_grid
    g_vals = G_TRAIN_VALS if g_vals is None else g_vals
    times = np.asarray(times, dtype=float); events = np.asarray(events, dtype=int)
    horizon = float(horizon)
    case = (times <= horizon) & (events == 1)
    ctrl = times > horizon
    y = np.where(case, 1, np.where(ctrl, 0, -1)).astype(int)
    w = np.zeros(times.shape, dtype=float)
    if case.any():
        g_case = step_value(g_grid, g_vals, times[case], left=True)
        assert (g_case > 0).all(), "censoring curve hit 0 at an event time - IPCW undefined"
        w[case] = 1.0 / g_case
    if ctrl.any():
        g_h = float(step_value(g_grid, g_vals, horizon, left=False)[0])
        assert g_h > 0, f"G({horizon}) = 0 - no follow-up beyond the horizon, IPCW undefined"
        w[ctrl] = 1.0 / g_h
    return y, w


def ipcw_auc(y, w, risk):
    """Weighted case-vs-control AUC; ties score 0.5. NaN if either arm is empty."""
    y = np.asarray(y, int); w = np.asarray(w, float); risk = np.asarray(risk, float)
    ci, co = y == 1, y == 0
    if not ci.any() or not co.any():
        return float("nan")
    rc, wc, ro, wo = risk[ci], w[ci], risk[co], w[co]
    cmp = (rc[:, None] > ro[None, :]).astype(float) + 0.5 * (rc[:, None] == ro[None, :])
    den = wc.sum() * wo.sum()
    return float(wc @ cmp @ wo / den) if den > 0 else float("nan")


def harrell_c(times, events, risk):
    """Harrell's C without lifelines. Higher risk must mean shorter survival.

    This is the estimator behind m0_clinical_model.json's ``cindex`` field, so the two are
    directly comparable. Uno's C, which protocol Table 7 also pre-specifies, is below.
    """
    t = np.asarray(times, float); e = np.asarray(events, int); r = np.asarray(risk, float)
    ti, tj = t[:, None], t[None, :]
    comparable = ((ti < tj) & (e[:, None] == 1)) | ((ti == tj) & (e[:, None] == 1) & (e[None, :] == 0))
    conc = (r[:, None] > r[None, :]).astype(float) + 0.5 * (r[:, None] == r[None, :])
    n = comparable.sum()
    return float((conc * comparable).sum() / n) if n else float("nan")


def uno_c(times, events, risk, g_grid=None, g_vals=None):
    """Uno's censoring-robust C (protocol Table 7). Case pairs weighted by 1 / G(T_i-)^2."""
    g_grid = G_TRAIN_GRID if g_grid is None else g_grid
    g_vals = G_TRAIN_VALS if g_vals is None else g_vals
    t = np.asarray(times, float); e = np.asarray(events, int); r = np.asarray(risk, float)
    g = step_value(g_grid, g_vals, t, left=True)
    wt = np.where(e == 1, 1.0 / np.maximum(g, 1e-12) ** 2, 0.0)
    comparable = (t[:, None] < t[None, :]) & (e[:, None] == 1)
    conc = (r[:, None] > r[None, :]).astype(float) + 0.5 * (r[:, None] == r[None, :])
    W = wt[:, None] * comparable
    den = W.sum()
    return float((conc * W).sum() / den) if den > 0 else float("nan")


def ipcw_auprc(y, w, risk):
    """IPCW time-dependent average precision at one horizon (protocol Table 7).

    Weighted precision-recall, stepped over unique risk values so ties form one step. The
    no-skill reference is the weighted case prevalence, which is printed alongside it because
    an AUPRC without its prevalence is uninterpretable.
    """
    y = np.asarray(y, int); w = np.asarray(w, float); r = np.asarray(risk, float)
    m = (y >= 0) & (w > 0)
    y, w, r = y[m], w[m], r[m]
    if not ((y == 1).any() and (y == 0).any()):
        return float("nan"), float("nan")
    order = np.argsort(-r, kind="mergesort")
    yy, ww, rr = y[order], w[order], r[order]
    tp = np.cumsum(ww * (yy == 1))
    fp = np.cumsum(ww * (yy == 0))
    last = np.r_[np.where(np.diff(rr) != 0)[0], rr.size - 1]      # one step per unique risk
    P = tp[last] / np.maximum(tp[last] + fp[last], 1e-12)
    R = tp[last] / tp[-1]
    ap = float(np.sum(np.diff(np.concatenate([[0.0], R])) * P))
    prevalence = float(ww[yy == 1].sum() / ww.sum())
    return ap, prevalence


def ipcw_brier(y, w, risk):
    """IPCW Brier score at one horizon (unusable patients carry weight 0)."""
    y = np.asarray(y, int); w = np.asarray(w, float); p = np.asarray(risk, float)
    m = (y >= 0) & (w > 0)
    return float((w[m] * (y[m] - p[m]) ** 2).sum() / w[m].sum()) if m.any() else float("nan")


def cloglog(p):
    return np.log(-np.log(1.0 - np.clip(np.asarray(p, float), 1e-8, 1 - 1e-8)))


def inv_cloglog(x):
    return 1.0 - np.exp(-np.exp(np.clip(np.asarray(x, float), -30.0, 30.0)))


def calibration_slope_intercept(y, w, pred_risk):
    """IPCW-weighted cloglog recalibration: cloglog(P) = a + b*cloglog(p_hat)."""
    y = np.asarray(y, int); w = np.asarray(w, float); x = cloglog(pred_risk)
    m = (y >= 0) & (w > 0)
    yy, ww, xx = y[m].astype(float), w[m], x[m]
    if yy.size < 10 or len(np.unique(yy)) < 2:
        return float("nan"), float("nan")
    fam = sm.families.Binomial(link=sm.families.links.CLogLog())
    slope = intercept = float("nan")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            slope = float(sm.GLM(yy, sm.add_constant(xx, has_constant="add"), family=fam,
                                 var_weights=ww).fit().params[1])
        except Exception:
            pass
        try:
            intercept = float(sm.GLM(yy, np.ones((yy.size, 1)), family=fam, var_weights=ww,
                                     offset=xx).fit().params[0])
        except Exception:
            pass
    return slope, intercept

In [ ]:
# ---- 7c. UNIT TESTS: IPCW on a toy case, then reproduce M0's own validation metrics -------------
print("--- IPCW labels / weights / AUC on a hand-computable toy case ---")
_g_grid = np.array([0.0, 50.0, 150.0]); _g_vals = np.array([1.0, 0.8, 0.5])
_T = np.array([40.0, 80.0, 90.0, 200.0, 300.0]); _E = np.array([1, 1, 0, 0, 1])
_R = np.array([0.9, 0.4, 0.5, 0.3, 0.2])
_y, _w = ipcw_labels_weights(_T, _E, 100.0, _g_grid, _g_vals)
print("  y (1 case / 0 control / -1 unusable):", _y.tolist(), " hand: [1, 1, -1, 0, 0]")
print("  hand w: case T=40 -> 1/G(40-)=1.00 ; case T=80 -> 1/G(80-)=1/0.8=1.25 ; "
      "controls -> 1/G(100)=1/0.8=1.25 ; censored T=90 -> 0")
print("  got  w:", np.round(_w, 6).tolist())
assert _y.tolist() == [1, 1, -1, 0, 0] and np.allclose(_w, [1.0, 1.25, 0.0, 1.25, 1.25])
_num = 1 * 1.25 + 1 * 1.25 + 1.25 * 1.25 + 1.25 * 1.25
_den = (1 + 1.25) * (1.25 + 1.25)
print(f"  hand AUC = {_num}/{_den} = {_num/_den:.9f}   got={ipcw_auc(_y,_w,_R):.9f}")
assert abs(ipcw_auc(_y, _w, _R) - _num / _den) < 1e-12
_Rt = np.array([0.3, 0.4, 0.5, 0.3, 0.2])              # one case tied with one control
_numt = 1 * 1.25 * 0.5 + 1 * 1.25 + 1.25 * 1.25 + 1.25 * 1.25
print(f"  a case tied with a control scores 0.5: hand={_numt/_den:.9f}  got={ipcw_auc(_y,_w,_Rt):.9f}")
assert abs(ipcw_auc(_y, _w, _Rt) - _numt / _den) < 1e-12
print(f"  Harrell C on 3 patients, 2 comparable pairs, both concordant: hand=1.0  "
      f"got={harrell_c(np.array([10.,20.,30.]), np.array([1,0,1]), np.array([0.9,0.5,0.7])):.9f}")
assert abs(harrell_c(np.array([10., 20., 30.]), np.array([1, 0, 1]), np.array([0.9, 0.5, 0.7])) - 1.0) < 1e-12
print(f"  Uno C on the same three, same ordering: got="
      f"{uno_c(np.array([10.,20.,30.]), np.array([1,0,1]), np.array([0.9,0.5,0.7]), _g_grid, _g_vals):.9f}"
      f"  (all pairs concordant -> 1.0 regardless of the weights)")
assert abs(uno_c(np.array([10., 20., 30.]), np.array([1, 0, 1]), np.array([0.9, 0.5, 0.7]),
                 _g_grid, _g_vals) - 1.0) < 1e-12
_ap, _prev = ipcw_auprc(_y, _w, _R)
print(f"  IPCW AUPRC on the toy case: hand=1.0 (perfect separation)  got={_ap:.9f}; "
      f"weighted case prevalence {_prev:.4f} = 2.25/4.75")
assert abs(_ap - 1.0) < 1e-12 and abs(_prev - 2.25 / 4.75) < 1e-12
_ap2, _ = ipcw_auprc(_y, _w, np.array([0.1, 0.2, 0.5, 0.9, 0.8]))    # perfectly WRONG ordering
print(f"  IPCW AUPRC with the ranking reversed: {_ap2:.4f} (must be well below the prevalence-free "
      f"perfect score)")
assert _ap2 < _ap

print("\n--- reproducing M0's frozen validation metrics from the raw val rows ---")
_va = (DEV_FRAME["split"] == "val").to_numpy()
_Tva = DEV_FRAME.loc[_va, "time_from_landmark"].to_numpy(float)
_Eva = DEV_FRAME.loc[_va, "event_indicator"].to_numpy(int)
print(f"  {'metric':16s} {'from JSON':>13s} {'recomputed':>13s} {'|diff|':>9s}")
_cj, _cg = M0["val_metrics"]["cindex"], harrell_c(_Tva, _Eva, M0_LP_DEV[_va])
print(f"  {'Harrell C':16s} {_cj:13.9f} {_cg:13.9f} {abs(_cj-_cg):9.1e}")
assert abs(_cj - _cg) < 1e-6
for _h in HORIZONS:
    _y2, _w2 = ipcw_labels_weights(_Tva, _Eva, _h)
    _aj, _ag = M0["val_metrics"][f"auc@{_h}"], ipcw_auc(_y2, _w2, M0_RISK_DEV[_h][_va])
    print(f"  {'IPCW AUC@' + str(int(_h)):16s} {_aj:13.9f} {_ag:13.9f} {abs(_aj-_ag):9.1e}"
          f"   cases={int((_y2==1).sum())} controls={int((_y2==0).sum())} "
          f"unusable={int((_y2==-1).sum())}")
    assert abs(_aj - _ag) < 1e-9, _h
print(f"\n  G(1825) = {float(step_value(G_TRAIN_GRID, G_TRAIN_VALS, 1825.0)[0]):.10f}   "
      f"G(1826) = {float(step_value(G_TRAIN_GRID, G_TRAIN_VALS, 1826.0)[0]):.10f}")
print(f"  validation patients with T > 1826: {int((_Tva > 1826).sum())} -> the C/D control set is "
      f"empty at 1826, which is why the frozen 5-year horizon is day 1825.")
M0_VAL_AUC_2Y = M0["val_metrics"]["auc@730.0"]
print(f"\n  THE NUMBER TO BEAT: M0 validation IPCW C/D AUROC at the 2-year horizon = "
      f"{M0_VAL_AUC_2Y:.4f} (Harrell C {M0['val_metrics']['cindex']:.4f}).")
print("IPCW EVALUATION VERIFIED against the frozen M0 model.")

## 8. Data pipeline — WebDataset in, patient-grouped batches out

The aggregator consumes a **set of views belonging to one patient**, so the patient is the unit of
a batch. WebDataset streams *images*, and a patient's images are not guaranteed to be contiguous
within a shard or even to live in the same shard. Grouping on the fly would either break patient
integrity or need an unbounded reorder buffer.

So the shards are streamed **exactly once per split** and materialized into a local uint8 memmap
plus a row index. Train + val is 4,872 images at 512x512x1 = **1.28 GB** on the Colab local SSD;
test adds 0.32 GB. After that the DataLoader is a plain patient-indexed dataset: deterministic,
trivially shuffled, and it never touches Drive-FUSE again — which on Drive is worth an order of
magnitude in epoch time.

The reader is `webdataset` with `.decode("l").to_tuple("png", "json")` exactly as the preprocess
HANDOFF specifies (`preprocess.shards.image_ext` is `"png"`; there is no `"img"` key). A `tarfile`
fallback producing the identical `(key, array, meta)` stream is included because the WebDataset
pipeline API has moved between releases and a training run should not die on a decoder rename; both
paths are asserted to agree on the first shard.

**Repeated views.** A patient can contribute more than one image of the same view. They are **not**
deduplicated: each image enters the set as its own element with its own view-type embedding, which
is what an attention set aggregator is for. The alternative — picking one — throws away a real
observation and needs an arbitrary rule.

**Missing views are masked, never imputed** (protocol section 20). A patient with no frontal has no
frontal element and a zero in the `view_available` vector. Both signals reach the model.

In [ ]:
# ---- 8a. Stream each split once into a local uint8 memmap ---------------------------------------
from PIL import Image


def _iter_tar_fallback(urls):
    """(key, HxW uint8 array, meta dict) straight from tarfile - no webdataset involved."""
    for u in urls:
        with tarfile.open(u, "r") as tf:
            cur_key, cur_img = None, None
            for m in tf:
                if not m.isfile():
                    continue
                k, _, ext = m.name.rpartition(".")
                data = tf.extractfile(m).read()
                if ext == IMAGE_EXT:
                    cur_key, cur_img = k, np.array(Image.open(io.BytesIO(data)).convert("L"))
                elif ext == "json":
                    assert k == cur_key, f"json {k} does not follow its png in {Path(u).name}"
                    yield k, cur_img, json.loads(data.decode())
                    cur_key, cur_img = None, None


def _iter_webdataset(urls):
    """(key, HxW uint8 array, meta dict) via webdataset, the HANDOFF contract."""
    ds = wds.WebDataset(urls, shardshuffle=False)      # default handler re-raises; never skip
    for sample in ds:
        key = sample["__key__"]
        img = np.array(Image.open(io.BytesIO(sample[IMAGE_EXT])).convert("L"))
        yield key, img, json.loads(sample["json"].decode() if isinstance(sample["json"], bytes)
                                   else sample["json"])


def materialize_split(split: str, force: bool = False):
    """Read every shard of ONE split once; return (memmap of images, index DataFrame)."""
    npy = LOCAL_CACHE / f"{split}_images.npy"
    idx = LOCAL_CACHE / f"{split}_index.parquet"
    side = LABELS[LABELS["split"] == split].reset_index(drop=True)
    n = len(side)
    assert n > 0, f"no sidecar rows for split {split!r}"
    if npy.exists() and idx.exists() and not force:
        arr = np.load(npy, mmap_mode="r")
        index = pd.read_parquet(idx)
        if arr.shape == (n, OUT_SIZE, OUT_SIZE) and len(index) == n:
            print(f"  {split}: cache hit ({n} images, {arr.nbytes/1e9:.2f} GB)")
            return arr, index
        print(f"  {split}: cache stale, rebuilding")

    urls = shard_urls(split)
    assert urls, f"no {split}-*.tar under {SHARD_DIR}"
    out = np.lib.format.open_memmap(npy, mode="w+", dtype=np.uint8, shape=(n, OUT_SIZE, OUT_SIZE))
    rows, seen, t0 = [], set(), time.time()
    reader = _iter_webdataset
    try:
        next(iter(_iter_webdataset(urls[:1])))
    except Exception as exc:                                   # noqa: BLE001
        print(f"  webdataset reader unusable ({type(exc).__name__}: {exc}); using the tarfile "
              f"fallback. Both produce the same (key, array, meta) stream.")
        reader = _iter_tar_fallback
    for i, (key, img, meta) in enumerate(reader(urls)):
        assert key not in seen, f"duplicate sample key in the {split} shards"
        seen.add(key)
        assert img.shape == (OUT_SIZE, OUT_SIZE) and img.dtype == np.uint8, \
            f"{key}: expected {OUT_SIZE}x{OUT_SIZE} uint8, got {img.shape} {img.dtype}"
        assert meta["key"] == key and meta["split"] == split, f"{key}: sidecar/member mismatch"
        out[i] = img
        rows.append({"row": i, "key": key, "empi_anon": str(meta["empi_anon"]),
                     "view": meta["view"], "contra_side": meta["contra_side"]})
    out.flush(); del out
    assert len(rows) == n, f"{split}: read {len(rows)} samples, sidecar has {n}"
    index = pd.DataFrame(rows)
    assert set(index["key"]) == set(side["key"]), f"{split}: shard keys differ from the sidecar"
    index.to_parquet(idx, index=False)
    print(f"  {split}: {n} images -> {npy.name} ({npy.stat().st_size/1e9:.2f} GB) "
          f"in {time.time()-t0:.0f}s")
    return np.load(npy, mmap_mode="r"), index


CACHE = {}
for _sp in ("train", "val"):
    CACHE[_sp] = materialize_split(_sp)

# the 31-px exact-zero border is a contract, not a bug -- verify it survived the round trip
_img0 = np.asarray(CACHE["train"][0][0])
_band = int(round(float(PP["mask_border_frac"]) * OUT_SIZE))
print(f"\n  border band = round({PP['mask_border_frac']} * {OUT_SIZE}) = {_band} px; "
      f"top/bottom/left/right all exactly zero: "
      f"{bool((_img0[:_band]==0).all() and (_img0[-_band:]==0).all() and (_img0[:,:_band]==0).all() and (_img0[:,-_band:]==0).all())}")
BORDER_PX = _band

## 9. Augmentation — capped by protocol section 13

> *"Use conservative training augmentation: rotation up to 5 degrees, translation and scale up to
> 5%, and mild intensity perturbation. **Avoid deformation that changes joint-space geometry.**"*

`model_image.augmentation` is `["rotation_5deg", "translate_0.05", "scale_0.05",
"brightness_contrast_0.1"]` and the cell below reads those four strings rather than hard-coding
numbers, so a config edit cannot silently diverge from what runs.

Three things are **forbidden** and the cell asserts they are absent:

* **`RandomResizedCrop`.** It jitters the aspect ratio, which *is* the joint-space deformation
  section 13 forbids. The outcome here is driven by joint-space narrowing; an augmentation that
  randomly rescales the tibiofemoral gap teaches the model that the gap width is noise.
* **Horizontal flip.** `standardize_to_left` already mirrored every right knee so all crops read as
  a left knee. Flipping would undo that and re-inject the side cue the crop pipeline removed.
* **Any elastic / perspective / shear warp**, for the same reason as the first.

After the affine transform the 31-px border band is re-zeroed. Rotation can otherwise drag interior
pixels into the band that the preprocessing deliberately blanked, and that band is where burned-in
laterality markers used to be.

Validation and test get **no augmentation at all** — normalization only, deterministically.

In [ ]:
# ---- 9. Augmentation, parsed from config, capped by protocol section 13 -------------------------
import torchvision.transforms.functional as TF

AUG_SPEC = list(MI["augmentation"])
_FORBIDDEN = ("randomresizedcrop", "flip", "elastic", "perspective", "shear", "erasing")
for _a in AUG_SPEC:
    assert not any(f in _a.lower().replace("_", "") for f in _FORBIDDEN), \
        f"augmentation {_a!r} is forbidden by protocol section 13"


def _num(prefix, default):
    for a in AUG_SPEC:
        if a.startswith(prefix):
            tail = a[len(prefix):].strip("_")
            tail = tail.replace("deg", "")
            try:
                return float(tail)
            except ValueError:
                return default
    return default


AUG = {
    "rotation_deg": _num("rotation", 0.0),
    "translate": _num("translate", 0.0),
    "scale": _num("scale", 0.0),
    "intensity": _num("brightness_contrast", 0.0),
}
assert AUG["rotation_deg"] <= 5.0 + 1e-9, "protocol section 13 caps rotation at 5 degrees"
assert AUG["translate"] <= 0.05 + 1e-9 and AUG["scale"] <= 0.05 + 1e-9, \
    "protocol section 13 caps translation and scale at 5%"
print("augmentation (train only):", AUG, "  from config:", AUG_SPEC)
print("forbidden and asserted absent: RandomResizedCrop, any flip, elastic/perspective/shear/erasing")

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def to_model_input(x: torch.Tensor) -> torch.Tensor:
    """(..., H, W) grayscale -> (..., 3, H, W) float, ImageNet-normalized.

    Deliberately out-of-place: integrated gradients feeds a float tensor that carries a
    gradient, and an in-place div_ on it would break the graph.
    """
    x = (x.to(torch.float32) / 255.0).unsqueeze(-3)           # (..., 1, H, W)
    x = x.expand(*x.shape[:-3], 3, *x.shape[-2:]).contiguous()
    return (x - IMAGENET_MEAN.to(x.device)) / IMAGENET_STD.to(x.device)


def zero_border_(x: torch.Tensor, band: int = None) -> torch.Tensor:
    """Re-blank the masked border band in place (rotation can drag pixels into it)."""
    b = BORDER_PX if band is None else band
    x[..., :b, :] = 0; x[..., -b:, :] = 0; x[..., :, :b] = 0; x[..., :, -b:] = 0
    return x


def augment_train(x_u8: torch.Tensor, rng: np.random.Generator) -> torch.Tensor:
    """Section-13-capped augmentation on a single (H, W) uint8 tensor. Returns uint8."""
    ang   = float(rng.uniform(-AUG["rotation_deg"], AUG["rotation_deg"]))
    tx    = int(round(float(rng.uniform(-AUG["translate"], AUG["translate"])) * x_u8.shape[-1]))
    ty    = int(round(float(rng.uniform(-AUG["translate"], AUG["translate"])) * x_u8.shape[-2]))
    scl   = 1.0 + float(rng.uniform(-AUG["scale"], AUG["scale"]))
    x = TF.affine(x_u8.unsqueeze(0), angle=ang, translate=[tx, ty], scale=scl, shear=[0.0, 0.0],
                  interpolation=TF.InterpolationMode.BILINEAR, fill=0)
    if AUG["intensity"] > 0:
        b = 1.0 + float(rng.uniform(-AUG["intensity"], AUG["intensity"]))
        c = 1.0 + float(rng.uniform(-AUG["intensity"], AUG["intensity"]))
        x = TF.adjust_contrast(TF.adjust_brightness(x, b), c)
    return zero_border_(x.squeeze(0))


_probe = torch.from_numpy(np.asarray(CACHE["train"][0][0]).copy())
_aug = augment_train(_probe, np.random.default_rng(0))
print(f"\nprobe: input {tuple(_probe.shape)} {_probe.dtype} -> augmented {tuple(_aug.shape)} "
      f"{_aug.dtype}; border still exactly zero: "
      f"{bool((_aug[:BORDER_PX]==0).all() and (_aug[:,:BORDER_PX]==0).all())}")
print("model input tensor:", tuple(to_model_input(_aug).shape), to_model_input(_aug).dtype)

In [ ]:
# ---- 9b. Patient-grouped dataset and collate ----------------------------------------------------
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

VIEW_ID = {v: i for i, v in enumerate(VIEWS)}
MAX_ELEMS = int(LABELS[LABELS["split"].isin(["train", "val"])].groupby("empi_anon").size().max())
print(f"max images per development patient = {MAX_ELEMS}  (set size the aggregator pads to)")

# Frozen train-only standardization of each clinical design (protocol section 12: every
# normalization parameter is fitted in development data). Means come from the frozen Cox JSON so
# a fusion model and its Cox counterpart are centred identically; the scale is the TRAIN standard
# deviation. M1 has its own entry because it is a different design on a different (KLG-eligible)
# patient set, so its train statistics are NOT M0's.
def _fit_clin_stats(frame, X, cox_json, design_columns):
    tr = (frame["split"] == "train").to_numpy()
    mean = np.array([float(cox_json["centering_means"][c]) for c in design_columns])
    std = X.to_numpy(float)[tr].std(axis=0, ddof=0)
    return dict(mean=mean, std=np.where(std < 1e-8, 1.0, std), columns=list(design_columns),
                n_train=int(tr.sum()), degenerate=[c for c, s in zip(design_columns, std)
                                                   if s < 1e-8])


CLIN_STATS = {
    "m0": _fit_clin_stats(DEV_FRAME, DEV_X, M0, DESIGN_COLUMNS),
    "m1": _fit_clin_stats(DEV_FRAME_M1, DEV_X_M1, M1, M1_DESIGN_COLUMNS),
}
CLIN_MEAN, CLIN_STD = CLIN_STATS["m0"]["mean"], CLIN_STATS["m0"]["std"]   # M0 names kept
for _d, _s in CLIN_STATS.items():
    print(f"clinical standardization [{_d}] fitted on {_s['n_train']} TRAIN rows only; "
          f"columns with zero train variance: {_s['degenerate']}")


def standardize_clinical(X: pd.DataFrame, design: str = "m0") -> np.ndarray:
    s = CLIN_STATS[design]
    assert list(X.columns) == s["columns"], (list(X.columns), s["columns"])
    return (X.to_numpy(dtype=float) - s["mean"]) / s["std"]


class PatientViewDataset(Dataset):
    """One item = one patient: a padded set of views, a presence mask, clinical vector, labels."""

    def __init__(self, split: str, frame: pd.DataFrame, X: pd.DataFrame, *, train: bool,
                 views_allowed=None, max_elems: int = None, design: str = "m0"):
        arr, index = CACHE[split]
        self.images = arr
        self.train = bool(train)
        self.design = str(design)               # which frozen clinical contract X follows
        self.views_allowed = set(VIEWS if views_allowed is None else views_allowed)
        self.max_elems = MAX_ELEMS if max_elems is None else int(max_elems)
        keep = index[index["view"].isin(self.views_allowed)]
        by_pat = defaultdict(list)
        for r, p, v in zip(keep["row"].to_numpy(), keep["empi_anon"].to_numpy(),
                           keep["view"].to_numpy()):
            by_pat[str(p)].append((int(r), VIEW_ID[v]))
        in_split = (frame["split"].to_numpy() == split)
        sub = frame[in_split].reset_index(drop=True)
        Xs = standardize_clinical(X, self.design)[in_split]
        self.pids, self.elems, self.clin, self.rows_kept = [], [], [], []
        self.n_no_image, self.n_truncated = 0, 0
        for i, p in enumerate(sub["empi_anon"].to_numpy()):
            e = sorted(by_pat.get(str(p), []))
            if not e:
                self.n_no_image += 1
                continue                                  # cannot score a patient with no crop
            if len(e) > self.max_elems:
                self.n_truncated += 1
                e = e[:self.max_elems]                    # deterministic, and counted
            self.pids.append(str(p)); self.elems.append(e); self.clin.append(Xs[i])
            self.rows_kept.append(i)
        self.frame = sub.iloc[self.rows_kept].reset_index(drop=True)
        self.clin = np.asarray(self.clin, dtype=np.float32)
        t = self.frame["time_from_landmark"].to_numpy(float)
        e = self.frame["event_indicator"].to_numpy(int)
        self.at_risk, self.target, self.k_event, self.n_scored = discretize_survival(t, e)
        self.time, self.event = t, e
        self.loss_weight = np.ones(len(self.pids))

    def __len__(self):
        return len(self.pids)

    def __getitem__(self, i):
        # Deterministic per (patient, epoch-seed): hashlib, never the salted builtin hash().
        _h = int(hashlib.sha1(self.pids[i].encode()).hexdigest()[:8], 16)
        rng = np.random.default_rng((_h ^ (int(torch.initial_seed()) & 0x7FFFFFFF)) % 2**32)
        e = self.elems[i]
        imgs = torch.zeros(self.max_elems, OUT_SIZE, OUT_SIZE, dtype=torch.uint8)
        vids = torch.zeros(self.max_elems, dtype=torch.long)
        mask = torch.zeros(self.max_elems, dtype=torch.bool)
        avail = torch.zeros(len(VIEWS), dtype=torch.float32)
        for j, (row, vid) in enumerate(e):
            x = torch.from_numpy(np.asarray(self.images[row]).copy())
            imgs[j] = augment_train(x, rng) if self.train else x
            vids[j] = vid; mask[j] = True; avail[vid] = 1.0
        return {
            "images": imgs, "view_id": vids, "mask": mask, "view_available": avail,
            "clinical": torch.from_numpy(self.clin[i]),
            "at_risk": torch.from_numpy(self.at_risk[i].astype(np.float32)),
            "target": torch.from_numpy(self.target[i].astype(np.float32)),
            "event": torch.tensor(float(self.event[i])),
            "time": torch.tensor(float(self.time[i])),
            "idx": torch.tensor(i),
        }


def make_loader(ds, batch_size, *, shuffle, seed=0, event_aware=False):
    """DataLoader. Event-aware sampling attaches the exact inverse-probability correction."""
    g = torch.Generator(); g.manual_seed(int(seed))
    if event_aware:
        p_nat = np.ones(len(ds))
        w = np.where(ds.event == 1, 1.0 / max(ds.event.mean(), 1e-9),
                     1.0 / max(1.0 - ds.event.mean(), 1e-9))
        w = w / w.sum()
        ds.loss_weight = (p_nat / len(ds)) / w              # undoes the oversampling exactly
        sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double),
                                        num_samples=len(ds), replacement=True, generator=g)
        return DataLoader(ds, batch_size=batch_size, sampler=sampler, num_workers=NUM_WORKERS,
                          pin_memory=True, drop_last=False, worker_init_fn=worker_init_fn,
                          generator=g)
    ds.loss_weight = np.ones(len(ds))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=True, drop_last=False, worker_init_fn=worker_init_fn, generator=g)


TRAIN_DS = PatientViewDataset("train", DEV_FRAME, DEV_X, train=True)
VAL_DS   = PatientViewDataset("val",   DEV_FRAME, DEV_X, train=False)
for nm, ds in (("train", TRAIN_DS), ("val", VAL_DS)):
    print(f"  {nm:5s}: {len(ds)} patients, {int(ds.event.sum())} events "
          f"({100*ds.event.mean():.1f}% - the NATURAL prevalence), "
          f"{ds.n_no_image} patients dropped for having no usable crop, "
          f"{ds.n_truncated} truncated to {MAX_ELEMS} views")
    print(f"         patients with no frontal: "
          f"{int((~VIEW_MASK.loc[[p for p in ds.pids if p in VIEW_MASK.index], 'frontal']).sum())}"
          f"  | patients contributing 0 scored intervals: {int((ds.n_scored == 0).sum())}")

## 10. The model — protocol section 14, verbatim

```
      per-view crop (512x512x1 -> 3ch)
                 |
        ConvNeXt-Tiny, ImageNet init          <- ONE encoder, weights SHARED across views
                 |  (global pooled, 768-d)
        + learned view-type embedding (32-d)  <- model_image.view_embedding_dim
                 |
        mask-aware attention set aggregator   <- model_image.aggregator == "attention"
                 |  (768-d patient image embedding)
        concat [ image embedding | 3-d view-availability | 14-d M0 clinical design ]
                 |
             MLP head
                 |
        10 hazard logits  ->  sigmoid  ->  h_1 .. h_10
```

**One architecture.** ConvNeXt-Tiny, pre-specified, ImageNet-initialized, shared across views.
There is no tournament and no per-view encoder: 533 primary events do not support either.

**The aggregator is mask-aware in the only way that matters.** Padded slots get their attention
logit set to `-inf` *before* the softmax, not zeroed after it. Zeroing after the softmax leaves the
padding in the denominator, which makes a one-view patient's embedding systematically smaller than a
three-view patient's — a missing-view artefact masquerading as signal. The unit check below shows
that padding a one-view patient to three slots changes nothing.

**Three modes from one class**, so the protocol section 24 comparisons come from the same code and
differ only in what is switched off:

| mode | image branch | clinical branch | protocol name |
| --- | --- | --- | --- |
| `clinical_only` | off | on | matched discrete-time comparator for M0 |
| `image_only` | on | off | M2 (frontal only) / M3 (multi-view) |
| `fusion` | on | on | **M4, the primary candidate** |

and `views_allowed` restricts the set to `["frontal"]` for the frontal-only arm, on the *same*
patients, which is what section 24 asks for.

**Head initialization.** The final bias is set to `logit(h_k)` under the observed training event
rate, so epoch 0 starts at the natural prevalence instead of at hazard 0.5. With ~14% of patients
having an event, starting at 0.5 wastes the first epochs and makes early stopping noisy.

In [ ]:
# ---- 10. Model ----------------------------------------------------------------------------------
import torch.nn as nn
import torch.nn.functional as F

ARCH = str(MI["architecture"])
assert ARCH == "convnext_tiny", f"protocol section 14 pre-specifies convnext_tiny, config says {ARCH}"
assert MI["pretrained"] == "imagenet" and bool(MI["shared_encoder_across_views"])
assert MI["aggregator"] == "attention"
VIEW_EMB_DIM = int(MI["view_embedding_dim"])


class MaskedAttentionPool(nn.Module):
    """Gated attention over a padded set. Padded slots are -inf BEFORE the softmax."""

    def __init__(self, dim: int, hidden: int = 128):
        super().__init__()
        self.V = nn.Linear(dim, hidden)
        self.U = nn.Linear(dim, hidden)
        self.w = nn.Linear(hidden, 1)

    def forward(self, x, mask):                      # x (B, E, D), mask (B, E) bool
        a = self.w(torch.tanh(self.V(x)) * torch.sigmoid(self.U(x))).squeeze(-1)   # (B, E)
        a = a.masked_fill(~mask, float("-inf"))
        a = torch.softmax(a, dim=1)
        a = torch.nan_to_num(a, nan=0.0)             # a patient with an empty set cannot occur
        return torch.einsum("be,bed->bd", a, x), a


class SurvivalFusionNet(nn.Module):
    """Protocol section 14: shared ConvNeXt-Tiny -> view embeddings + mask -> attention -> hazards."""

    def __init__(self, *, n_intervals: int, n_clinical: int, mode: str = "fusion",
                 arch: str = ARCH, pretrained: bool = True, view_emb_dim: int = VIEW_EMB_DIM,
                 head_hidden: int = 256, dropout: float = 0.2, base_hazard=None):
        super().__init__()
        assert mode in ("fusion", "image_only", "clinical_only")
        self.mode, self.n_intervals, self.arch = mode, n_intervals, arch
        self.use_image = mode in ("fusion", "image_only")
        self.use_clinical = mode in ("fusion", "clinical_only")
        if self.use_image:
            self.encoder = timm.create_model(arch, pretrained=pretrained, num_classes=0)
            self.feat_dim = int(self.encoder.num_features)
            self.view_emb = nn.Embedding(len(VIEWS), view_emb_dim)
            self.proj = nn.Linear(self.feat_dim + view_emb_dim, self.feat_dim)
            self.pool = MaskedAttentionPool(self.feat_dim)
            # The image embedding is normalised ON ITS OWN, before the concatenation. A single
            # LayerNorm over the concatenated vector would compute its statistics over 768 image
            # dimensions and 14 clinical ones, so the image block would set the scale and the
            # clinical block would be squashed - which would quietly weaken the very contrast
            # this model exists to measure. The clinical vector arrives already standardised on
            # train, and view_available is 0/1, so neither needs one.
            self.img_norm = nn.LayerNorm(self.feat_dim)
        else:
            self.feat_dim = 0
        in_dim = (self.feat_dim + len(VIEWS) if self.use_image else 0) + \
                 (n_clinical if self.use_clinical else 0)
        assert in_dim > 0, "a model with neither branch has no inputs"
        self.head = nn.Sequential(
            nn.Linear(in_dim, head_hidden), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(head_hidden, n_intervals))
        if base_hazard is not None:
            b = torch.as_tensor(base_hazard, dtype=torch.float32).clamp(1e-4, 1 - 1e-4)
            with torch.no_grad():
                self.head[-1].bias.copy_(torch.log(b / (1 - b)))
                self.head[-1].weight.mul_(0.01)

    def embed_images(self, images, view_id, mask):
        B, E = images.shape[:2]
        flat = images.reshape(B * E, *images.shape[2:])
        f = self.encoder(to_model_input(flat)).reshape(B, E, self.feat_dim)
        f = self.proj(torch.cat([f, self.view_emb(view_id)], dim=-1))
        f = f * mask.unsqueeze(-1)                    # padded rows carry no gradient
        return self.pool(f, mask)

    def forward(self, batch, return_attention: bool = False):
        parts, attn = [], None
        if self.use_image:
            emb, attn = self.embed_images(batch["images"], batch["view_id"], batch["mask"])
            parts += [self.img_norm(emb), batch["view_available"]]
        if self.use_clinical:
            parts.append(batch["clinical"])
        logits = self.head(torch.cat(parts, dim=-1))
        return (logits, attn) if return_attention else logits


def base_hazard_from(ds) -> np.ndarray:
    """Marginal discrete hazards under the observed TRAIN prevalence: events_k / at_risk_k."""
    at_risk = ds.at_risk.sum(axis=0)
    events = ds.target.sum(axis=0)
    return np.clip(events / np.maximum(at_risk, 1.0), 1e-4, 1 - 1e-4)


BASE_HAZARD = base_hazard_from(TRAIN_DS)
print("marginal train hazards h_k (events_k / at_risk_k):", np.round(BASE_HAZARD, 5).tolist())
print("implied 5-year risk under the marginal model:",
      round(float(risk_at_horizon(BASE_HAZARD[None, :], 1825.0)[0]), 4),
      "| observed train event fraction:", round(float(TRAIN_DS.event.mean()), 4))

In [ ]:
# ---- 10b. UNIT TESTS: masked attention, mode wiring, parameter count ----------------------------
torch.manual_seed(0)
_pool = MaskedAttentionPool(8).eval()
_x = torch.randn(1, 3, 8)
_m1 = torch.tensor([[True, False, False]])
_out1, _a1 = _pool(_x, _m1)
_out_single, _ = _pool(_x[:, :1], torch.tensor([[True]]))
print("--- masked attention pooling ---")
print(f"  one real view padded to 3 slots vs the same view unpadded: max |diff| = "
      f"{float((_out1-_out_single).abs().max()):.3e}   (must be 0 - padding is -inf pre-softmax)")
assert torch.allclose(_out1, _out_single, atol=1e-6)
print(f"  attention over the padded slots: {np.round(_a1.detach().numpy()[0], 6).tolist()} "
      f"(sums to {float(_a1.sum()):.6f})")
assert abs(float(_a1.sum()) - 1.0) < 1e-6 and float(_a1[0, 1:].abs().sum()) == 0.0
_m2 = torch.tensor([[True, True, False]])
_out2, _a2 = _pool(_x, _m2)
print(f"  adding a second real view changes the embedding: "
      f"{float((_out2-_out1).abs().max()):.4f} > 0")
assert float((_out2 - _out1).abs().max()) > 1e-6

print("\n--- mode wiring (no pretrained download in this check) ---")
for _mode, _expect_enc in (("fusion", True), ("image_only", True), ("clinical_only", False)):
    _net = SurvivalFusionNet(n_intervals=N_INTERVALS, n_clinical=len(DESIGN_COLUMNS),
                             mode=_mode, pretrained=False, base_hazard=BASE_HAZARD)
    _n = sum(p.numel() for p in _net.parameters())
    print(f"  {_mode:14s} encoder={_expect_enc}  clinical={_net.use_clinical}  "
          f"parameters={_n/1e6:.2f} M  head out={_net.head[-1].out_features}")
    assert hasattr(_net, "encoder") == _expect_enc and _net.head[-1].out_features == N_INTERVALS
    assert hasattr(_net, "img_norm") == _expect_enc
    if _mode == "clinical_only":
        _b = _net.head[-1].bias.detach().numpy()
        _h0 = 1 / (1 + np.exp(-_b))
        print(f"    head bias initialised at the marginal hazards: "
              f"{np.round(_h0, 5).tolist()}")
        assert np.allclose(_h0, BASE_HAZARD, atol=1e-5)
    del _net

## 11. The loss in torch, checked against the numpy reference

The training loss must be the same function as the one hand-checked in section 6. It is written
against the logits (`binary_cross_entropy_with_logits`, so the log-sigmoid is computed in the
stable form), reduced with the at-risk mask, and averaged over patients with the optional
prevalence weights. The cell asserts agreement with `dt_nll_numpy` on random input to 1e-6.

Patients with `n_scored == 0` (censored inside the first interval) contribute a zero row. They are
counted and printed rather than dropped silently, because "how many patients does the likelihood
actually see" is a number a reviewer will ask for.

In [ ]:
# ---- 11. Censoring-aware discrete-time NLL in torch ---------------------------------------------
def dt_nll_torch(logits, at_risk, target, sample_weight=None):
    """Same estimator as dt_nll_numpy, computed from logits for numerical stability."""
    ll = F.binary_cross_entropy_with_logits(logits.float(), target.float(), reduction="none")
    per_patient = (ll * at_risk.float()).sum(dim=1)
    if sample_weight is None:
        return per_patient.mean(), per_patient
    w = sample_weight.float()
    return (w * per_patient).sum() / w.sum().clamp_min(1e-12), per_patient


_rng = np.random.default_rng(7)
_lg = _rng.normal(size=(64, N_INTERVALS)) * 2.0
_t = _rng.uniform(0, 1826, size=64)
_e = (_rng.uniform(size=64) < 0.3).astype(int)
_ar_, _tg_, _, _ns_ = discretize_survival(_t, _e)
_h_ = 1.0 / (1.0 + np.exp(-_lg))
_np_mean, _np_per = dt_nll_numpy(_h_, _ar_, _tg_)
_pt_mean, _pt_per = dt_nll_torch(torch.tensor(_lg), torch.tensor(_ar_), torch.tensor(_tg_))
print("--- torch loss vs the hand-checked numpy reference (64 random patients) ---")
print(f"  mean   numpy={_np_mean:.10f}  torch={float(_pt_mean):.10f}  "
      f"|diff|={abs(_np_mean-float(_pt_mean)):.2e}")
print(f"  per-patient max |diff| = {float(np.abs(_np_per - _pt_per.numpy()).max()):.2e}")
assert abs(_np_mean - float(_pt_mean)) < 1e-6
assert float(np.abs(_np_per - _pt_per.numpy()).max()) < 1e-6
_w = torch.tensor(_rng.uniform(0.5, 2.0, size=64))
print(f"  weighted mean  numpy={dt_nll_numpy(_h_,_ar_,_tg_,_w.numpy())[0]:.10f}  "
      f"torch={float(dt_nll_torch(torch.tensor(_lg),torch.tensor(_ar_),torch.tensor(_tg_),_w)[0]):.10f}")
assert abs(dt_nll_numpy(_h_, _ar_, _tg_, _w.numpy())[0]
           - float(dt_nll_torch(torch.tensor(_lg), torch.tensor(_ar_), torch.tensor(_tg_), _w)[0])) < 1e-6
print(f"\n  patients the likelihood actually sees: train "
      f"{int((TRAIN_DS.n_scored > 0).sum())}/{len(TRAIN_DS)}, val "
      f"{int((VAL_DS.n_scored > 0).sum())}/{len(VAL_DS)}  "
      f"(the remainder were censored inside interval 0 and carry no scored interval)")
print(f"  scored intervals: train {int(TRAIN_DS.at_risk.sum())}, val {int(VAL_DS.at_risk.sum())}")

## 12. Training machinery — optimizer, schedule, and the resumable per-seed loop

This section defines how a seed is trained. **Nothing runs yet**: section 13 fires the one-batch
smoke test first, and only then trains the five seeds.

Protocol section 15, item by item:

| requirement | implementation |
| --- | --- |
| AdamW, pre-specified LR range, weight decay, gradient clipping, cosine decay | `model_image.optimizer/lr/weight_decay/grad_clip_norm/lr_schedule/warmup_epochs` — read from config, no search |
| natural prevalence retained in val and test | val/test loaders never resample; `EVENT_AWARE_MINIBATCH` (off by default) carries the exact inverse-probability correction into the loss |
| early stopping on validation NLL, fixed patience | `model_image.early_stopping.monitor == "val_nll"`, `patience == 8` |
| five pre-specified seeds, hazards averaged | `SEEDS`, `average_hazard` |
| report validation variability across seeds | printed per seed and summarised as mean ± SD |
| tuning on development data with a pre-specified budget | **no tuning at all.** The one hyper-parameter set in `config/feasibility.yaml` is used as written. With 373 training events, a search is a way of fitting the validation set. |

**Resumability.** Colab disconnects. Each seed writes `{tag}_seed{seed}.pt` to Drive after every
epoch and marks it `complete` when it early-stops or hits `max_epochs`. Re-running the cell skips a
complete seed and resumes an incomplete one from its last epoch, with the optimizer, scheduler, AMP
scaler and RNG states restored — so a resumed run is the run it would have been.

Each checkpoint also stores a hash of the frozen training contract (architecture, intervals, loss,
seeds, augmentation, LR, the design columns). If the contract changes, the checkpoint is refused
rather than silently reused.

**A note on cost.** The crops are 512x512 as protocol section 13 requires, and the encoder sees
every view of every patient, so one epoch is ~2,590 patients x 1.6 views = ~4,270 forward passes at
512^2. On an A100 that is a few minutes; on a T4 it will not fit at batch 32 — raise
`GRAD_ACCUM_STEPS` and lower `BATCH_SIZE_OVERRIDE` rather than downscaling the image, which would
be a section 13 deviation and would have to be recorded as one.

In [ ]:
# ---- 12a. Optimizer, schedule, train/eval steps -------------------------------------------------
assert MI["optimizer"] == "adamw" and MI["lr_schedule"] == "cosine"
assert MI["early_stopping"]["monitor"] == "val_nll"
BATCH_SIZE = int(BATCH_SIZE_OVERRIDE or MI["batch_size"])
MAX_EPOCHS = int(MAX_EPOCHS_OVERRIDE or MI["max_epochs"])
PATIENCE   = int(MI["early_stopping"]["patience"])
LR         = float(MI["lr"])
WD         = float(MI["weight_decay"])
CLIP       = float(MI["grad_clip_norm"])
WARMUP     = int(MI["warmup_epochs"])
print(f"batch={BATCH_SIZE} patients (x up to {MAX_ELEMS} views) | grad_accum={GRAD_ACCUM_STEPS} | "
      f"max_epochs={MAX_EPOCHS} | patience={PATIENCE} | lr={LR} wd={WD} clip={CLIP} warmup={WARMUP}")

TRAIN_CONTRACT = {
    "architecture": ARCH, "pretrained": MI["pretrained"], "shared_encoder": True,
    "view_embedding_dim": VIEW_EMB_DIM, "aggregator": MI["aggregator"],
    "n_intervals": N_INTERVALS, "interval_days": INTERVAL_DAYS, "loss": "censoring_aware_nll",
    "optimizer": "adamw", "lr": LR, "weight_decay": WD, "grad_clip_norm": CLIP,
    "lr_schedule": "cosine", "warmup_epochs": WARMUP, "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "augmentation": AUG_SPEC,
    "seeds": SEEDS, "design_columns": DESIGN_COLUMNS, "views": VIEWS,
    "event_aware_minibatch": bool(EVENT_AWARE_MINIBATCH),
}
CONTRACT_HASH = hashlib.sha256(json.dumps(TRAIN_CONTRACT, sort_keys=True).encode()).hexdigest()[:16]
print("training contract hash:", CONTRACT_HASH)


def make_scheduler(opt, steps_per_epoch):
    total = max(1, MAX_EPOCHS * steps_per_epoch)
    warm = max(1, WARMUP * steps_per_epoch)

    def f(step):
        if step < warm:
            return (step + 1) / warm
        p = (step - warm) / max(1, total - warm)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, p)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)


def to_device(batch):
    return {k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v)
            for k, v in batch.items()}


@torch.no_grad()
def predict_hazards(model, ds, batch_size=None) -> np.ndarray:
    """(n_patients, n_intervals) hazards in dataset order. No augmentation, no shuffling."""
    model.eval()
    loader = DataLoader(ds, batch_size=batch_size or BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
    out = np.zeros((len(ds), N_INTERVALS), dtype=np.float64)
    for batch in loader:
        b = to_device(batch)
        with torch.amp.autocast("cuda", enabled=(AMP and DEVICE.type == "cuda")):
            logits = model(b)
        out[b["idx"].cpu().numpy()] = torch.sigmoid(logits.float()).cpu().numpy()
    return out


def val_nll_of(hazards, ds) -> float:
    return dt_nll_numpy(hazards, ds.at_risk, ds.target)[0]


def train_one_epoch(model, loader, opt, sched, scaler, ds) -> float:
    model.train()
    total, n = 0.0, 0
    opt.zero_grad(set_to_none=True)
    for step, batch in enumerate(loader):
        b = to_device(batch)
        w = torch.as_tensor(ds.loss_weight[b["idx"].cpu().numpy()], dtype=torch.float32,
                            device=DEVICE)
        with torch.amp.autocast("cuda", enabled=(AMP and DEVICE.type == "cuda")):
            logits = model(b)
        loss, _ = dt_nll_torch(logits.float(), b["at_risk"], b["target"], w)
        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            sched.step()
        total += float(loss) * b["at_risk"].shape[0]; n += b["at_risk"].shape[0]
    return total / max(n, 1)

In [ ]:
# ---- 12b. Per-seed training with resume ----------------------------------------------------------
def ckpt_path(tag: str, seed: int) -> Path:
    return CKPT_DIR / f"{tag}_seed{seed}.pt"


def train_one_seed(tag: str, seed: int, *, mode: str, views_allowed=None, arch: str = ARCH,
                   train_ds=None, val_ds=None, design: str = "m0", verbose: bool = True) -> dict:
    """Train ONE seed. Skips a complete checkpoint; resumes an incomplete one."""
    path = ckpt_path(tag, seed)
    if path.exists() and not FORCE_RETRAIN:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        if ck.get("contract_hash") != CONTRACT_HASH:
            raise AssertionError(
                f"{path.name} was written under a different training contract "
                f"({ck.get('contract_hash')} != {CONTRACT_HASH}). Delete it deliberately or set "
                f"FORCE_RETRAIN, and record the change as a protocol deviation.")
        if ck.get("complete"):
            if verbose:
                print(f"  [{tag} seed {seed}] complete - best val NLL {ck['best_val_nll']:.6f} "
                      f"at epoch {ck['best_epoch']} (resumed from Drive, not retrained)")
            return ck
    else:
        ck = None

    seed_everything(seed)
    tr = train_ds if train_ds is not None else PatientViewDataset(
        "train", DEV_FRAME, DEV_X, train=True, views_allowed=views_allowed, design=design)
    va = val_ds if val_ds is not None else PatientViewDataset(
        "val", DEV_FRAME, DEV_X, train=False, views_allowed=views_allowed, design=design)
    assert tr.design == va.design == design, "train and val datasets use different designs"
    n_clinical = len(CLIN_STATS[design]["columns"])
    model = SurvivalFusionNet(n_intervals=N_INTERVALS, n_clinical=n_clinical, mode=mode,
                              arch=arch, pretrained=True,
                              base_hazard=base_hazard_from(tr)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    loader = make_loader(tr, BATCH_SIZE, shuffle=True, seed=seed,
                         event_aware=EVENT_AWARE_MINIBATCH)
    sched = make_scheduler(opt, max(1, math.ceil(len(loader) / GRAD_ACCUM_STEPS)))
    scaler = torch.amp.GradScaler("cuda", enabled=(AMP and DEVICE.type == "cuda"))
    start_epoch, best, best_epoch, bad, history = 0, float("inf"), -1, 0, []
    best_state = ck["best_model"] if (ck is not None and "best_model" in ck) else None
    if ck is not None:
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["optimizer"])
        sched.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
        torch.set_rng_state(ck["rng_torch"]); np.random.set_state(ck["rng_numpy"])
        random.setstate(ck["rng_python"])
        start_epoch, best, best_epoch, bad = ck["epoch"] + 1, ck["best_val_nll"], ck["best_epoch"], ck["bad"]
        history = list(ck["history"])
        if verbose:
            print(f"  [{tag} seed {seed}] resuming at epoch {start_epoch} "
                  f"(best val NLL so far {best:.6f})")

    for epoch in range(start_epoch, MAX_EPOCHS):
        t0 = time.time()
        tr_loss = train_one_epoch(model, loader, opt, sched, scaler, tr)
        vh = predict_hazards(model, va)
        v_nll = val_nll_of(vh, va)
        improved = v_nll < best - 1e-6
        if improved:
            best, best_epoch, bad = v_nll, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if best_state is None:              # only possible if the very first epoch is NaN
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        history.append({"epoch": epoch, "train_nll": tr_loss, "val_nll": v_nll,
                        "lr": sched.get_last_lr()[0], "secs": time.time() - t0})
        if verbose:
            print(f"    epoch {epoch:2d}  train NLL {tr_loss:.6f}  val NLL {v_nll:.6f}"
                  f"{'  *' if improved else ''}  lr {sched.get_last_lr()[0]:.2e}  "
                  f"{time.time()-t0:.0f}s  bad={bad}")
        ck = {"model": model.state_dict(), "best_model": best_state,
              "optimizer": opt.state_dict(), "scheduler": sched.state_dict(),
              "scaler": scaler.state_dict(), "epoch": epoch, "best_val_nll": best,
              "best_epoch": best_epoch, "bad": bad, "history": history,
              "complete": bool(bad >= PATIENCE or epoch == MAX_EPOCHS - 1),
              "contract_hash": CONTRACT_HASH, "tag": tag, "seed": seed, "mode": mode,
              "arch": arch, "views_allowed": list(views_allowed or VIEWS),
              "design": design, "n_clinical": n_clinical,
              "rng_torch": torch.get_rng_state(), "rng_numpy": np.random.get_state(),
              "rng_python": random.getstate()}
        torch.save(ck, path)
        if bad >= PATIENCE:
            if verbose:
                print(f"    early stop: val NLL has not improved for {PATIENCE} epochs "
                      f"(best {best:.6f} at epoch {best_epoch})")
            break
    del model, opt, loader
    torch.cuda.empty_cache() if DEVICE.type == "cuda" else None
    return ck


def load_seed_model(tag: str, seed: int, *, mode: str, arch: str = ARCH, design: str = "m0"):
    ck = torch.load(ckpt_path(tag, seed), map_location="cpu", weights_only=False)
    assert ck.get("complete"), f"{tag} seed {seed} is not finished"
    assert ck["contract_hash"] == CONTRACT_HASH
    assert ck.get("design", "m0") == design, f"{tag} seed {seed} was trained on a different design"
    net = SurvivalFusionNet(n_intervals=N_INTERVALS,
                            n_clinical=len(CLIN_STATS[design]["columns"]), mode=mode,
                            arch=arch, pretrained=False)
    net.load_state_dict(ck["best_model"])
    return net.to(DEVICE).eval(), ck

## 13. Smoke test, then the full run

The plan's Track C verification list names this explicitly: *"a 1-batch forward-pass smoke test in
Colab before the full run"*. It checks the shapes, that the loss is finite and near the marginal
likelihood rather than a wild number, that gradients reach the view embedding, the attention scorer
and the head (i.e. the graph really is connected end to end), and that a padded one-view patient in
the same batch as a three-view patient does not corrupt either.

Set `SMOKE_ONLY = True` in section 0 to stop after the smoke test. The cell after it is the one
that spends GPU hours.

**The five arms.** All five come from the same `SurvivalFusionNet` class with different switches, so
the protocol section 24 comparisons differ only in what is turned off, never in code path:
`m4_fusion` (primary), `m3_image`, `m2_frontal`, `m0d_clinical` and `m4_frontal`.

In [ ]:
# ---- 13. One-batch forward/backward smoke test ---------------------------------------------------
seed_everything(SEEDS[0])
_smoke = SurvivalFusionNet(n_intervals=N_INTERVALS, n_clinical=len(DESIGN_COLUMNS),
                           mode="fusion", pretrained=True, base_hazard=BASE_HAZARD).to(DEVICE)
_loader = make_loader(TRAIN_DS, min(BATCH_SIZE, 8), shuffle=True, seed=SEEDS[0])
_batch = to_device(next(iter(_loader)))
print("batch shapes:", {k: tuple(v.shape) for k, v in _batch.items() if torch.is_tensor(v)})
print("views per patient in this batch:", _batch["mask"].sum(1).tolist())
print("view_available rows (frontal, lateral, sunrise):",
      _batch["view_available"].cpu().numpy().astype(int).tolist())

_t0 = time.time()
with torch.amp.autocast("cuda", enabled=(AMP and DEVICE.type == "cuda")):
    _logits, _attn = _smoke(_batch, return_attention=True)
_loss, _per = dt_nll_torch(_logits.float(), _batch["at_risk"], _batch["target"])
_loss.backward()
_fwd = time.time() - _t0
print(f"\nlogits {tuple(_logits.shape)} | attention {tuple(_attn.shape)} | "
      f"loss = {float(_loss):.6f} | forward+backward {_fwd:.2f}s")
assert _logits.shape == (_batch["at_risk"].shape[0], N_INTERVALS)
assert torch.isfinite(_loss) and float(_loss) > 0
_ref = dt_nll_numpy(np.tile(BASE_HAZARD, (len(_per), 1)),
                    _batch["at_risk"].cpu().numpy(), _batch["target"].cpu().numpy())[0]
print(f"loss under the marginal-hazard model on the same batch = {_ref:.6f} "
      f"(the initialised head should start near this)")
_grads = {n: float(p.grad.abs().sum()) for n, p in _smoke.named_parameters()
          if p.grad is not None and ("view_emb" in n or n.endswith("head.4.weight")
                                     or "pool.w" in n)}
print("gradient reaches:", {k: f"{v:.3e}" for k, v in _grads.items()})
assert all(v > 0 for v in _grads.values()), "some branch is disconnected from the loss"
assert all(torch.isfinite(p.grad).all() for p in _smoke.parameters() if p.grad is not None)
_a = _attn.detach().float().cpu().numpy()
_m = _batch["mask"].cpu().numpy()
print(f"attention sums to 1 on every row: {bool(np.allclose(_a.sum(1), 1.0, atol=1e-5))}; "
      f"mass on padded slots: {float(np.abs(_a * ~_m).sum()):.3e}")
assert np.allclose(_a.sum(1), 1.0, atol=1e-5) and float(np.abs(_a * ~_m).sum()) < 1e-8
del _smoke, _batch, _logits, _attn, _loss
torch.cuda.empty_cache() if DEVICE.type == "cuda" else None
print("\nSMOKE TEST PASSED")
if SMOKE_ONLY:
    raise SystemExit("SMOKE_ONLY is set - stopping before the full run.")

In [ ]:
# ---- 13b. Run the five seeds for every development arm (only after the smoke test) ---------------
ARMS = {
    "m4_fusion":       dict(mode="fusion",        views_allowed=VIEWS,       label="M4 multimodal (PRIMARY)"),
    "m3_image":        dict(mode="image_only",    views_allowed=VIEWS,       label="M3 multi-view image"),
    "m2_frontal":      dict(mode="image_only",    views_allowed=["frontal"], label="M2 frontal-only image"),
    "m0d_clinical":    dict(mode="clinical_only", views_allowed=VIEWS,       label="clinical-only discrete-time"),
    "m4_frontal":      dict(mode="fusion",        views_allowed=["frontal"], label="M4 restricted to frontal (section 24)"),
    # Protocol Table 7 M1 ("M0 plus inferred KLG"), on the KLG-eligible subset only
    # (Secondary objective 2). It is a SUBSET arm: it scores fewer patients than every other
    # arm, so it is deliberately kept OUT of the primary common comparison set and gets its
    # own eligible-subset comparison in section 16. The primary stays M4 versus M0.
    "m1_klg":          dict(mode="clinical_only", views_allowed=VIEWS,       design="m1",
                            label="M1 inferred KLG + clinical, KLG-eligible subset (Secondary objective 2)"),
}
# Arms whose patient set is a strict subset of the cohort. Nothing that has to be paired across
# ALL models may include one of these, or the paired set collapses to the subset.
SUBSET_ARMS = {"m1_klg"}


def arm_frames(spec):
    """The (frame, design matrix) an arm trains and scores on — M0's, or M1's eligible subset."""
    return (DEV_FRAME_M1, DEV_X_M1) if spec.get("design", "m0") == "m1" else (DEV_FRAME, DEV_X)


SEED_RESULTS = defaultdict(dict)
for tag, spec in ARMS.items():
    _fr, _X = arm_frames(spec)
    _dz = spec.get("design", "m0")
    print(f"\n=== {tag}: {spec['label']}  (mode={spec['mode']}, views={spec['views_allowed']}, "
          f"design={_dz}) ===")
    tr_ds = PatientViewDataset("train", _fr, _X, train=True,
                               views_allowed=spec["views_allowed"], design=_dz)
    va_ds = PatientViewDataset("val", _fr, _X, train=False,
                               views_allowed=spec["views_allowed"], design=_dz)
    print(f"  patients: train {len(tr_ds)} ({int(tr_ds.event.sum())} events), "
          f"val {len(va_ds)} ({int(va_ds.event.sum())} events)"
          + (f"  [{tr_ds.n_no_image + va_ds.n_no_image} patients have no view in this arm]"
             if spec["views_allowed"] != VIEWS else "")
          + ("  [KLG-eligible patients only — protocol Secondary objective 2]"
             if tag in SUBSET_ARMS else ""))
    for seed in SEEDS:
        ck = train_one_seed(tag, seed, mode=spec["mode"], views_allowed=spec["views_allowed"],
                            train_ds=tr_ds, val_ds=va_ds, design=_dz)
        SEED_RESULTS[tag][seed] = {"best_val_nll": ck["best_val_nll"],
                                   "best_epoch": ck["best_epoch"],
                                   "n_epochs": len(ck["history"]),
                                   "complete": bool(ck["complete"])}
    v = np.array([SEED_RESULTS[tag][s]["best_val_nll"] for s in SEEDS])
    print(f"  across-seed validation NLL: mean {v.mean():.6f}  SD {v.std(ddof=1):.6f}  "
          f"range [{v.min():.6f}, {v.max():.6f}]")

print("\n--- across-seed variability (protocol section 15) ---")
_rows = []
for tag in ARMS:
    v = np.array([SEED_RESULTS[tag][s]["best_val_nll"] for s in SEEDS])
    ep = [SEED_RESULTS[tag][s]["best_epoch"] for s in SEEDS]
    _rows.append({"arm": tag, "label": ARMS[tag]["label"], "val_nll_mean": v.mean(),
                  "val_nll_sd": v.std(ddof=1), "val_nll_min": v.min(), "val_nll_max": v.max(),
                  "best_epochs": str(ep)})
SEED_TABLE = pd.DataFrame(_rows)
SEED_TABLE.to_csv(RESULTS_DIR / "seed_variability.csv", index=False)
print(SEED_TABLE.to_string(index=False))

## 14. Ensemble, recalibration on validation, and the FREEZE

Three things happen here and then nothing about the model may change again.

**1. Ensemble.** Per-interval hazards are averaged across the five seeds (`average_hazard`), then
converted to survival and horizon risks once. The section 7 check showed why the order matters.

**2. Horizon-specific recalibration, fitted on validation only.** Protocol section 15: *"fit a
single horizon-specific recalibration model on the validation set if required. Freeze it before
test evaluation."* The model is `cloglog(P) = a + b * cloglog(p_hat)`, IPCW-weighted at that
horizon — the same cloglog recalibration M0 is assessed with, so "slope" means the same thing for
both. One `(a, b)` pair per horizon per arm, fitted on 371 validation patients with 54 events.

With 54 validation events, a recalibration slope is estimated with a wide interval. That is a real
limitation and it is recorded in the manifest rather than hidden: the frozen slope is applied to
test as-is, and the *test* calibration slope after recalibration is reported as the honest measure.

**3. Operating point.** Protocol Table 7: *"One threshold selected in validation data to prioritize
sensitivity."* The threshold is the lowest recalibrated risk at the primary horizon that reaches
`TARGET_SENSITIVITY` on validation. Its test sensitivity, specificity, PPV, NPV and alert rate are
reported — not re-selected.

Then `FROZEN_MANIFEST.json` is written to Drive: the architecture, the interval grid, the ensemble
rule, every recalibration coefficient, the threshold, the training-contract hash, the environment,
and a **hash of the analysis functions themselves** (`inspect.getsource` over the exact functions the
sealed cell calls). Protocol section 12 requires the analysis script to be frozen before the test
set is read; this is the machine-checkable form of that claim.

### Which comparison is *the* primary one

Protocol Table 7 pre-specifies **M4 versus M0, IPCW cumulative/dynamic AUROC, at 5 years**, with a
paired patient-level bootstrap and 2,000 replicates, and protocol section 18 limits primary
inference to **one** comparison. That is what is declared primary here.

`config/feasibility.yaml` calls the 2-year horizon "co-primary", and the plan's non-negotiables
call it co-primary too, because the 5-year outcome status is **determined** for only 1,401 of the
3,709 cohort patients (37.8%) — 533 observed events plus 868 patients censored administratively at
day 1826 — and for 1,133 of the 2,968 development patients (38.2%). That is
`n_status_determined_5y`, and it is the figure every document in this project uses when follow-up
maturity is invoked; `outputs/sample_size.md` reconciles it against the two counts it must not be
confused with (`n_full_5y_record_coverage` = 916 cohort / 746 development, and
`n_followup_reaches_day_1825` = 869 cohort / 707 development). No metric changes with the choice —
IPCW handles administrative censoring correctly — but the argument has to rest on one number.
Two co-primary endpoints
and "primary inference is limited to one comparison" cannot both hold, so the tension is resolved
the only way that does not invent authority: the 5-year comparison in the protocol table is the
single primary inference, and the 2-year comparison is reported in the **first secondary family
under FDR control**. Both are computed from the same bootstrap replicates, so nothing is lost — but
only one carries an unadjusted claim. **A reviewer should be told this explicitly.**

In [ ]:
# ---- 14a. Ensemble the five seeds on validation ---------------------------------------------------
def ensemble_hazards(tag: str, ds, *, mode: str, arch: str = ARCH,
                     design: str = "m0") -> tuple[np.ndarray, list]:
    """Average per-interval hazards across the five seeds (config ensemble == 'average_hazard')."""
    per_seed = []
    for seed in SEEDS:
        net, _ = load_seed_model(tag, seed, mode=mode, arch=arch, design=design)
        per_seed.append(predict_hazards(net, ds))
        del net
        torch.cuda.empty_cache() if DEVICE.type == "cuda" else None
    return average_hazard(per_seed), per_seed


def risk_score(hazards) -> np.ndarray:
    """Horizon-free ranking score for Harrell C: total cumulative hazard -log S(1826)."""
    return -np.log(np.clip(1.0 - np.asarray(hazards, float), 1e-12, 1.0)).sum(axis=1)


VAL_DS_BY_ARM, VAL_HAZ, VAL_HAZ_SEEDS = {}, {}, {}
for tag, spec in ARMS.items():
    _fr, _X = arm_frames(spec)
    ds = PatientViewDataset("val", _fr, _X, train=False,
                            views_allowed=spec["views_allowed"], design=spec.get("design", "m0"))
    h, per_seed = ensemble_hazards(tag, ds, mode=spec["mode"], design=spec.get("design", "m0"))
    VAL_DS_BY_ARM[tag], VAL_HAZ[tag], VAL_HAZ_SEEDS[tag] = ds, h, per_seed
    nlls = [dt_nll_numpy(s, ds.at_risk, ds.target)[0] for s in per_seed]
    print(f"{tag:14s} val NLL per seed {np.round(nlls,4).tolist()} -> ensemble "
          f"{dt_nll_numpy(h, ds.at_risk, ds.target)[0]:.6f}")

# M0 on exactly the validation patients each arm scores, so every comparison is paired. Indexed
# by PATIENT ID, not by row position: the m1_klg arm scores a different (smaller) frame, and a
# positional index into DEV_FRAME would silently line up the wrong patients.
_M0_VAL_POS = {str(p): i for i, p in
               enumerate(DEV_FRAME.loc[DEV_FRAME["split"] == "val", "empi_anon"].astype(str))}
_M0_VAL_RISK_ALL = {h: M0_RISK_DEV[h][(DEV_FRAME['split'] == 'val').to_numpy()] for h in HORIZONS}
_M0_VAL_LP_ALL = M0_LP_DEV[(DEV_FRAME['split'] == 'val').to_numpy()]


def m0_val_index(ds) -> np.ndarray:
    return np.array([_M0_VAL_POS[str(p)] for p in ds.pids], dtype=int)


M0_VAL_RISK = {tag: {h: _M0_VAL_RISK_ALL[h][m0_val_index(ds)] for h in HORIZONS}
               for tag, ds in VAL_DS_BY_ARM.items()}
M0_VAL_LP = {tag: _M0_VAL_LP_ALL[m0_val_index(ds)] for tag, ds in VAL_DS_BY_ARM.items()}

print("\n--- validation discrimination (ensemble, natural prevalence, train-G IPCW weights) ---")
_rows = []
for tag, ds in VAL_DS_BY_ARM.items():
    r = {"arm": tag, "n": len(ds), "events": int(ds.event.sum()),
         "val_nll": dt_nll_numpy(VAL_HAZ[tag], ds.at_risk, ds.target)[0],
         "harrell_c": harrell_c(ds.time, ds.event, risk_score(VAL_HAZ[tag])),
         "m0_harrell_c": harrell_c(ds.time, ds.event, M0_VAL_LP[tag])}
    for h in HORIZONS:
        y, w = ipcw_labels_weights(ds.time, ds.event, h)
        r[f"auc@{int(h)}"] = ipcw_auc(y, w, risk_at_horizon(VAL_HAZ[tag], h))
        r[f"m0_auc@{int(h)}"] = ipcw_auc(y, w, M0_VAL_RISK[tag][h])
    _rows.append(r)
VAL_TABLE = pd.DataFrame(_rows)
VAL_TABLE.to_csv(RESULTS_DIR / "val_metrics.csv", index=False)
print(VAL_TABLE.round(4).to_string(index=False))
print(f"\nM0 (Cox) validation reference on the full 371: Harrell C {M0['val_metrics']['cindex']:.4f}, "
      f"AUC@730 {M0['val_metrics']['auc@730.0']:.4f}, AUC@1825 {M0['val_metrics']['auc@1825.0']:.4f}")
print("The m0_auc@* columns re-evaluate the SAME Cox model on each arm's patient subset, which is "
      "the paired reference the difference is taken against.")
print("m1_klg is scored on the KLG-eligible patients only (protocol Secondary objective 2), so "
      "its row is NOT comparable to the others' by level — only to the M0 reference on the same "
      "rows. It is excluded from the primary common test set in section 16.")

In [ ]:
# ---- 14b. Horizon-specific recalibration, fitted on VALIDATION only -------------------------------
TARGET_SENSITIVITY = 0.90        # protocol Table 7: the operating point prioritises sensitivity
PRIMARY_ARM = "m4_fusion"
PRIMARY_HORIZON_DAYS = 1825.0    # protocol Table 7. See the markdown above on the 2-year tension.
SECONDARY_HORIZONS = [h for h in HORIZONS if h != PRIMARY_HORIZON_DAYS]
assert PRIMARY_HORIZON_DAYS in HORIZONS


def fit_recalibration(hazards, ds) -> dict:
    """One (a, b) per horizon from the validation set. Applied unchanged to test."""
    out = {}
    for h in HORIZONS:
        y, w = ipcw_labels_weights(ds.time, ds.event, h)
        p = risk_at_horizon(hazards, h)
        b, a = calibration_slope_intercept(y, w, p)
        if not (np.isfinite(a) and np.isfinite(b)):
            a, b = 0.0, 1.0                       # identity; recorded, never silent
            note = "GLM did not converge on 54 validation events - identity recalibration frozen"
        else:
            note = ""
        out[str(h)] = {"intercept": float(a), "slope": float(b), "note": note,
                       "n_cases": int((y == 1).sum()), "n_controls": int((y == 0).sum())}
    return out


def apply_recalibration(p, recal_h: dict) -> np.ndarray:
    """P_recal = inv_cloglog(a + b * cloglog(p_hat)); the frozen horizon-specific transform."""
    return inv_cloglog(float(recal_h["intercept"]) + float(recal_h["slope"]) * cloglog(p))


RECAL = {tag: fit_recalibration(VAL_HAZ[tag], VAL_DS_BY_ARM[tag]) for tag in ARMS}
print("--- frozen recalibration (validation only) ---")
for tag in ARMS:
    for h in HORIZONS:
        r = RECAL[tag][str(h)]
        print(f"  {tag:14s} @{int(h):4d}d  intercept {r['intercept']:+.4f}  slope {r['slope']:+.4f}"
              f"  (cases {r['n_cases']}, controls {r['n_controls']}) {r['note']}")

_ds = VAL_DS_BY_ARM[PRIMARY_ARM]
_p_val = apply_recalibration(risk_at_horizon(VAL_HAZ[PRIMARY_ARM], PRIMARY_HORIZON_DAYS),
                             RECAL[PRIMARY_ARM][str(PRIMARY_HORIZON_DAYS)])
_y, _w = ipcw_labels_weights(_ds.time, _ds.event, PRIMARY_HORIZON_DAYS)
_usable = _y >= 0
_cands = np.unique(np.round(_p_val[_usable], 6))
THRESHOLD = None
for thr in np.sort(_cands):
    sens = float(((_p_val >= thr) & (_y == 1)).sum() / max((_y == 1).sum(), 1))
    if sens >= TARGET_SENSITIVITY:
        THRESHOLD = float(thr)
    else:
        break
assert THRESHOLD is not None, "no validation threshold reaches the target sensitivity"
VAL_SENS_AT_THRESHOLD = float(((_p_val >= THRESHOLD) & (_y == 1)).sum() / max((_y == 1).sum(), 1))
VAL_SPEC_AT_THRESHOLD = float(((_p_val < THRESHOLD) & (_y == 0)).sum() / max((_y == 0).sum(), 1))
print(f"\n--- frozen operating point (validation, {PRIMARY_ARM} @ {int(PRIMARY_HORIZON_DAYS)}d) ---")
print(f"  threshold {THRESHOLD:.6f}  ->  val sensitivity {VAL_SENS_AT_THRESHOLD:.3f} "
      f"(target {TARGET_SENSITIVITY}), specificity {VAL_SPEC_AT_THRESHOLD:.3f}, "
      f"alert rate {float((_p_val >= THRESHOLD).mean()):.3f}")
print("  This threshold is NOT re-selected on test. Its test operating characteristics are "
      "whatever they turn out to be.")

In [ ]:
# ---- 14c. FREEZE ----------------------------------------------------------------------------------
BOOTSTRAP_N = 2000               # protocol Table 7. model_clinical.bootstrap_n (500) is a
                                 # development value and is deliberately NOT inherited.
assert BOOTSTRAP_N == 2000 and int(MC["bootstrap_n"]) == 500

ANALYSIS_FUNCTIONS = [discretize_survival, dt_nll_numpy, hazards_to_survival, risk_at_horizon,
                      average_hazard, step_value, ipcw_labels_weights, ipcw_auc, harrell_c,
                      uno_c, ipcw_auprc, ipcw_brier, cloglog, inv_cloglog,
                      calibration_slope_intercept,
                      apply_imputer, spline_basis, replay_from_json, build_clinical_design,
                      fit_recalibration, apply_recalibration, ensemble_hazards, risk_score,
                      predict_hazards]
ANALYSIS_SRC = "\n".join(inspect.getsource(f) for f in ANALYSIS_FUNCTIONS)
ANALYSIS_HASH = hashlib.sha256(ANALYSIS_SRC.encode()).hexdigest()

FROZEN_MANIFEST = {
    "frozen_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "protocol_sections": [12, 13, 14, 15, 16, 17, 18, 20, 21, 22, 24, 25],
    "model": {
        "architecture": ARCH, "init": "imagenet", "shared_weights_across_views": True,
        "view_embedding_dim": VIEW_EMB_DIM, "aggregator": "masked_gated_attention",
        "survival_head": {"kind": "discrete_time", "n_intervals": N_INTERVALS,
                          "interval_days": INTERVAL_DAYS, "edges": EDGES.tolist(),
                          "loss": "censoring_aware_discrete_time_nll"},
        "clinical_input": {"design_columns": DESIGN_COLUMNS,
                           "centering_means": {c: float(v) for c, v in
                                               zip(DESIGN_COLUMNS, CLIN_MEAN)},
                           "train_sd": {c: float(v) for c, v in zip(DESIGN_COLUMNS, CLIN_STD)},
                           "source": "m0_clinical_model.json, replayed not refitted"},
        "clinical_input_m1": {"design_columns": M1_DESIGN_COLUMNS,
                              "centering_means": {c: float(v) for c, v in
                                                  zip(M1_DESIGN_COLUMNS, CLIN_STATS["m1"]["mean"])},
                              "train_sd": {c: float(v) for c, v in
                                           zip(M1_DESIGN_COLUMNS, CLIN_STATS["m1"]["std"])},
                              "eligibility": M1_ELIGIBILITY,
                              "source": "m1_klg_model.json, replayed not refitted; protocol "
                                        "Table 7 M1 on the KLG-eligible subset only"},
        "subset_arms": sorted(SUBSET_ARMS),
        "arms": {t: {"mode": s["mode"], "views": s["views_allowed"], "label": s["label"]}
                 for t, s in ARMS.items()},
    },
    "training": TRAIN_CONTRACT,
    "training_contract_hash": CONTRACT_HASH,
    "ensemble": {"rule": "average_hazard", "seeds": SEEDS,
                 "note": "per-interval hazards averaged across seeds, then converted once"},
    "seed_variability_val_nll": {t: {"mean": float(np.mean([SEED_RESULTS[t][s]['best_val_nll'] for s in SEEDS])),
                                     "sd": float(np.std([SEED_RESULTS[t][s]['best_val_nll'] for s in SEEDS], ddof=1))}
                                 for t in ARMS},
    "recalibration": {"form": "cloglog(P) = intercept + slope * cloglog(p_hat)",
                      "fitted_on": "validation only (371 patients, 54 events)",
                      "coefficients": RECAL},
    "operating_point": {"arm": PRIMARY_ARM, "horizon_days": PRIMARY_HORIZON_DAYS,
                        "rule": "lowest validation risk reaching the target sensitivity",
                        "target_sensitivity": TARGET_SENSITIVITY, "threshold": THRESHOLD,
                        "val_sensitivity": VAL_SENS_AT_THRESHOLD,
                        "val_specificity": VAL_SPEC_AT_THRESHOLD},
    "evaluation": {
        "ipcw_censoring_curve": "train reverse-KM from m0_clinical_model.json (frozen; identical "
                                "weights for every model, so the paired difference is clean)",
        "horizons_days": HORIZONS,
        "primary_comparison": f"{PRIMARY_ARM} vs M0_clinical_penalized_cox",
        "primary_metric": "ipcw_cumulative_dynamic_auroc",
        "primary_horizon_days": PRIMARY_HORIZON_DAYS,
        "primary_estimand": "paired difference in IPCW C/D AUROC (fusion minus M0) on the same "
                            "test patients, patient-level bootstrap percentile CI",
        "bootstrap_replicates": BOOTSTRAP_N,
        "bootstrap_note": "protocol Table 7 pre-specifies 2,000; model_clinical.bootstrap_n = 500 "
                          "is a development value and is not inherited",
        "secondary_families": {
            "horizons": [f"{PRIMARY_ARM} vs M0 at {int(h)} d" for h in SECONDARY_HORIZONS],
            "modality": ["m3_image vs m0d_clinical", "m4_fusion vs m3_image",
                         "m4_fusion vs m0d_clinical"],
            "views": ["m4_fusion vs m4_frontal", "m3_image vs m2_frontal"],
            "secondary_objective_2": [f"{a} vs m1_klg on the KLG-eligible test subset"
                                      for a in ("m2_frontal", "m3_image", "m4_fusion")],
        },
        "secondary_objective_2_note": "protocol Secondary objective 2 compares raw-image "
                                      "prediction with an inferred-KLG-plus-clinical comparator "
                                      "IN THE SUBSET WITH ELIGIBLE BILATERAL FRONTAL IMAGES, so "
                                      "this family is evaluated on its own eligible common set "
                                      "with its own bootstrap replicates. It is never pooled "
                                      "with the primary comparison, which stays M4 vs M0 on the "
                                      "full common set.",
        "multiplicity": "primary inference limited to ONE comparison (protocol section 18); every "
                        "secondary comparison is Benjamini-Hochberg FDR-controlled WITHIN its "
                        "declared family",
        "subgroup_suppression": {"suppress_below_events": 50, "emphasise_ci_below_events": 100},
    },
    "analysis_script_sha256": ANALYSIS_HASH,
    "analysis_functions": [f.__name__ for f in ANALYSIS_FUNCTIONS],
    "environment": ENVIRONMENT,
    "data": {"shards": {k: v["n_shards"] for k, v in SHARD_REPORT.items()},
             "cohort": FROZEN_COHORT, "kept_images": FROZEN_IMAGES,
             "labels_csv_sha256": hashlib.sha256(LABELS_CSV.read_bytes()).hexdigest(),
             "m0_json_sha256": hashlib.sha256(M0_JSON.read_bytes()).hexdigest()},
}
MANIFEST_PATH = RESULTS_DIR / "FROZEN_MANIFEST.json"
MANIFEST_PATH.write_text(json.dumps(FROZEN_MANIFEST, indent=2, default=str))

print("=" * 96)
print("FROZEN".center(96))
print("=" * 96)
print(f"  model                 {ARCH}, ImageNet init, shared across {len(VIEWS)} views, "
      f"masked attention aggregator")
print(f"  survival head         discrete time, {N_INTERVALS} x {INTERVAL_DAYS:.1f} d, "
      f"censoring-aware NLL")
print(f"  ensemble rule         average_hazard over seeds {SEEDS}")
print(f"  recalibration         cloglog(P) = a + b*cloglog(p_hat), per horizon, fitted on val, "
      f"{len(RECAL)} arms x {len(HORIZONS)} horizons")
print(f"  operating threshold   {THRESHOLD:.6f} at {int(PRIMARY_HORIZON_DAYS)} d "
      f"(val sensitivity {VAL_SENS_AT_THRESHOLD:.3f})")
print(f"  primary comparison    {PRIMARY_ARM} vs M0, IPCW C/D AUROC @ {int(PRIMARY_HORIZON_DAYS)} d, "
      f"paired difference, {BOOTSTRAP_N} replicates")
print(f"  training contract     {CONTRACT_HASH}")
print(f"  analysis script       sha256 {ANALYSIS_HASH}")
print(f"  written to            {MANIFEST_PATH}")
print("=" * 96)
print("Nothing about the model, the ensemble rule, the recalibration or the threshold may change\n"
      "from this point on. Any change requires a dated entry in outputs/protocol_deviations.md\n"
      "and invalidates the sealed-test read.")

## 15. Protocol section 25 — the DenseNet121 robustness model

> *"Train a simpler DenseNet121 frontal-image model as a single robustness architecture **after the
> primary model is locked**."*

That ordering is the whole point, so this section sits **after** section 14's freeze and **before**
the sealed test read. It is trained on the same training patients, selected on the same validation
set with the same early-stopping rule, and recalibrated on validation exactly like every other arm.
Nothing it produces can change the primary model — the manifest is already written and its hash is
checked in the sealed cell.

Putting it here rather than after the test read means the locked test set is opened **once**, for
every locked model at the same time, which is what protocol section 12 asks for. Training it after
the test read would need a second read.

It is a **robustness check, not a competitor.** If DenseNet121 beats ConvNeXt-Tiny, that is
reported as instability of the architecture choice, not as a new primary model. Swapping it in
would be exactly the post hoc model selection protocol section 27 lists as a recurring failure.

The other section 25 items — self-supervised pretraining, repeated grouped cross-validation,
performance by image-to-index interval and calendar era — are out of scope for this notebook and
are recorded as such at the end.

In [ ]:
# ---- 15. DenseNet121 frontal-only robustness model (trained AFTER the freeze) ---------------------
ROBUST_ARCH = str(MI["robustness_model"])
assert ROBUST_ARCH == "densenet121"
assert MANIFEST_PATH.exists(), "the primary model must be frozen before the robustness model runs"

ARMS["r1_densenet_frontal"] = dict(mode="image_only", views_allowed=["frontal"],
                                   label=f"{ROBUST_ARCH} frontal-only (section 25 robustness)",
                                   arch=ROBUST_ARCH)
_tr = PatientViewDataset("train", DEV_FRAME, DEV_X, train=True, views_allowed=["frontal"])
_va = PatientViewDataset("val", DEV_FRAME, DEV_X, train=False, views_allowed=["frontal"])
print(f"=== r1_densenet_frontal: {ROBUST_ARCH}, frontal only ===")
for seed in SEEDS:
    ck = train_one_seed("r1_densenet_frontal", seed, mode="image_only",
                        views_allowed=["frontal"], arch=ROBUST_ARCH, train_ds=_tr, val_ds=_va)
    SEED_RESULTS["r1_densenet_frontal"][seed] = {"best_val_nll": ck["best_val_nll"],
                                                 "best_epoch": ck["best_epoch"],
                                                 "n_epochs": len(ck["history"]),
                                                 "complete": bool(ck["complete"])}
_v = np.array([SEED_RESULTS["r1_densenet_frontal"][s]["best_val_nll"] for s in SEEDS])
print(f"  across-seed validation NLL: mean {_v.mean():.6f}  SD {_v.std(ddof=1):.6f}")

VAL_DS_BY_ARM["r1_densenet_frontal"] = _va
VAL_HAZ["r1_densenet_frontal"], VAL_HAZ_SEEDS["r1_densenet_frontal"] = ensemble_hazards(
    "r1_densenet_frontal", _va, mode="image_only", arch=ROBUST_ARCH)
RECAL["r1_densenet_frontal"] = fit_recalibration(VAL_HAZ["r1_densenet_frontal"], _va)
M0_VAL_RISK["r1_densenet_frontal"] = {h: _M0_VAL_RISK_ALL[h][m0_val_index(_va)] for h in HORIZONS}
M0_VAL_LP["r1_densenet_frontal"] = _M0_VAL_LP_ALL[m0_val_index(_va)]
print(f"  val NLL (ensemble) {dt_nll_numpy(VAL_HAZ['r1_densenet_frontal'], _va.at_risk, _va.target)[0]:.6f} "
      f"| ConvNeXt frontal-only {dt_nll_numpy(VAL_HAZ['m2_frontal'], VAL_DS_BY_ARM['m2_frontal'].at_risk, VAL_DS_BY_ARM['m2_frontal'].target)[0]:.6f}")

FROZEN_MANIFEST["robustness"] = {
    "architecture": ROBUST_ARCH, "views": ["frontal"], "trained_after_freeze": True,
    "seeds": SEEDS, "recalibration": RECAL["r1_densenet_frontal"],
    "val_nll_mean": float(_v.mean()), "val_nll_sd": float(_v.std(ddof=1)),
    "role": "robustness check only; it can never replace the frozen primary model",
    "section_25_items_not_run": ["self-supervised pretraining vs ImageNet init",
                                 "repeated 5-fold grouped CV for selection stability",
                                 "performance by image-to-index interval",
                                 "performance by calendar era"],
}
FROZEN_MANIFEST["frozen_addendum_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
MANIFEST_PATH.write_text(json.dumps(FROZEN_MANIFEST, indent=2, default=str))
print(f"\nmanifest updated with the robustness arm -> {MANIFEST_PATH}")
print("The primary model, ensemble rule, recalibration and threshold are UNCHANGED.")

---
# 16. SEALED TEST — read once

**Stop. Read this before running the next cell.**

Everything below opens the locked test set. It can happen **once**. The cell writes a marker file
to Drive the moment the test pixels are read, and refuses to run if that marker already exists.
There is deliberately no override flag: if a second read is genuinely necessary, an operator must
delete `test_seal/TEST_READ_ONCE.json` by hand and write a dated entry in
`outputs/protocol_deviations.md`. That friction is the feature.

### What must be true first (`model_image.test.unlock_requires`)

| config condition | machine check |
| --- | --- |
| model frozen | `FROZEN_MANIFEST.json` exists and its `training_contract_hash` equals the live `CONTRACT_HASH`; every seed checkpoint is `complete` |
| ensemble rule frozen | manifest `ensemble.rule == "average_hazard"` over exactly `SEEDS` |
| thresholds frozen | manifest carries a numeric `operating_point.threshold` |
| analysis script frozen | `sha256` of `inspect.getsource` over the analysis functions equals the manifest's `analysis_script_sha256` |
| crop QA signed off | `GATE_PASSED` from section 2 |
| laterality QA audit passed | checked in section 2, re-asserted here |

plus `UNLOCK_TEST = True` in section 0, which nobody sets by accident.

### What is computed, and nothing else

* **Primary (one comparison, unadjusted):** the paired difference in IPCW cumulative/dynamic AUROC,
  `m4_fusion` minus M0, at day 1825, with a **patient-level** percentile bootstrap over **2,000**
  replicates.
* **Secondary, FDR-controlled within four declared families:** the other two horizons; the
  modality ladder (image vs clinical vs fusion); frontal-only vs multi-view; and protocol
  **Secondary objective 2** — raw image versus the inferred-KLG-plus-clinical comparator (M1),
  which by the protocol's own wording is evaluated *in the subset with eligible bilateral frontal
  images* and therefore has its own patient set and its own bootstrap replicates. The `m1_klg`
  arm is held out of the primary common set for exactly that reason: folding a subset arm into
  the intersection would shrink the paired set for every model and quietly change the primary
  comparison.
* The rest of protocol Table 7: AUROC at 1 / 2 / 5 years, Harrell C **and** Uno C, IPCW Brier,
  time-dependent AUPRC with its weighted no-skill prevalence, calibration slope and
  calibration-in-the-large at each horizon, and the frozen operating point applied — not
  re-selected.
* Protocol section 21 subgroups and section 22 explainability, in sections 17 and 18, from the
  artefacts this cell produces — **not** from a second read.

### Why the estimand is a paired difference and not an absolute AUROC

The locked test set has 741 patients and 106 events. That clears protocol section 16's numeric
floor (500 total primary events and 100 test events; the cohort has 533 and 106). It does **not**
buy a precise absolute AUROC: the analyst's own simulation puts the 5-year AUROC half-width at
about **0.065** against an adopted target of **±0.05**. An absolute claim of the form "the fusion
model achieves an AUROC of 0.72" would therefore be reported at a precision the data does not
support.

A **paired** difference is a different quantity. Both models are scored on the same patients within
every bootstrap replicate, so the patient-level variance that dominates an absolute AUROC largely
cancels, and the difference is estimated far more precisely than either level. That is also exactly
what the scientific question asks — *does imaging add value beyond routine clinical variables* — so
the estimand and the question match.

The consequence is that the paired difference is emitted **from the start**: `boot_indices` is drawn
once and every model, every horizon and every comparison is evaluated on those same rows. It is not
possible to add pairing afterwards, which is why it is not left to the analysis stage.

The comparison set is the patients **every full-cohort arm can score** — a patient with no
frontal cannot be scored by a frontal-only model, and comparing models on different patients is
not a paired comparison. The size of that common set and the number of patients it excludes are
printed before any metric. The KLG-eligible arm (`m1_klg`) is excluded from that intersection and
compared on its own eligible set, which is also printed.

In [ ]:
# ---- 16. SEALED TEST CELL - THIS RUNS ONCE ------------------------------------------------------
print("=" * 96)
print("SEALED TEST EVALUATION".center(96))
print("=" * 96)

# ---- (i) unlock conditions ----------------------------------------------------------------------
UNLOCK = list(MI["test"]["unlock_requires"])
assert bool(MI["test"]["read_once"]), "config model_image.test.read_once must be true"
_checks = {}
_live = json.loads(MANIFEST_PATH.read_text())
_checks["model frozen"] = (
    MANIFEST_PATH.exists() and _live["training_contract_hash"] == CONTRACT_HASH and
    all(ckpt_path(t, s).exists() and SEED_RESULTS[t][s]["complete"]
        for t in ARMS for s in SEEDS))
_checks["ensemble rule frozen"] = (_live["ensemble"]["rule"] == "average_hazard" and
                                   list(_live["ensemble"]["seeds"]) == list(SEEDS))
_checks["thresholds frozen"] = isinstance(_live["operating_point"]["threshold"], (int, float))
_checks["analysis script frozen"] = (
    hashlib.sha256("\n".join(inspect.getsource(f) for f in ANALYSIS_FUNCTIONS).encode()).hexdigest()
    == _live["analysis_script_sha256"])
_checks["crop QA signed off"] = bool(GATE_PASSED)
_checks["laterality QA audit passed"] = bool(GATE_PASSED)
for cond in UNLOCK:
    ok = _checks.get(cond)
    print(f"  {'PASS' if ok else 'FAIL'}  unlock_requires: {cond}")
    assert cond in _checks, f"config lists an unlock condition with no machine check: {cond!r}"
    assert ok, f"unlock condition not satisfied: {cond!r}"
assert UNLOCK_TEST, ("UNLOCK_TEST is False. Set it in section 0 only when you intend to spend the "
                     "single permitted read of the locked test set.")

# ---- (ii) one-time marker -----------------------------------------------------------------------
MARKER = SEAL_DIR / "TEST_READ_ONCE.json"
if MARKER.exists():
    raise AssertionError(
        f"THE TEST SET HAS ALREADY BEEN READ.\n{MARKER}\n"
        f"{MARKER.read_text()}\n"
        "Protocol section 12 permits one scripted evaluation of the locked test set. There is no "
        "override flag here on purpose. A second read requires deleting this marker by hand AND a "
        "dated entry in outputs/protocol_deviations.md explaining why the first read did not count.")

assert SHARD_REPORT["test"]["n_shards"] > 0, "no test-*.tar on Drive; run preprocess_colab.ipynb "\
    "with INCLUDE_TEST=True first (that is a deliberate act, not a default)"

# Structural check of the test tars BEFORE the marker is written. This reads MEMBER NAMES only -
# no pixel, no label, no outcome - so it is not a read of the sealed content; and doing it first
# means a malformed or incomplete shard set fails the run instead of burning the single permitted
# read on it.
_n_test, _bad = 0, []
for _u in SHARD_REPORT["test"]["urls"]:
    _info = scan_tar(_u)
    _uniq = sorted(set(_info["keys"]))
    _n_test += len(_uniq)
    if [e for _, e in _info["order"]] != [IMAGE_EXT, "json"] * len(_uniq):
        _bad.append(f"{Path(_u).name}: members are not contiguous {IMAGE_EXT} then json")
    _sp = set(LABELS.loc[LABELS["key"].isin(_uniq), "split"])
    if _sp - {"test"}:
        _bad.append(f"{Path(_u).name} carries {sorted(_sp)}, not test")
assert not _bad, _bad
assert _n_test == int((LABELS["split"] == "test").sum()) == FROZEN_IMAGES["test"], (
    f"test shards hold {_n_test} samples; the sidecar says "
    f"{int((LABELS['split'] == 'test').sum())} and the frozen contract says "
    f"{FROZEN_IMAGES['test']}. Resolve this BEFORE spending the read.")
SHARD_REPORT["test"]["n_samples"] = _n_test
print(f"  test shards structurally verified from member names only: {_n_test} samples, "
      f"no split mixing. The marker is written next.")

MARKER.write_text(json.dumps({
    "read_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "frozen_manifest_sha256": hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest(),
    "analysis_script_sha256": _live["analysis_script_sha256"],
    "training_contract_hash": CONTRACT_HASH,
    "primary_comparison": _live["evaluation"]["primary_comparison"],
    "primary_horizon_days": PRIMARY_HORIZON_DAYS,
    "bootstrap_replicates": BOOTSTRAP_N,
    "note": "written BEFORE any metric was computed. The test pixels were read at this moment; "
            "a crash after this point still counts as the one permitted read.",
}, indent=2))
print(f"\n  one-time marker written: {MARKER}")

# ---- (iii) open the seal ------------------------------------------------------------------------
SEAL_OPEN = True
CACHE["test"] = materialize_split("test")
TEST_FRAME, TEST_X = build_clinical_design(["test"])
M0_LP_TEST, M0_RISK_TEST = replay_from_json(M0, TEST_X)
TEST_FRAME_M1, TEST_X_M1 = build_clinical_design(["test"], design="m1")
M1_LP_TEST, M1_RISK_TEST = replay_from_json(M1, TEST_X_M1)
assert len(TEST_FRAME_M1) == M1_ELIGIBILITY["n_eligible_by_split"]["test"]
print(f"  M1 (KLG-eligible) test subset: {len(TEST_FRAME_M1)} of {len(TEST_FRAME)} patients, "
      f"{int(TEST_FRAME_M1['event_indicator'].sum())} events")
assert len(TEST_FRAME) == FROZEN_COHORT["test"]["patients"]
assert int(TEST_FRAME["event_indicator"].sum()) == FROZEN_COHORT["test"]["events"]
assert SHARD_REPORT["test"]["n_samples"] in (None, FROZEN_IMAGES["test"])
print(f"  test: {len(TEST_FRAME)} patients, {int(TEST_FRAME['event_indicator'].sum())} events, "
      f"{len(CACHE['test'][1])} images (contract {FROZEN_IMAGES['test']})")
assert len(CACHE["test"][1]) == FROZEN_IMAGES["test"]

In [ ]:
# ---- 16b. Test predictions for every locked arm, on a COMMON patient set ------------------------
TEST_DS_BY_ARM, TEST_HAZ, TEST_RISK, TEST_RISK_RECAL = {}, {}, {}, {}
for tag, spec in ARMS.items():
    _fr = TEST_FRAME_M1 if spec.get("design", "m0") == "m1" else TEST_FRAME
    _X = TEST_X_M1 if spec.get("design", "m0") == "m1" else TEST_X
    ds = PatientViewDataset("test", _fr, _X, train=False, views_allowed=spec["views_allowed"],
                            design=spec.get("design", "m0"))
    h, _ = ensemble_hazards(tag, ds, mode=spec["mode"], arch=spec.get("arch", ARCH),
                            design=spec.get("design", "m0"))
    TEST_DS_BY_ARM[tag], TEST_HAZ[tag] = ds, h
    TEST_RISK[tag] = {hh: risk_at_horizon(h, hh) for hh in HORIZONS}
    TEST_RISK_RECAL[tag] = {hh: apply_recalibration(TEST_RISK[tag][hh], RECAL[tag][str(hh)])
                            for hh in HORIZONS}
    print(f"  {tag:22s} {len(ds):4d} patients scored, {ds.n_no_image} unscoreable "
          f"(no view in this arm), {ds.n_truncated} truncated to {MAX_ELEMS} views"
          + ("   [SUBSET ARM: KLG-eligible patients only]" if tag in SUBSET_ARMS else ""))

# The primary common set intersects the FULL-COHORT arms only. Including a subset arm would
# shrink the paired set for every model, which would silently change the primary comparison.
PRIMARY_ARMS = [t for t in TEST_DS_BY_ARM if t not in SUBSET_ARMS]
COMMON = sorted(set.intersection(*[set(TEST_DS_BY_ARM[t].pids) for t in PRIMARY_ARMS]))
_all_pids = set(TEST_FRAME["empi_anon"].astype(str))
print(f"\n  PAIRED COMPARISON SET (protocol section 18: the same test patients for every model)")
print(f"    common to all {len(PRIMARY_ARMS)} full-cohort arms: {len(COMMON)} of "
      f"{len(_all_pids)} test patients")
print(f"    excluded: {len(_all_pids - set(COMMON))} - patients no frontal-only arm can score, "
      f"plus any with no usable crop at all")
print(f"    subset arms held out of this set: {sorted(SUBSET_ARMS)} (evaluated below on their "
      f"own eligible common set, protocol Secondary objective 2)")

_pos = {tag: {p: i for i, p in enumerate(ds.pids)} for tag, ds in TEST_DS_BY_ARM.items()}
_m0pos = {p: i for i, p in enumerate(TEST_FRAME["empi_anon"].astype(str))}
CIDX = {tag: np.array([_pos[tag][p] for p in COMMON]) for tag in PRIMARY_ARMS}
M0IDX = np.array([_m0pos[p] for p in COMMON])

T_COM = TEST_DS_BY_ARM[PRIMARY_ARM].time[CIDX[PRIMARY_ARM]]
E_COM = TEST_DS_BY_ARM[PRIMARY_ARM].event[CIDX[PRIMARY_ARM]].astype(int)
for tag in PRIMARY_ARMS:
    ds = TEST_DS_BY_ARM[tag]
    assert np.array_equal(ds.time[CIDX[tag]], T_COM) and np.array_equal(ds.event[CIDX[tag]].astype(int), E_COM), \
        f"{tag}: label misalignment on the common set"
print(f"    common set: {len(COMMON)} patients, {int(E_COM.sum())} events "
      f"({100*E_COM.mean():.1f}% - the NATURAL test prevalence, never resampled)")

MODELS = {tag: {h: TEST_RISK_RECAL[tag][h][CIDX[tag]] for h in HORIZONS} for tag in PRIMARY_ARMS}
# M0 enters UNRECALIBRATED. Its absolute risks come from its own frozen baseline survival, and
# applying an image-model recalibration fitted on validation to it would report a comparator that
# does not exist. The recalibration is monotone, so the AUROC comparison is unaffected either way;
# only the Brier and calibration columns would change, and there the honest M0 is the published one.
MODELS["m0_cox"] = {h: M0_RISK_TEST[h][M0IDX] for h in HORIZONS}
RANK_SCORE = {tag: risk_score(TEST_HAZ[tag])[CIDX[tag]] for tag in PRIMARY_ARMS}
RANK_SCORE["m0_cox"] = M0_LP_TEST[M0IDX]

# IPCW labels/weights depend only on (T, E) and the FROZEN train censoring curve, so they are
# computed once and indexed by the bootstrap rows. That is what makes the weights identical for
# every model in every replicate.
YW = {h: ipcw_labels_weights(T_COM, E_COM, h) for h in HORIZONS}
for h in HORIZONS:
    y, w = YW[h]
    print(f"    @{int(h):4d} d: cases {int((y==1).sum())}, controls {int((y==0).sum())}, "
          f"censored before the horizon {int((y==-1).sum())}")

# ---- protocol Secondary objective 2: the KLG-ELIGIBLE comparison set ---------------------------
# "Compare raw-image prediction with an inferred KLG plus clinical comparator IN THE SUBSET WITH
# ELIGIBLE BILATERAL FRONTAL IMAGES." That subset is smaller than the primary common set, so it
# gets its own intersection, its own IPCW labels and (in the next cell) its own replicates. The
# frozen Cox M1 is carried alongside the trained m1_klg arm, exactly as m0_cox is carried
# alongside m0d_clinical.
_KLG_ARMS = sorted(SUBSET_ARMS) + ["m2_frontal", "m3_image", "m4_fusion"]
COMMON_KLG = sorted(set.intersection(*[set(TEST_DS_BY_ARM[t].pids) for t in _KLG_ARMS]))
KIDX = {t: np.array([_pos[t][p] for p in COMMON_KLG]) for t in _KLG_ARMS}
_m1pos = {p: i for i, p in enumerate(TEST_FRAME_M1["empi_anon"].astype(str))}
M1IDX = np.array([_m1pos[p] for p in COMMON_KLG])
T_KLG = TEST_DS_BY_ARM["m1_klg"].time[KIDX["m1_klg"]]
E_KLG = TEST_DS_BY_ARM["m1_klg"].event[KIDX["m1_klg"]].astype(int)
for t in _KLG_ARMS:
    ds = TEST_DS_BY_ARM[t]
    assert np.array_equal(ds.time[KIDX[t]], T_KLG) and \
        np.array_equal(ds.event[KIDX[t]].astype(int), E_KLG), f"{t}: label misalignment (KLG set)"
MODELS_KLG = {t: {h: TEST_RISK_RECAL[t][h][KIDX[t]] for h in HORIZONS} for t in _KLG_ARMS}
MODELS_KLG["m1_cox"] = {h: M1_RISK_TEST[h][M1IDX] for h in HORIZONS}
YW_KLG = {h: ipcw_labels_weights(T_KLG, E_KLG, h) for h in HORIZONS}
N_KLG = len(COMMON_KLG)
print(f"\n  SECONDARY OBJECTIVE 2 SET (KLG-eligible): {N_KLG} patients, {int(E_KLG.sum())} events "
      f"({100*E_KLG.mean():.1f}%), from the {len(TEST_FRAME_M1)} KLG-eligible test patients")
print(f"    arms compared here: {_KLG_ARMS} plus the frozen Cox m1_cox")

In [ ]:
# ---- 16c. ONE set of bootstrap replicates, shared by every model and every comparison ------------
BOOT_SEED = int(CFG["reproducibility"]["random_seed"])
_rng = np.random.default_rng(BOOT_SEED)
N_COM = len(COMMON)
BOOT = _rng.integers(0, N_COM, size=(BOOTSTRAP_N, N_COM))     # patient-level, drawn ONCE
print(f"bootstrap: {BOOTSTRAP_N} replicates x {N_COM} patients, seed {BOOT_SEED}. "
      f"Every model, horizon and comparison is evaluated on THESE rows.")
print("Protocol section 18: all confidence intervals are patient-level. A patient contributes all "
      "of its views together, so repeated views cannot inflate the effective sample size.")

MODEL_KEYS = list(MODELS.keys())
# Harrell's C is O(n^2) per replicate. It is bootstrapped only for the two models the primary
# comparison involves; every other arm gets a point estimate, and the omission is stated in the
# table rather than papered over with a narrower-looking interval.
C_BOOT_MODELS = [PRIMARY_ARM, "m0_cox"]
BOOT_AUC = {(m, h): np.full(BOOTSTRAP_N, np.nan) for m in MODEL_KEYS for h in HORIZONS}
BOOT_BRIER = {(m, h): np.full(BOOTSTRAP_N, np.nan) for m in MODEL_KEYS for h in HORIZONS}
BOOT_AUPRC = {(m, h): np.full(BOOTSTRAP_N, np.nan) for m in MODEL_KEYS for h in HORIZONS}
BOOT_C = {m: np.full(BOOTSTRAP_N, np.nan) for m in C_BOOT_MODELS}

_t0 = time.time()
for b in range(BOOTSTRAP_N):
    idx = BOOT[b]
    tb, eb = T_COM[idx], E_COM[idx]
    for h in HORIZONS:
        y, w = YW[h][0][idx], YW[h][1][idx]
        if not ((y == 1).any() and (y == 0).any()):
            continue
        for m in MODEL_KEYS:
            p = MODELS[m][h][idx]
            BOOT_AUC[(m, h)][b] = ipcw_auc(y, w, p)
            BOOT_BRIER[(m, h)][b] = ipcw_brier(y, w, p)
            BOOT_AUPRC[(m, h)][b] = ipcw_auprc(y, w, p)[0]
    for m in C_BOOT_MODELS:
        BOOT_C[m][b] = harrell_c(tb, eb, RANK_SCORE[m][idx])
    if (b + 1) % 250 == 0:
        print(f"  {b+1}/{BOOTSTRAP_N} replicates  ({time.time()-_t0:.0f}s)")
print(f"bootstrap done in {time.time()-_t0:.0f}s; Harrell C bootstrapped for {C_BOOT_MODELS}")

# ---- a SECOND set of replicates, for the KLG-eligible set only ---------------------------------
# Protocol Secondary objective 2 is defined on a different patient set, so it cannot share the
# replicates above (those index rows of COMMON). It gets its own draw from the SAME seed stream,
# and every comparison inside that family is paired on these rows. Nothing here touches the
# primary comparison.
_rng_klg = np.random.default_rng(BOOT_SEED + 1)
BOOT_KLG = _rng_klg.integers(0, N_KLG, size=(BOOTSTRAP_N, N_KLG))
KLG_MODEL_KEYS = list(MODELS_KLG.keys())
BOOT_AUC_KLG = {(m, h): np.full(BOOTSTRAP_N, np.nan) for m in KLG_MODEL_KEYS for h in HORIZONS}
_t1 = time.time()
for b in range(BOOTSTRAP_N):
    idx = BOOT_KLG[b]
    for h in HORIZONS:
        y, w = YW_KLG[h][0][idx], YW_KLG[h][1][idx]
        if not ((y == 1).any() and (y == 0).any()):
            continue
        for m in KLG_MODEL_KEYS:
            BOOT_AUC_KLG[(m, h)][b] = ipcw_auc(y, w, MODELS_KLG[m][h][idx])
print(f"KLG-eligible bootstrap: {BOOTSTRAP_N} replicates x {N_KLG} patients, seed "
      f"{BOOT_SEED + 1}, done in {time.time()-_t1:.0f}s")


def pct_ci(v, alpha=0.05):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if v.size < 2:
        return float("nan"), float("nan"), 0
    return (float(np.percentile(v, 100 * alpha / 2)),
            float(np.percentile(v, 100 * (1 - alpha / 2))), int(v.size))


def boot_p_two_sided(d):
    """Two-sided bootstrap p for H0: difference = 0, floored at 1/B (a bootstrap cannot resolve 0)."""
    d = np.asarray(d, float); d = d[np.isfinite(d)]
    if d.size < 2:
        return float("nan")
    p = 2.0 * min((d <= 0).mean(), (d >= 0).mean())
    return float(min(1.0, max(p, 1.0 / d.size)))


def bh_fdr(pvals):
    """Benjamini-Hochberg adjusted p-values, within one declared family."""
    p = np.asarray(pvals, float)
    ok = np.isfinite(p)
    out = np.full(p.shape, np.nan)
    q = p[ok]
    if q.size == 0:
        return out
    order = np.argsort(q)
    ranked = q[order] * q.size / (np.arange(q.size) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adj = np.empty_like(ranked)
    adj[order] = np.clip(ranked, 0, 1)
    out[ok] = adj
    return out

In [ ]:
# ---- 16d. PRIMARY INFERENCE - exactly one comparison, unadjusted --------------------------------
A, B = PRIMARY_ARM, "m0_cox"
H = PRIMARY_HORIZON_DAYS
_yh, _wh = YW[H]
est_a = ipcw_auc(_yh, _wh, MODELS[A][H])
est_b = ipcw_auc(_yh, _wh, MODELS[B][H])
diff_boot = BOOT_AUC[(A, H)] - BOOT_AUC[(B, H)]
lo, hi, nvalid = pct_ci(diff_boot)
p_prim = boot_p_two_sided(diff_boot)
alo, ahi, _ = pct_ci(BOOT_AUC[(A, H)])
blo, bhi, _ = pct_ci(BOOT_AUC[(B, H)])

print("=" * 96)
print(f"PRIMARY: IPCW cumulative/dynamic AUROC, {A} minus M0 penalized Cox, at day {int(H)}")
print("=" * 96)
print(f"  {A:14s}  AUROC {est_a:.4f}  95% CI [{alo:.4f}, {ahi:.4f}]  "
      f"(half-width {0.5*(ahi-alo):.4f})")
print(f"  {'M0 Cox':14s}  AUROC {est_b:.4f}  95% CI [{blo:.4f}, {bhi:.4f}]  "
      f"(half-width {0.5*(bhi-blo):.4f})")
print(f"  PAIRED DIFFERENCE  {est_a-est_b:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  "
      f"(half-width {0.5*(hi-lo):.4f})  two-sided bootstrap p = {p_prim:.4f}")
print(f"  valid replicates: {nvalid}/{BOOTSTRAP_N}")
print(f"\n  The paired difference is the estimand. The two absolute AUROCs are shown for context "
      f"only:\n  the analyst's simulation puts the 5-year absolute half-width near 0.065 against an "
      f"adopted\n  target of 0.05, so an absolute claim is under-precise. Protocol section 16's "
      f"numeric floor\n  (500 total / 100 test events) IS met, at 533 and 106.")
print(f"\n  Validation reference for orientation: M0 val AUROC @730 = {M0_VAL_AUC_2Y:.4f}, "
      f"@1825 = {M0['val_metrics']['auc@1825.0']:.4f}.")

PRIMARY_RESULT = {"comparison": f"{A} - m0_cox", "metric": "ipcw_cd_auroc",
                  "horizon_days": H, "n_patients": N_COM, "n_events": int(E_COM.sum()),
                  "auc_a": est_a, "auc_a_ci": [alo, ahi],
                  "auc_b": est_b, "auc_b_ci": [blo, bhi],
                  "difference": est_a - est_b, "difference_ci": [lo, hi],
                  "p_two_sided": p_prim, "n_valid_replicates": nvalid,
                  "adjusted": False,
                  "note": "the single primary inference (protocol section 18)"}

In [ ]:
# ---- 16e. SECONDARY families, FDR-controlled within each ---------------------------------------
FAMILIES = {
    "horizons": [(PRIMARY_ARM, "m0_cox", h) for h in SECONDARY_HORIZONS],
    "modality": [("m3_image", "m0d_clinical", H), ("m4_fusion", "m3_image", H),
                 ("m4_fusion", "m0d_clinical", H)],
    "views":    [("m4_fusion", "m4_frontal", H), ("m3_image", "m2_frontal", H)],
    "robustness": [("m2_frontal", "r1_densenet_frontal", H)],
}
_rows = []
for fam, comps in FAMILIES.items():
    ps = []
    for a, b, h in comps:
        d = BOOT_AUC[(a, h)] - BOOT_AUC[(b, h)]
        l, u, nv = pct_ci(d)
        ps.append(boot_p_two_sided(d))
        _rows.append({"family": fam, "model_a": a, "model_b": b, "horizon_days": h,
                      "auc_a": ipcw_auc(*YW[h], MODELS[a][h]),
                      "auc_b": ipcw_auc(*YW[h], MODELS[b][h]),
                      "difference": ipcw_auc(*YW[h], MODELS[a][h]) - ipcw_auc(*YW[h], MODELS[b][h]),
                      "ci_lo": l, "ci_hi": u, "p_raw": ps[-1], "n_valid": nv})
    adj = bh_fdr(ps)
    for i in range(len(comps)):
        _rows[-len(comps) + i]["p_bh_within_family"] = float(adj[i])
SECONDARY = pd.DataFrame(_rows)
SECONDARY.to_csv(RESULTS_DIR / "test_secondary_comparisons.csv", index=False)
print("SECONDARY COMPARISONS (Benjamini-Hochberg WITHIN each declared family; never pooled "
      "across families, and never mixed with the primary)")
print(SECONDARY.round(4).to_string(index=False))
print("\nEvery row above shares the SAME 2,000 bootstrap replicates and the SAME test patients as "
      "the primary comparison, so any difference between two rows is itself paired.")

# ---- protocol Secondary objective 2, on the KLG-eligible set -----------------------------------
# "Compare raw-image prediction with an inferred KLG plus clinical comparator in the subset with
# eligible bilateral frontal images." Its own patient set, its own replicates, its own BH family.
# It is NEVER pooled with the families above and it never touches the primary claim.
FAMILY_SO2 = [(a, b, H) for a in ("m2_frontal", "m3_image", "m4_fusion")
              for b in ("m1_klg", "m1_cox")]
_rows2, _ps2 = [], []
for a, b, h in FAMILY_SO2:
    d = BOOT_AUC_KLG[(a, h)] - BOOT_AUC_KLG[(b, h)]
    l, u, nv = pct_ci(d)
    _ps2.append(boot_p_two_sided(d))
    _rows2.append({"family": "secondary_objective_2", "model_a": a, "model_b": b,
                   "horizon_days": h, "n_patients": N_KLG, "n_events": int(E_KLG.sum()),
                   "auc_a": ipcw_auc(*YW_KLG[h], MODELS_KLG[a][h]),
                   "auc_b": ipcw_auc(*YW_KLG[h], MODELS_KLG[b][h]),
                   "difference": (ipcw_auc(*YW_KLG[h], MODELS_KLG[a][h])
                                  - ipcw_auc(*YW_KLG[h], MODELS_KLG[b][h])),
                   "ci_lo": l, "ci_hi": u, "p_raw": _ps2[-1], "n_valid": nv})
_adj2 = bh_fdr(_ps2)
for i in range(len(FAMILY_SO2)):
    _rows2[i]["p_bh_within_family"] = float(_adj2[i])
SECONDARY_OBJ2 = pd.DataFrame(_rows2)
SECONDARY_OBJ2.to_csv(RESULTS_DIR / "test_secondary_objective_2.csv", index=False)
print(f"\nSECONDARY OBJECTIVE 2 - raw image vs inferred-KLG-plus-clinical, on the "
      f"{N_KLG} KLG-eligible test patients ({int(E_KLG.sum())} events), day {int(H)}")
print("  m1_klg is the trained discrete-time arm; m1_cox is the frozen penalized-Cox M1 from "
      "m1_klg_model.json - the same pairing m0d_clinical / m0_cox has in the primary ladder.")
print(SECONDARY_OBJ2.round(4).to_string(index=False))
print("  Benjamini-Hochberg is applied WITHIN this family only. These rows use their own "
      "bootstrap replicates over their own patient set and are not comparable, replicate for "
      "replicate, with the families above.")

In [ ]:
# ---- 16f. Per-model discrimination, accuracy, calibration and the frozen operating point --------
BOOTSTRAP_CALIBRATION = True     # 2 models x 1 horizon x 2,000 weighted GLMs; a few minutes
_rows = []
for m in MODEL_KEYS:
    c_lo, c_hi, _ = pct_ci(BOOT_C[m]) if m in BOOT_C else (float("nan"), float("nan"), 0)
    r = {"model": m, "harrell_c": harrell_c(T_COM, E_COM, RANK_SCORE[m]),
         "c_lo": c_lo, "c_hi": c_hi,
         "c_ci_bootstrapped": m in BOOT_C,
         "uno_c": uno_c(T_COM, E_COM, RANK_SCORE[m])}
    for h in HORIZONS:
        y, w = YW[h]
        p = MODELS[m][h]
        a_lo, a_hi, _ = pct_ci(BOOT_AUC[(m, h)])
        b_lo, b_hi, _ = pct_ci(BOOT_BRIER[(m, h)])
        pr_lo, pr_hi, _ = pct_ci(BOOT_AUPRC[(m, h)])
        ap, wprev = ipcw_auprc(y, w, p)
        slope, citl = calibration_slope_intercept(y, w, p)
        r.update({f"auc@{int(h)}": ipcw_auc(y, w, p), f"auc@{int(h)}_lo": a_lo,
                  f"auc@{int(h)}_hi": a_hi, f"brier@{int(h)}": ipcw_brier(y, w, p),
                  f"brier@{int(h)}_lo": b_lo, f"brier@{int(h)}_hi": b_hi,
                  f"auprc@{int(h)}": ap, f"auprc@{int(h)}_lo": pr_lo, f"auprc@{int(h)}_hi": pr_hi,
                  f"auprc_noskill@{int(h)}": wprev,
                  f"slope@{int(h)}": slope, f"citl@{int(h)}": citl,
                  f"predrisk@{int(h)}": float(p.mean())})
    _rows.append(r)
TEST_TABLE = pd.DataFrame(_rows)
TEST_TABLE.to_csv(RESULTS_DIR / "test_metrics.csv", index=False)
_show = ["model", "harrell_c", "c_lo", "c_hi", "c_ci_bootstrapped", "uno_c"] + \
        [c for h in HORIZONS for c in (f"auc@{int(h)}", f"auc@{int(h)}_lo", f"auc@{int(h)}_hi")]
print("--- discrimination on the common test set (risks are the FROZEN recalibrated ones) ---")
print(TEST_TABLE[_show].round(4).to_string(index=False))
print("  harrell_c matches the estimator behind M0's own `cindex` field; uno_c is the "
      "censoring-robust variant protocol Table 7 also names.")
print("\n--- time-dependent AUPRC (protocol Table 7). Read every value against its no-skill "
      "column: at ~14% weighted prevalence, 0.30 is good and 0.14 is nothing. ---")
print(TEST_TABLE[["model"] + [c for h in HORIZONS for c in
                              (f"auprc@{int(h)}", f"auprc@{int(h)}_lo", f"auprc@{int(h)}_hi",
                               f"auprc_noskill@{int(h)}")]].round(4).to_string(index=False))
print("\n--- calibration after the frozen validation recalibration ---")
print(TEST_TABLE[["model"] + [c for h in HORIZONS for c in
                              (f"slope@{int(h)}", f"citl@{int(h)}", f"predrisk@{int(h)}",
                               f"brier@{int(h)}")]].round(4).to_string(index=False))
print("A slope far from 1 after recalibration means the validation fit (54 events) did not "
      "transport; that is a finding, not something to re-fit.")

if BOOTSTRAP_CALIBRATION:
    print(f"\nbootstrapping the calibration slope for {PRIMARY_ARM} and m0_cox @ {int(H)} d ...")
    _t0 = time.time()
    _cal = {m: np.full(BOOTSTRAP_N, np.nan) for m in (PRIMARY_ARM, "m0_cox")}
    for b in range(BOOTSTRAP_N):
        idx = BOOT[b]
        y, w = YW[H][0][idx], YW[H][1][idx]
        for m in _cal:
            _cal[m][b] = calibration_slope_intercept(y, w, MODELS[m][H][idx])[0]
    for m, v in _cal.items():
        l, u, nv = pct_ci(v)
        print(f"  {m:14s} calibration slope {calibration_slope_intercept(*YW[H], MODELS[m][H])[0]:.3f} "
              f"95% CI [{l:.3f}, {u:.3f}]  ({nv} valid replicates, {time.time()-_t0:.0f}s)")

# ---- the frozen operating point, applied not re-selected ---------------------------------------
_p = MODELS[PRIMARY_ARM][H]
_y, _w = YW[H]
_alert = _p >= THRESHOLD
_tp = int((_alert & (_y == 1)).sum()); _fn = int((~_alert & (_y == 1)).sum())
_fp = int((_alert & (_y == 0)).sum()); _tn = int((~_alert & (_y == 0)).sum())
OPERATING = {"threshold": THRESHOLD, "horizon_days": H,
             "sensitivity": _tp / max(_tp + _fn, 1), "specificity": _tn / max(_tn + _fp, 1),
             "ppv": _tp / max(_tp + _fp, 1), "npv": _tn / max(_tn + _fn, 1),
             "alert_rate": float(_alert.mean()), "tp": _tp, "fp": _fp, "tn": _tn, "fn": _fn,
             "val_sensitivity_at_selection": VAL_SENS_AT_THRESHOLD}
print(f"\n--- frozen operating point at {THRESHOLD:.6f} (selected on validation, NOT re-selected) ---")
for k in ("sensitivity", "specificity", "ppv", "npv", "alert_rate"):
    print(f"  {k:12s} {OPERATING[k]:.3f}")
print(f"  confusion among usable patients: TP {_tp}  FP {_fp}  TN {_tn}  FN {_fn}")

In [ ]:
# ---- 16g. Persist everything the manuscript needs ------------------------------------------------
TEST_RESULTS = {
    "read_utc": json.loads(MARKER.read_text())["read_utc"],
    "frozen_manifest_sha256": hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest(),
    "analysis_script_sha256": ANALYSIS_HASH,
    "bootstrap": {"n_replicates": BOOTSTRAP_N, "seed": BOOT_SEED, "unit": "patient",
                  "paired": True, "shared_replicates_across_all_comparisons": True},
    "comparison_set": {"n_patients": N_COM, "n_events": int(E_COM.sum()),
                       "n_test_patients_total": len(TEST_FRAME),
                       "n_excluded_from_pairing": len(TEST_FRAME) - N_COM,
                       "reason": "a frontal-only arm cannot score a patient with no frontal; a "
                                 "paired comparison requires the same patients for every model"},
    "primary": PRIMARY_RESULT,
    "secondary": json.loads(SECONDARY.to_json(orient="records")),
    "secondary_objective_2": {
        "definition": "protocol Secondary objective 2 - raw-image prediction versus an inferred "
                      "KLG plus clinical comparator, in the subset with eligible bilateral "
                      "frontal images",
        "comparison_set": {"n_patients": N_KLG, "n_events": int(E_KLG.sum()),
                           "n_klg_eligible_test_patients": len(TEST_FRAME_M1),
                           "bootstrap_seed": BOOT_SEED + 1,
                           "note": "its own patient set and its own replicates; never pooled "
                                   "with the primary comparison"},
        "rows": json.loads(SECONDARY_OBJ2.to_json(orient="records"))},
    "per_model": json.loads(TEST_TABLE.to_json(orient="records")),
    "operating_point": OPERATING,
    "ipcw_weights": "frozen train reverse-KM from m0_clinical_model.json; identical for every model",
}
(RESULTS_DIR / "test_results.json").write_text(json.dumps(TEST_RESULTS, indent=2, default=str))

# artefacts the explainability and subgroup sections reuse - so neither re-reads the test shards
TEST_ARTIFACTS = {"common_pids": COMMON, "cidx": CIDX, "m0idx": M0IDX, "T": T_COM, "E": E_COM,
                  "models": MODELS, "rank_score": RANK_SCORE, "yw": YW, "boot": BOOT,
                  "hazards": TEST_HAZ, "datasets": TEST_DS_BY_ARM}
print(f"written: {RESULTS_DIR/'test_results.json'}, test_metrics.csv, "
      f"test_secondary_comparisons.csv, test_secondary_objective_2.csv")
print("\nTHE LOCKED TEST SET IS NOW CLOSED. Sections 17 and 18 reuse TEST_ARTIFACTS and never "
      "re-open a shard.")

## 17. Protocol section 22 — explainability and shortcut testing

Four required analyses, all on the **already-materialized** test cache. No shard is re-opened and
no metric from this section can change anything: the manifest was frozen in section 14 and the test
read is recorded.

1. **Grad-CAM and integrated gradients** on a stratified sample of true positives, false positives,
   true negatives and false negatives, defined by the frozen operating point at the primary horizon.
2. **Occlusion** over medial, lateral, patellofemoral and image-border regions.
3. **Saliency after randomization** — encoder weights randomized, and separately a model trained on
   permuted training labels. Attributions that survive either are attributions of the image
   statistics, not of the model's learned signal, and the maps should degrade.
4. **Prediction change after masking text, markers and borders.**

### Where "medial" is in these crops

The crops are standardized so every knee reads as a **left** knee. `preprocess.bilateral_display_convention`
is `"radiological"`, so the patient's right knee sits on the image left and the patient's **left**
knee on the image right; within an isolated left-knee crop the body midline is therefore toward the
**image left edge**, and medial is the image-left third. Right knees were mirrored to match, so the
rule is uniform.

**This is a derivation from the preprocessing convention, not a measurement.** `MEDIAL_SIDE` is a
single constant below; confirm it once against the crop QA contact sheet and flip it if the sheet
disagrees. Reporting a medial-compartment result with the sides swapped would be a clinically
meaningful error, so it is worth the thirty seconds.

Maps are described as **model-attribution visualizations, not validated imaging biomarkers**
(protocol section 22, final line).

In [ ]:
# ---- 17a. Attribution utilities -----------------------------------------------------------------
MEDIAL_SIDE = "image_left"      # derived above; confirm once against the crop QA contact sheet
assert MEDIAL_SIDE in ("image_left", "image_right")
assert "TEST_ARTIFACTS" in dir(), "run the sealed test cell first - this section reuses its output"

_INNER = slice(BORDER_PX, OUT_SIZE - BORDER_PX)
_W = OUT_SIZE - 2 * BORDER_PX
_third = _W // 3
_left_cols = slice(BORDER_PX, BORDER_PX + _third)
_right_cols = slice(OUT_SIZE - BORDER_PX - _third, OUT_SIZE - BORDER_PX)
REGIONS = {
    "medial": (_INNER, _left_cols if MEDIAL_SIDE == "image_left" else _right_cols),
    "lateral": (_INNER, _right_cols if MEDIAL_SIDE == "image_left" else _left_cols),
    "patellofemoral": (slice(BORDER_PX, BORDER_PX + _W // 3),
                       slice(BORDER_PX + _third, OUT_SIZE - BORDER_PX - _third)),
    "border_ring": None,        # handled separately: a 24-px ring just inside the masked band
    "central_joint": (slice(BORDER_PX + _W // 3, BORDER_PX + 2 * _W // 3),
                      slice(BORDER_PX + _third, OUT_SIZE - BORDER_PX - _third)),
}


def occlude(img_u8: torch.Tensor, region: str) -> torch.Tensor:
    x = img_u8.clone()
    if region == "border_ring":
        r = 24
        x[..., BORDER_PX:BORDER_PX + r, :] = 0; x[..., -BORDER_PX - r:-BORDER_PX, :] = 0
        x[..., :, BORDER_PX:BORDER_PX + r] = 0; x[..., :, -BORDER_PX - r:-BORDER_PX] = 0
        return x
    if region == "wide_border":
        b = 2 * BORDER_PX
        x[..., :b, :] = 0; x[..., -b:, :] = 0; x[..., :, :b] = 0; x[..., :, -b:] = 0
        return x
    if region == "corners":
        c = 96
        for rs in (slice(0, c), slice(-c, None)):
            for cs in (slice(0, c), slice(-c, None)):
                x[..., rs, cs] = 0
        return x
    rs, cs = REGIONS[region]
    x[..., rs, cs] = 0
    return x


def cumhaz_score(logits: torch.Tensor) -> torch.Tensor:
    """Scalar per patient, monotone in risk: total cumulative hazard -sum log(1 - h_k)."""
    return -torch.log1p(-torch.sigmoid(logits.float()).clamp(max=1 - 1e-6)).sum(dim=1)


def batch_from(ds, i: int) -> dict:
    b = {k: (v.unsqueeze(0) if torch.is_tensor(v) else v) for k, v in ds[i].items()}
    return to_device(b)


def grad_cam(net, batch: dict, elem: int) -> np.ndarray:
    """Grad-CAM for ONE view element, from the patient's FULL forward pass.

    The whole set goes through the encoder, the view embeddings, the attention pool, the
    clinical concatenation and the head; the gradient of the patient's cumulative hazard is
    then read off the chosen element's final feature map. Attributing the score of a
    single-view counterfactual instead would answer a different question, and for a
    three-view patient it would not be the score the model actually produced.
    """
    net.zero_grad(set_to_none=True)
    B, E = batch["images"].shape[:2]
    flat = to_model_input(batch["images"].reshape(B * E, *batch["images"].shape[2:]))
    fmap = net.encoder.forward_features(flat)                       # (B*E, C, h, w)
    fmap.retain_grad()
    pooled = net.encoder.forward_head(fmap, pre_logits=True).reshape(B, E, net.feat_dim)
    f = net.proj(torch.cat([pooled, net.view_emb(batch["view_id"])], dim=-1))
    f = f * batch["mask"].unsqueeze(-1)
    emb, _ = net.pool(f, batch["mask"])
    parts = [net.img_norm(emb), batch["view_available"]]
    if net.use_clinical:
        parts.append(batch["clinical"])
    cumhaz_score(net.head(torch.cat(parts, dim=-1))).backward()
    A = fmap.reshape(B, E, *fmap.shape[1:])[0, elem]                # (C, h, w)
    G = fmap.grad.reshape(B, E, *fmap.shape[1:])[0, elem]
    cam = torch.relu((G.mean(dim=(1, 2), keepdim=True) * A).sum(dim=0))
    cam = cam.detach().float().cpu().numpy()
    cam = cam - cam.min()
    return cam / (cam.max() + 1e-12)


def integrated_gradients(net, batch: dict, elem: int, steps: int = 32) -> np.ndarray:
    """Straight-line integrated gradients from a zero-image baseline (the masked border value)."""
    base = torch.zeros_like(batch["images"][:, elem], dtype=torch.float32)
    inp = batch["images"][:, elem].float()
    total = torch.zeros_like(inp)
    for s in range(1, steps + 1):
        xi = (base + (s / steps) * (inp - base)).requires_grad_(True)
        b2 = dict(batch)
        imgs = batch["images"].float().clone()
        imgs[:, elem] = xi
        b2["images"] = imgs
        score = cumhaz_score(net(b2))
        g, = torch.autograd.grad(score.sum(), xi)
        total = total + g
    ig = ((inp - base) * total / steps).squeeze(0).detach().float().cpu().numpy()
    return np.abs(ig) / (np.abs(ig).max() + 1e-12)

In [ ]:
# ---- 17b. Stratified TP / FP / TN / FN sample, Grad-CAM, IG, occlusion --------------------------
N_PER_CELL = 6
_ds = TEST_ARTIFACTS["datasets"][PRIMARY_ARM]
_ci = TEST_ARTIFACTS["cidx"][PRIMARY_ARM]
_p = TEST_ARTIFACTS["models"][PRIMARY_ARM][PRIMARY_HORIZON_DAYS]
_y = TEST_ARTIFACTS["yw"][PRIMARY_HORIZON_DAYS][0]
_alert = _p >= THRESHOLD
CELLS = {"TP": np.where(_alert & (_y == 1))[0], "FP": np.where(_alert & (_y == 0))[0],
         "TN": np.where(~_alert & (_y == 0))[0], "FN": np.where(~_alert & (_y == 1))[0]}
_rng = np.random.default_rng(BOOT_SEED)
SAMPLE = {k: _ci[_rng.choice(v, size=min(N_PER_CELL, len(v)), replace=False)]
          for k, v in CELLS.items() if len(v)}
print("stratified explainability sample (frozen operating point, primary horizon):",
      {k: (len(CELLS[k]), len(SAMPLE.get(k, []))) for k in CELLS})

_net, _ = load_seed_model(PRIMARY_ARM, SEEDS[0], mode=ARMS[PRIMARY_ARM]["mode"])
_net.eval()
_maps, _occ_rows = {}, []
for cell, idxs in SAMPLE.items():
    for i in idxs:
        b = batch_from(_ds, int(i))
        n_el = int(b["mask"].sum())
        for e in range(n_el):
            # keyed by the ELEMENT index, not the view id: a patient can contribute two frontals
            # and keying on the view would silently overwrite one of their maps
            _maps[(cell, int(i), int(e), int(b["view_id"][0, e]))] = {
                "gradcam": grad_cam(_net, b, e),
                "ig": integrated_gradients(_net, b, e, steps=16)}
        with torch.no_grad():
            base = float(cumhaz_score(_net(b)))
            row = {"cell": cell, "patient_row": int(i), "n_views": n_el, "base_score": base}
            for reg in ("medial", "lateral", "patellofemoral", "border_ring", "central_joint",
                        "wide_border", "corners"):
                b2 = dict(b); b2["images"] = occlude(b["images"], reg)
                row[f"delta_{reg}"] = float(cumhaz_score(_net(b2))) - base
            _occ_rows.append(row)
OCCLUSION = pd.DataFrame(_occ_rows)
OCCLUSION.to_csv(RESULTS_DIR / "explainability_occlusion.csv", index=False)
print(f"\n--- occlusion: change in cumulative hazard when a region is blanked (n={len(OCCLUSION)}) ---")
print(OCCLUSION.groupby("cell")[[c for c in OCCLUSION.columns if c.startswith("delta_")]]
      .mean().round(4).to_string())
print("\nInterpretation guide, in advance of the numbers:")
print("  central_joint should move the score MOST - that is where joint-space narrowing lives.")
print("  border_ring / wide_border / corners should move it LEAST - those are the bands the")
print("  preprocessing already blanked. A large border effect means the model found a burned-in")
print("  artefact and protocol section 22 requires an artefact review before anything is claimed.")
_b = OCCLUSION[["delta_border_ring", "delta_wide_border", "delta_corners"]].abs().to_numpy().max()
_c = OCCLUSION["delta_central_joint"].abs().mean()
print(f"\n  max |border-family effect| = {_b:.4f} vs mean |central-joint effect| = {_c:.4f} "
      f"-> ratio {_b/max(_c,1e-9):.2f}")
print("  ARTEFACT REVIEW TRIGGERED" if _b > _c else "  no border-dominant attribution detected")

In [ ]:
# ---- 17c. Randomization controls: random encoder weights, and permuted training labels ----------
RUN_LABEL_RANDOMIZATION = True

def _map_stats(m):
    m = np.asarray(m, float)
    return {"top10pct_mass": float(np.sort(m.ravel())[-max(1, m.size // 10):].sum() / (m.sum() + 1e-12)),
            "entropy": float(-(m / (m.sum() + 1e-12) * np.log((m / (m.sum() + 1e-12)) + 1e-12)).sum())}


_rand = SurvivalFusionNet(n_intervals=N_INTERVALS, n_clinical=len(DESIGN_COLUMNS),
                          mode=ARMS[PRIMARY_ARM]["mode"], pretrained=False,
                          base_hazard=BASE_HAZARD).to(DEVICE).eval()
_rows = []
for (cell, i, e, vid), mp in list(_maps.items())[:24]:
    b = batch_from(_ds, i)
    r_cam = grad_cam(_rand, b, e)
    _rows.append({"cell": cell, "view": VIEWS[vid], "model": "trained", **_map_stats(mp["gradcam"])})
    _rows.append({"cell": cell, "view": VIEWS[vid], "model": "random_weights", **_map_stats(r_cam)})
    _rows.append({"cell": cell, "view": VIEWS[vid], "model": "trained_IG", **_map_stats(mp["ig"])})
RANDOMIZATION = pd.DataFrame(_rows)
print("--- saliency concentration: trained vs randomly-initialised encoder ---")
print(RANDOMIZATION.groupby("model")[["top10pct_mass", "entropy"]].agg(["mean", "std"]).round(4).to_string())
print("A trained map should be MORE concentrated (higher top-10% mass, lower entropy) than a random"
      "\none. If they are indistinguishable, the map is reporting image statistics, not the model.")
del _rand

if RUN_LABEL_RANDOMIZATION:
    print("\n--- label-randomization control: one seed trained on PERMUTED training labels ---")
    RAND_FRAME = DEV_FRAME.copy()
    _tr_mask = (RAND_FRAME["split"] == "train").to_numpy()
    _perm = np.random.default_rng(BOOT_SEED).permutation(int(_tr_mask.sum()))
    for _c in ("event_indicator", "time_from_landmark"):
        _v = RAND_FRAME.loc[_tr_mask, _c].to_numpy()
        RAND_FRAME.loc[_tr_mask, _c] = _v[_perm]
    _rtr = PatientViewDataset("train", RAND_FRAME, DEV_X, train=True)
    _rva = PatientViewDataset("val", RAND_FRAME, DEV_X, train=False)
    _ck = train_one_seed("x_labelrand", SEEDS[0], mode=ARMS[PRIMARY_ARM]["mode"],
                         views_allowed=VIEWS, train_ds=_rtr, val_ds=_rva)
    print(f"  best val NLL on permuted labels {_ck['best_val_nll']:.6f} vs the real model's "
          f"{SEED_RESULTS[PRIMARY_ARM][SEEDS[0]]['best_val_nll']:.6f}")
    print("  (a permuted-label model that matches the real one means the real one learned nothing)")
    _lr, _ = load_seed_model("x_labelrand", SEEDS[0], mode=ARMS[PRIMARY_ARM]["mode"])
    _rows = []
    for (cell, i, e, vid), mp in list(_maps.items())[:24]:
        b = batch_from(_ds, i)
        _rows.append({"cell": cell, "model": "label_randomized", **_map_stats(grad_cam(_lr, b, e))})
    RANDOMIZATION = pd.concat([RANDOMIZATION, pd.DataFrame(_rows)], ignore_index=True)
    del _lr
RANDOMIZATION.to_csv(RESULTS_DIR / "explainability_randomization.csv", index=False)
print("\nThese maps are MODEL-ATTRIBUTION VISUALIZATIONS. They are not validated imaging "
      "biomarkers and must not be described as showing where disease is (protocol section 22).")

In [ ]:
# ---- 17d. Render the stratified attribution figure ------------------------------------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

_pick = []
for cell in ("TP", "FP", "TN", "FN"):
    _pick += [k for k in _maps if k[0] == cell][:3]
if _pick:
    fig, axes = plt.subplots(3, len(_pick), figsize=(2.1 * len(_pick), 6.6), squeeze=False)
    for c, key in enumerate(_pick):
        cell, i, e, vid = key
        img = np.asarray(_ds.images[_ds.elems[i][e][0]])        # the exact image this map is of
        cam = _maps[key]["gradcam"]
        cam_up = np.kron(cam, np.ones((OUT_SIZE // cam.shape[0], OUT_SIZE // cam.shape[1])))
        cam_up = np.pad(cam_up, ((0, OUT_SIZE - cam_up.shape[0]), (0, OUT_SIZE - cam_up.shape[1])))
        axes[0][c].imshow(img, cmap="gray"); axes[0][c].set_title(f"{cell} {VIEWS[vid]}", fontsize=8)
        axes[1][c].imshow(img, cmap="gray"); axes[1][c].imshow(cam_up, cmap="jet", alpha=0.45)
        axes[2][c].imshow(_maps[key]["ig"], cmap="magma")
        for r in range(3):
            axes[r][c].set_xticks([]); axes[r][c].set_yticks([])
    for r, lab in enumerate(["crop", "Grad-CAM", "integrated gradients"]):
        axes[r][0].set_ylabel(lab, fontsize=8)
    fig.tight_layout()
    _fig = RESULTS_DIR / "explainability_maps.png"
    fig.savefig(_fig, dpi=150); plt.close(fig)
    print("attribution figure ->", _fig)
    print("No title is baked into the image; the caption belongs in the manuscript text.")
del _net
torch.cuda.empty_cache() if DEVICE.type == "cuda" else None

## 18. Protocol section 21 — subgroup performance and equity audit

Report event prevalence, discrimination, calibration-in-the-large, calibration slope and the frozen
operating point within sex, age under 65 versus 65 or older, self-reported Black / White / Asian,
obesity, weight-bearing status, and frontal-only versus multi-view acquisition.

**The suppression rule is not decoration.** Point estimates are suppressed below **50 events** and
confidence intervals are emphasized below **100**. The whole test set has **106 events**. Any split
of it into two groups produces at least one group under 53, and every three-level split produces
groups well under 50. **Expect almost everything on this table to be suppressed**, and expect the
one or two surviving cells to carry intervals wide enough to contain both "much better" and "much
worse".

That is the honest output, and it is printed as such rather than filled with numbers that look like
findings. A subgroup AUROC computed on 11 events is noise with three decimal places.

Race and ethnicity are **evaluation strata only** — they are not in the predictor set (protocol
section 21). Similar performance across groups is described as *subgroup performance*, never as
evidence that the model is equitable.

In [ ]:
# ---- 18. Subgroup table with the protocol section 21 suppression rule ----------------------------
SUPPRESS_BELOW, EMPHASISE_CI_BELOW = 50, 100
_tf = TEST_FRAME.set_index(TEST_FRAME["empi_anon"].astype(str)).loc[COMMON]
_multi = VIEW_MASK.loc[[p for p in COMMON]].sum(axis=1).to_numpy()

SUBGROUPS = {
    "overall": np.ones(len(COMMON), bool),
    "sex: female": _tf["sex_female"].fillna(0).to_numpy().astype(bool),
    "sex: male": ~_tf["sex_female"].fillna(0).to_numpy().astype(bool),
    "age < 65": (_tf["age_at_index"].to_numpy(float) < 65),
    "age >= 65": (_tf["age_at_index"].to_numpy(float) >= 65),
    "obesity flag": _tf["obesity"].fillna(0).to_numpy().astype(bool),
    "no obesity flag": ~_tf["obesity"].fillna(0).to_numpy().astype(bool),
    "weight-bearing frontal": _tf["weight_bearing_frontal"].fillna(0).to_numpy().astype(bool),
    "not weight-bearing": ~_tf["weight_bearing_frontal"].fillna(0).to_numpy().astype(bool),
    "acquisition: frontal only": (_multi == 1) & VIEW_MASK.loc[COMMON, "frontal"].to_numpy(),
    "acquisition: multi-view": (_multi > 1),
}
if "race_major" in _tf.columns:
    for _g in ("Black", "White", "Asian"):
        SUBGROUPS[f"race: {_g}"] = (_tf["race_major"].astype(str).str.lower()
                                    .str.contains(_g.lower(), na=False).to_numpy())

_y, _w = TEST_ARTIFACTS["yw"][PRIMARY_HORIZON_DAYS]
_p = TEST_ARTIFACTS["models"][PRIMARY_ARM][PRIMARY_HORIZON_DAYS]
_rows = []
for name, m in SUBGROUPS.items():
    m = np.asarray(m, bool)
    n, n_ev = int(m.sum()), int(E_COM[m].sum())
    supp = n_ev < SUPPRESS_BELOW
    note = (f"SUPPRESSED: {n_ev} events < {SUPPRESS_BELOW}" if supp else
            (f"interpret via CI: {n_ev} events < {EMPHASISE_CI_BELOW}" if n_ev < EMPHASISE_CI_BELOW
             else ""))
    r = {"subgroup": name, "n": n, "events": n_ev,
         "event_prevalence": (n_ev / n if n else float("nan")), "note": note}
    if supp or n < 20 or not ((_y[m] == 1).any() and (_y[m] == 0).any()):
        r.update({k: float("nan") for k in
                  ("auroc", "auroc_lo", "auroc_hi", "slope", "citl", "sensitivity", "specificity",
                   "alert_rate")})
    else:
        bidx = np.where(m)[0]
        boot = np.array([ipcw_auc(_y[bidx][s], _w[bidx][s], _p[bidx][s])
                         for s in np.random.default_rng(BOOT_SEED).integers(
                             0, len(bidx), size=(BOOTSTRAP_N, len(bidx)))])
        lo, hi, _ = pct_ci(boot)
        slope, citl = calibration_slope_intercept(_y[m], _w[m], _p[m])
        al = _p[m] >= THRESHOLD
        r.update({"auroc": ipcw_auc(_y[m], _w[m], _p[m]), "auroc_lo": lo, "auroc_hi": hi,
                  "slope": slope, "citl": citl,
                  "sensitivity": float((al & (_y[m] == 1)).sum() / max((_y[m] == 1).sum(), 1)),
                  "specificity": float((~al & (_y[m] == 0)).sum() / max((_y[m] == 0).sum(), 1)),
                  "alert_rate": float(al.mean())})
    _rows.append(r)
SUBGROUP_TABLE = pd.DataFrame(_rows)
SUBGROUP_TABLE.to_csv(RESULTS_DIR / "test_subgroups.csv", index=False)
print(SUBGROUP_TABLE.round(4).to_string(index=False))

_n_supp = int(SUBGROUP_TABLE["note"].str.startswith("SUPPRESSED").sum())
_n_ci = int(SUBGROUP_TABLE["note"].str.startswith("interpret").sum())
print(f"\n{_n_supp} of {len(SUBGROUP_TABLE)} subgroups are SUPPRESSED (fewer than "
      f"{SUPPRESS_BELOW} events) and {_n_ci} more must be read through their intervals "
      f"(fewer than {EMPHASISE_CI_BELOW}).")
print("With 106 test events this is the expected outcome, not a failure of the analysis. Report it "
      "as\ninsufficient precision for subgroup inference. Do NOT report the suppressed cells with a "
      "footnote;\nsuppression means they are not shown.")
print("Similar performance across groups, where it can be estimated at all, is SUBGROUP "
      "PERFORMANCE.\nIt is not evidence that the model is equitable (protocol section 21).")

## 19. What this notebook produced, and what it did not

### Files on Drive

| file | contents |
| --- | --- |
| `results_t7/FROZEN_MANIFEST.json` | the complete frozen specification: architecture, interval grid, ensemble rule, every recalibration coefficient, the operating threshold, the training-contract hash, the analysis-script SHA-256, the environment |
| `results_t7/environment.json` | resolved versions of every package that touches a number, plus the GPU |
| `results_t7/seed_variability.csv` | per-arm validation NLL across the five seeds |
| `results_t7/val_metrics.csv` | validation discrimination for every arm and the paired M0 reference |
| `results_t7/test_results.json` | the primary paired difference, every secondary family, the operating point, the comparison-set definition |
| `results_t7/test_metrics.csv`, `test_secondary_comparisons.csv`, `test_subgroups.csv` | the same numbers in table form |
| `results_t7/test_secondary_objective_2.csv` | protocol Secondary objective 2: raw image versus the inferred-KLG-plus-clinical comparator (M1), on the KLG-eligible test subset |
| `results_t7/explainability_*.csv`, `explainability_maps.png` | protocol section 22 outputs |
| `test_seal/TEST_READ_ONCE.json` | the one-time marker; its presence blocks any further test read |
| `checkpoints/{arm}_seed{seed}.pt` | resumable per-seed checkpoints with the training-contract hash |

### For the manuscript

* **TRIPOD+AI** and **CLAIM 2024** are the reporting instruments (protocol section 27), and
  **PROBAST+AI** should be re-run before submission.
* Quote **12 identified parameters** for M0 (13 design columns), not 13, in any
  events-per-parameter statement, and report no individual age-spline hazard ratio
  (`identifiability.level_unidentified_columns`). M1 has 14 columns and 13 identified.
* **M0 contains no inferred KLG.** Protocol Table 6 makes the dataset-inferred contralateral KLG
  a secondary comparator and Table 7 puts it in M1; the clinical comparator in the primary
  estimand is age, sex, comorbidities, pain and the image-to-index interval. What KLG adds is
  reported separately, as M1, in `outputs/clinical_m1_klg_report.md` and in the Secondary
  objective 2 family here. See deviations **D14** and **D15** in `outputs/protocol_deviations.md`.
* The primary result is a **paired difference in IPCW cumulative/dynamic AUROC**, not an absolute
  AUROC. Say so in the abstract.
* The outcome is **observed procedure utilization**. It reflects structural disease, symptoms,
  patient preference, access, surgeon recommendation and local practice. Do not describe it as
  biological need for surgery (protocol section 4 interpretation boundary).
* Attribution maps are **model-attribution visualizations, not validated imaging biomarkers**.

### What a reviewer should push on

1. **The 5-year versus 2-year primary.** Protocol Table 7 says 5 years; the config and the plan call
   2 years co-primary. Section 14 nominates the 5-year comparison as the single primary inference
   and demotes 2 years to the first secondary family. That is a defensible reading of protocol
   section 18, but it *is* a reading, and the alternative (2-year primary, 5-year secondary) would
   change which number carries an unadjusted claim.
2. **`MEDIAL_SIDE`.** Derived from the display convention, not measured. If the crop QA sheet
   disagrees, every medial/lateral occlusion statement flips.
3. **The paired comparison set.** Frontal-only arms cannot score a patient with no frontal, so the
   paired set is smaller than 741. The excluded count is printed; it should appear in the flow
   diagram.
4. **Recalibration on 54 validation events.** The frozen intercept and slope are estimated from a
   small sample and applied to test unchanged. A poor test calibration slope after recalibration is
   a real limitation, not something to re-fit.
5. **`EVENT_AWARE_MINIBATCH` defaults to off.** Section 15 permits event-aware minibatches; this
   notebook keeps the natural prevalence everywhere, which is the more conservative reading.
6. **No hyper-parameter search.** The single configured setting is used as written. That is
   deliberate at 373 training events, but it means the reported performance is not the best
   achievable — only the best *pre-specified*.
7. **12.4% of development patients are silent in the likelihood.** 367 of 2,968 are censored inside
   the first 182.6-day interval and therefore contribute no scored interval under the strict
   discrete-time convention (section 6). A fractional-exposure convention would recover them at the
   cost of an assumption. The choice is conservative, and it is a choice.
8. **The image is fed at the full 512x512 crop.** No downscaling, because protocol section 13
   specifies 512 and the outcome is driven by joint-space width. That is why the run needs a large
   GPU; a downscaled variant would be a recorded deviation, not a convenience.

### Explicitly out of scope here

Every item below is **recorded in the canonical register**, `outputs/protocol_deviations.md`
(entries **D16**, **D17** and **D18**), with its reason and what completing it would take. This
list is a pointer, not the record — a scope declaration that lives only in a notebook markdown
cell is not a deviation log.

* **Protocol section 25** (D17): self-supervised pretraining versus ImageNet init, and performance
  by image-to-index interval or calendar era. Repeated five-fold grouped cross-validation for
  selection stability IS run, inside the training split, by `src/model_clinical.py`.
* **Protocol section 24** (D16): landmark day 30 / 180, the endpoint-definition sensitivity, the
  KLG-4 exclusion in the KLG-eligible subset, earliest-versus-most-recent eligible image, the
  weight-bearing-frontal-only model, and the exclude-autoimmune/trauma analysis. These re-open the
  cohort or the label, so they belong upstream of this notebook, and running them here would need
  a second test read. (The race-included and no-pain-predictor sensitivities ARE run, on M0, by
  `src/model_clinical.py`.)
* **Decision-curve analysis** (protocol section 18, exploratory; D18) — the net-benefit
  computation belongs with the manuscript figures, and it needs no further model output beyond
  `test_results.json`.

In [ ]:
# ---- 19. Final summary ---------------------------------------------------------------------------
print("=" * 96)
print("RUN SUMMARY".center(96))
print("=" * 96)
print(f"  gate                 crop QA signed PASS, section 23 image audit and laterality audit on record")
print(f"  data                 train {len(TRAIN_DS)} / val {len(VAL_DS)} patients; "
      f"{FROZEN_IMAGES['train']} / {FROZEN_IMAGES['val']} images")
print(f"  arms trained         {list(ARMS)}")
print(f"  seeds                {SEEDS} (hazards averaged)")
print(f"  frozen manifest      {MANIFEST_PATH}")
print(f"  analysis hash        {ANALYSIS_HASH[:32]}...")
if (SEAL_DIR / 'TEST_READ_ONCE.json').exists():
    _r = json.loads((SEAL_DIR / 'TEST_READ_ONCE.json').read_text())
    print(f"  test read            {_r['read_utc']} (once)")
    print(f"  PRIMARY              {PRIMARY_RESULT['comparison']} @ {int(PRIMARY_HORIZON_DAYS)} d: "
          f"{PRIMARY_RESULT['difference']:+.4f} "
          f"[{PRIMARY_RESULT['difference_ci'][0]:+.4f}, {PRIMARY_RESULT['difference_ci'][1]:+.4f}], "
          f"p = {PRIMARY_RESULT['p_two_sided']:.4f}")
    print(f"  comparison set       {N_COM} patients / {int(E_COM.sum())} events "
          f"({len(TEST_FRAME) - N_COM} test patients not scoreable by every arm)")
else:
    print("  test read            NOT PERFORMED - the seal is intact")
print("=" * 96)
print("Record any deviation from the frozen manifest in outputs/protocol_deviations.md with a date\n"
      "and a rationale. Nothing in this notebook may be re-run against the test set.")